# NOIPA: Multi-agent system for data quality

NOIPA is a multi-agent system for validating and cleaning Italian public-administration CSV datasets, developed as part of the "Machine Learning" of the "Data Science and Management" course. The authors:
- Bruni Sofia
- Sebastiani Mattia
- Turco Michele

# 1) Introduction and Architecture Overview

The notebook explains the pipeline end-to-end and shows the real production code used to run it. 
It is **runnable top-to-bottom** on `Data/spesa.csv` or any other CSV file: every cell executes the real production code, developed on separate python files in order to maintain modularity, and produces a Markdown narrative report at the end.
In order to provide the reader with a clear understanding of the pipeline, it is useful to first look at the overall architecture of the system, which is composed of two main parts:
- **Validation** (read-only): infers per-column dtypes, measures completeness, detects format inconsistencies, surfaces anomalies, checks cross-column coherence, and finds duplicate records.
- **Cleaning**: derives a deterministic remediation plan from the validation findings, then runs a **generator / critic** loop that synthesises one Python cleaning function per inconsistent column, applies the plan plus the generated cleaners, verifies the result, and writes a narrative report.

----- placeholder for architecture diagram -----

All agents are [Pydantic AI](https://ai.pydantic.dev) agents backed by `openai-responses:gpt-5.4-mini`. 
Every structured output is a Pydantic model: the prompt states *what* to do, the Pydantic schema states *how* the answer must be shaped, and host-side code owns correctness checks and retries.


# 2) Setup

As a first step, we load the OpenAI API key from the `.env` file, import all the necessary modules, and patch the notebook event loop so that Pydantic AI can run cleanly inside Jupyter, and enable Logfire tracing. The lower-level plumbing stays inside the production modules and is exercised later through the stage functions.

In [ ]:
# Import standard libraries and third-party dependencies
from __future__ import annotations
import json
import math
import os
import re
import sys
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path
from typing import Any, Callable, Literal
import asyncio

import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field, field_validator
from pydantic_ai.exceptions import UsageLimitExceeded
from pydantic_ai.usage import UsageLimits
import logfire
from pydantic_ai import Agent, CodeExecutionTool, PromptedOutput


# Pydantic AI agents call asyncio.run() internally; Jupyter already has a running loop, so we patch it to allow re-entry.
nest_asyncio.apply()

# Read OPENAI_API_KEY from the repository .env if available.
load_dotenv(dotenv_path=Path(".env"), override=False)

# Configure Logfire observability after the agent module is loaded.
# from src.core.agents import setup_logfire
# setup_logfire()

# Cache helpers for validation, remediation, and orchestration artifacts.
from src.core.cache import (
    load_anomaly,
    load_completeness,
    load_consistency,
    load_cross_column,
    load_duplicates,
    load_remediation_plan,
    load_schema_handoff,
    save_anomaly,
    save_completeness,
    save_consistency,
    save_cross_column,
    save_duplicates,
    save_remediation_plan,
    save_schema_handoff,
    save_validation_results,
)

# Pydantic contracts used by the displayed production functions.
from src.core.models import (
    AnomalyDetectionReport,
    AnomalyFinding,
    CleanerRepairContext,
    CleanerValidationIssue,
    CleaningReport,
    ColumnCleanerExecutionReport,
    ColumnConsistencyReport,
    ColumnCleanerProgram,
    AnomalySummaryOutput,
    CleanerRepairDiagnosis,
    ColumnCleaningRequest,
    CompletenessAnalysisReport,
    ConsistencyValidationReport,
    ConsistencyVerificationReport,
    CrossColumnFinding,
    CrossColumnSummaryOutput,
    CrossColumnValidationReport,
    DatasetDtypeInference,
    DuplicateDetectionReport,
    DuplicateRecordGroup,
    DuplicateSummaryOutput,
    FinalPipelineReport,
    FindingDiff,
    FormatConsistencyFinding,
    GeneratedCleanerArtifact,
    NarrativeFrontMatter,
    NarrativeReport,
    NarrativeReportSection,
    OrchestrationStepResult,
    RemediationPlan,
    SchemaColumnEntry,
    SchemaHandoff,
    SchemaIssue,
    SchemaSummaryOutput,
)

# Shared tools used across validation and cleaning cells.
from src.tools import (
    PLACEHOLDER_TOKENS,
    ColumnFormatFacts,
    FormatOutlierExample,
    SchemaDuplicateGroup,
    attach_profile_text,
    attach_text_document,
    build_dtype_inference_text,
    detect_date_order_violations,
    detect_duplicate_semantic_conflicts,
    detect_exact_duplicate_groups,
    gzip_text_to_base64,
    infer_duplicate_key_columns,
    is_valid_schema_name,
    load_dataset_frame,
    naming_rule_reason,
    normalized_schema_name,
    numeric_pattern_allows_variable_width,
    run_agent_with_backoff,
    suggest_schema_name,
    value_shape,
)
from src.tools.completeness_tools import (
    CompletenessColumnProfile,
    CompletenessProfile,
    compute_missing_like_mask,
    detect_placeholder_values,
    sample_placeholder_examples,
)
from src.tools.schema_tools import (
    ColumnProfile,
    DatasetProfile,
    compute_datetime_parse_pct,
    compute_empty_like_pct,
    compute_numeric_parse_pct,
    sample_non_null_values,
)

# Format profiling helpers used by the displayed format-consistency cells.
from src.tools.format_tools import (
    build_column_format_profile,
    infer_format_semantic_hint,
    is_plain_numeric_value,
    select_outlier_examples,
)

# Quality-tool internals surfaced by the displayed anomaly/cross-column/duplicate cells.
from src.tools.quality_tools import (
    _eligible_column_pair,
    _find_pattern_columns,
    _looks_like_period_column,
    _normalized_row_signature,
    _normalized_series,
    _schema_column_map,
    _series_to_numeric,
    normalize_cell_text,
    render_cell_text,
)

from src.validation._summary import summarize_validation_report

from src.validation.anomaly import _duplicate_semantic_suppressed_columns

from src.validation.consistency import (
    _build_suggested_strategy,
    _normalize_expected_pattern,
    _profile_schema_guided_inconsistencies,
    _run_column_format_checks_async,
    _schema_pattern_is_ambiguous,
)
from src.validation.schema import _normalize_dtype_inference_choice, build_schema_issues

# Cleaning request helpers.
from src.cleaning.request import (
    _augment_datetime_strategy,
    _augment_yyyymm_strategy,
    _build_datetime_expected_pattern,
)

# Generator / critic loop helpers and entrypoints.
from src.cleaning.generation import (
    GENERATOR_USAGE_LIMITS,
    ProgressCallback,
    _build_cleaner_generation_prompt,
    _generation_lock,
    _run_cleaner_generation_locked,
    _stagnation_temperature,
    format_validation_issue,
    rebuild_verified_program,
    run_cleaner_repair_critic,
    validation_issue_fingerprint,
    _GenerationProgress,
)

# Host-side validator helpers for generated cleaner programs.
from src.cleaning.validation import (
    _diagnose_datetime_component_order,
    _suggest_corrected_datetime,
    build_runtime_exception_issue,
    build_validation_issue,
    detect_shadowed_delimiter_branches,
    dominant_datetime_example,
    dominant_output_shape,
    is_parseable_output,
    load_cleaner_callable,
    matches_dominant_datetime_format,
    matches_request_target_pattern,
    requires_fixed_output_shape,
    validate_generated_cleaner_program,
)

# Deterministic remediation planning.
from src.cleaning.remediation import (
    _resolve_validation_results,
    build_remediation_plan,
)

# Cleaner application helpers and entrypoints.
from src.cleaning.application import (
    _apply_column_renames,
    _apply_dtype_casts,
    _apply_exact_duplicate_column_drops,
    _apply_exact_duplicate_row_drops,
    _apply_placeholder_nulls,
    _apply_string_lowercasing,
    _clone_remediation_plan,
    _load_artifact_program,
)
from src.cleaning.paths import (
    cleaned_dataset_path,
    cleaning_cache_dir,
    load_cleaner_manifest,
    save_cleaner_manifest,
)
from src.cleaning.runtime import apply_cleaner_to_series

# Verification and reporting entrypoints.
from src.cleaning.verification import (
    _print_diff_table,
    _numeric_original_names,
    _diff_summary,
    _schema_rename_map,
)
from src.cleaning.reporting import (
    _compute_cleaned_non_null_counts,
    _build_frontmatter_brief,
    _build_narrative_section_specs,
    _polish_narrative_body,
    save_final_report,
    save_narrative_report,
)


# 3) Dataset

The system is deisgned to run on any CSV file provided by the an italian public administration. The two datasets that can be used to test the system are `spesa.csv` and `attivazioniCessazioni.csv`, both to be placed in the `Data/` directory. The former is a small dataset of ~20k rows about public spending, while the latter is a much larger dataset of ~1.5M rows about employee activations and cessations.

In order to keep the runtime of the notebook reasonable and for demonstration purposes, we default to `Data/spesa.csv`. 
To run the system on `Data/attivazioniCessazioni.csv`, simply change the `DATASET_PATH` variable in the next cell and re-run all the cells below.

In [ ]:
# Define the dataset path and verify it exists before proceeding.
DATASET_PATH = Path("Data/numeric_inconsistent_dataset_1000x8.csv").resolve()
assert DATASET_PATH.exists(), f"dataset not found: {DATASET_PATH}"
print(DATASET_PATH)

In [ ]:
raw_df = load_dataset_frame(DATASET_PATH)
print(f"{len(raw_df):,} rows x {len(raw_df.columns)} columns")
raw_df.head()


# 4) Data Contracts (Pydantic Models)
For the pipeline to function reliably, every stage of it must define its data contracts explicitly before any data flows through it. A data contract specifies the exact shape, type, and validity rules of the data a component accepts and produces, so that any deviation is caught immediately at the handoff point rather than propagated silently downstream. Pydantic is the library used here to define and enforce those contracts. This section displays the full contract layer, grouping its classes by responsibility rather than by raw file order, so that the reader can follow them in the same sequence the pipeline actually uses them, moving from the most foundational definitions to the most orchestrating ones.
The classes are grouped into five stages, each corresponding to a distinct responsibility in the pipeline, from shared primitives to final orchestration:
1. Shared modeling primitives: controlled vocabularies and type literals.
2. Schema profiling, inference, and handoff models: what the pipeline believes each column should be.
3. Validation and verification models: the evidence produced by the validation half and by post-cleaning verification.
4. Cleaning and repair models: the generator input/output contracts and execution logs.
5. Planning, reporting, and orchestration models: how stage-level outputs are merged into final pipeline artifacts.

The only important external type referenced from outside `models.py` is `SchemaDuplicateGroup`, which comes from `tools` because it is produced by deterministic schema profiling helpers rather than by the model module itself.


### 4.1) Controlled vocabularies and type literals

Before any model class can be defined, the controlled vocabularies that constrain them must be declared. These are expressed as type literals, a Python construct that restricts a value to a fixed set of allowed strings, and they cover four distinct axes along which the pipeline enforces discipline. The cell defines the following four groups of literals:
- **Valid pandas dtypes**: A set of valid pandas dtypes a column may be cast to: Int64, Float64, datetime64[ns], string, boolean, and object. 
- **Semantic roles**: The semantic role a column may play, which is split between numeric columns, where the distinction between a measurable quantity, a numeric identifier, and an ordinal indicator matters for how the column is treated downstream, and string columns, where the role determines whether a value is an identifier, a category, a name, or free text.
- **Remediation vocabularies**: The actions the pipeline may apply to a column or dataset, the type of object being remediated, the confidence and risk levels associated with a remediation decision, and the status that tracks whether a remediation was planned, applied, or failed. 
- **Failure categories**: The failure categories used by the validation and verification stages to classify the issues they detect, ranging from runtime exceptions to shape mismatches and dtype incompatibilities.

Because these literals are declared at module level and referenced by every model class that follows, any value that falls outside them is rejected by Pydantic at instantiation time, which means that invalid states cannot be represented in the pipeline's data layer at all.

In [ ]:
# Shared type vocabularies used across the pipeline.
VALID_PANDAS_DTYPE = Literal["Int64", "Float64", "datetime64[ns]", "string", "boolean", "object"]

# Logic role for numeric and strig columns
NUMERIC_ROLE = Literal[
    "measure",    # real quantity used arithmetically
    "code",       # numeric identifier that should not be treated as a quantity
    "indicator",  # bounded numeric status / ordinal flag
]

STRING_ROLE = Literal[ "identifier", "categorical", "name","free_text"]

# Remediation actions, object types, confidence/risk levels, and status values
REMEDIATION_ACTION_TYPE = Literal[
    "rename_column",
    "replace_placeholders_with_null",
    "generate_cleaner",
    "drop_exact_duplicate_column",
    "drop_exact_duplicate_rows",
    "cast_dtype",
    "manual_review",
    "report_only",
    "drop_rows_candidate",
]
REMEDIATION_OBJECT_TYPE = Literal["column", "column_pair", "row_group", "dataset"]
REMEDIATION_CONFIDENCE = Literal["low", "medium", "high"]
REMEDIATION_RISK_LEVEL = Literal["low", "medium", "high"]
REMEDIATION_STATUS = Literal["planned", "applied", "proposed_not_applied", "failed", "not_needed"]

# Validation and verification failure categories, used to classify and summarize the issues found by those stages.
VALIDATION_FAILURE_CATEGORY = Literal[
    "program_mismatch",
    "verification_report_failed",
    "non_self_contained_function",
    "runtime_exception",
    "shadowed_specific_branch",
    "dominant_value_modified",
    "outlier_unchanged",
    "outlier_returned_none",
    "unrecoverable_outlier_not_nulled",
    "wrong_output_shape",
    "not_parseable_as_target_dtype",
    "not_matching_target_pattern",
]


### 4.2) Schema profiling, inference, and handoff models

Once the controlled vocabularies are in place, the contracts governing schema analysis can be defined. These models cover the full scope of this stage: a statistical profile of each column, the dtype and semantic role inferred from it, any naming or structural issues detected, and finally a handoff object that holds all of that information in a single structure available to the stages that follow.

The starting point is a statistical snapshot of the raw data. `ColumnProfile` defines the shape of that snapshot for a single column: how many non-null rows it has, how many distinct values, what percentage of its values are expected to parse as numeric or datetime, and a small sample of representative values. `DatasetProfile` is simply the collection of those per-column snapshots for the full dataset. These two models are purely declarative: they specify what will be observed and recorded, before any interpretation takes place.

``` python 
class ColumnProfile(BaseModel):
    """Statistical snapshot of a single column, computed once from the raw DataFrame.
    Everything downstream â€” schema checks, completeness, format â€” derives from this."""
    column_name: str
    pandas_dtype: str                                    # Inferred dtype obtained from LLM call
    non_null_rows: int = Field(ge=0)                     
    distinct_non_null_values: int = Field(ge=0)          
    numeric_parse_pct: float = Field(ge=0, le=100)       
    datetime_parse_pct: float = Field(ge=0, le=100)     
    empty_like_pct: float = Field(ge=0, le=100)          
    sample_values: list[str] = Field(default_factory=list)  # small representative sample for LLM prompts

class DatasetProfile(BaseModel):
    """Full profile of a dataset: one ColumnProfile per column."""
    dataset_name: str
    total_rows: int = Field(ge=0)
    total_columns: int = Field(ge=0)
    columns_profiles: list[ColumnProfile] = Field(default_factory=list)
```

`ColumnDtypeInference` defines the output contract for the dtype inference step on a single column. Beyond the inferred dtype, it requires a semantic role to be assigned when the column is numeric or string, using the literals declared in 4.1, and a dominant value pattern to be identified when one is detectable, such as a date format or a numeric code structure. The rationale field requires the conclusion to be justified explicitly, making the inference step auditable rather than opaque. `DatasetDtypeInference` holds one such record per column and represents the full output expected for a given dataset.

``` python
class ColumnDtypeInference(BaseModel):
    column_name: str
    pandas_dtype: VALID_PANDAS_DTYPE
    numeric_role: NUMERIC_ROLE | None = Field(
        default=None,
        description="Only set when pandas_dtype is Int64 or Float64.",
    )
    string_role: STRING_ROLE | None = Field(
        default=None,
        description="Only set when pandas_dtype is string.",
    )
    detected_pattern: str | None = Field(
        default=None,
        description=(
            "Describe the dominant value format when a clear pattern is present. "
            "Examples: 'YYYY-MM', 'DD/MM/YYYY', 'Italian decimal comma (1.234,56)', "
            "'6-digit numeric code', 'ISO 3166-1 alpha-2 country code'. "
            "Leave null when no consistent pattern is detectable."
            "Pick the most common pattern only, no mixed results."
        ),
    )
    rationale: str

class DatasetDtypeInference(BaseModel):
    columns: list[ColumnDtypeInference]
```

`SchemaIssue` defines the structure of a single problem found during schema analysis. Each issue is attached to a column and carries enough information to act on it: the type of problem, its severity, the evidence supporting it, and a suggested fix. When confidence in the fix is not high, the suggested_strategy field takes over, requiring a more cautious remediation approach to be specified rather than a precise correction.

``` python 
class SchemaIssue(BaseModel):
    """A discrete issue found in schema validation, dtype inference, or naming checks."""
    column_name: str
    issue_type: str = Field(
        description="Use values such as naming_standard, reserved_word_risk, inferred_type_mismatch, or duplicate_column_semantics."
    )
    severity: str = Field(description="Use low, medium, or high.")
    evidence: str
    fix_confidence: str = Field(description="Use high, medium, or low.")
    suggested_fix: str
    suggested_strategy: str = Field(
        description="If fix_confidence is not high, provide a cautious remediation strategy instead of a precise correction."
    )
```

`SchemaColumnEntry` is the central per-column record of the schema analysis stage. It merges into a single object everything that is known about a column at this point: the dtype and semantic role inferred in the previous step, the statistical measurements recorded in the profile, and the outcome of the naming check, including a rename suggestion when the current name does not meet the expected standard.

``` python
class SchemaColumnEntry(BaseModel):
    """All facts about one column: dtype inference, statistics, and naming checks."""
    name: str
    pandas_dtype: str
    numeric_role: NUMERIC_ROLE | None = None
    string_role: STRING_ROLE | None = None
    detected_pattern: str | None = None
    rationale: str
    non_null_rows: int = Field(ge=0)
    distinct_non_null_values: int = Field(ge=0)
    numeric_parse_pct: float = Field(ge=0, le=100)
    datetime_parse_pct: float = Field(ge=0, le=100)
    empty_like_pct: float = Field(ge=0, le=100)
    sample_values: list[str] = Field(default_factory=list)
    naming_valid: bool
    rename_suggestion: str | None = None
    naming_reason: str | None = None
```

`SchemaHandoff` defines the top-level structure that holds the per-column entries, the list of issues, and any groups of columns that share a canonical name and may therefore be semantically redundant. `SchemaDuplicateGroup` defines the shape of those redundancy groups. The summary field holds a human-readable account of the analysis, and `SchemaSummaryOutput` defines the output contract for the summarisation step.

``` python
class SchemaHandoff(BaseModel):
    """Complete schema analysis result. Column-centric: all dtype, statistical, and naming
    facts per column are merged. Issues and duplicate groups are kept at the top level
    for easy scanning by downstream fixing agents."""
    dataset_name: str
    total_rows: int = Field(ge=0)
    total_columns: int = Field(ge=0)
    columns: list[SchemaColumnEntry] = Field(default_factory=list)
    issues: list[SchemaIssue] = Field(default_factory=list)
    duplicate_groups: list[SchemaDuplicateGroup] = Field(default_factory=list)
    summary: str = ""

class SchemaSummaryOutput(BaseModel):
    summary: str

class SchemaDuplicateGroup(BaseModel):
    """A group of columns that normalize to the same canonical name,
    suggesting possible semantic overlap (e.g. 'Provincia Sede' and 'provincia_sede')."""
    canonical_name: str                          # the shared normalized name
    columns: list[str] = Field(default_factory=list)  # original column names in this group
```

## 4.3) Validation findings
The validation stage is responsible for systematically examining the dataset across five distinct dimensions and recording what it finds. The models defined here are the contracts that structure those findings: they do not trigger any analysis, but define precisely what form the evidence must take once the analysis is performed. The five dimensions covered are completeness, format consistency, anomalies, cross-column conflicts, and duplicate records.

The starting point is a statistical snapshot of completeness. `CompletenessColumnProfile` records, for a single column, how many rows contain a real value versus a null-like or placeholder token, together with a few representative examples of those placeholder tokens. `CompletenessProfile` holds those per-column snapshots as a dataset-level view, adding an overall completeness percentage and the full list of distinct placeholder spellings detected anywhere in the dataset.

``` python
class CompletenessColumnProfile(BaseModel):
    """Statistical completeness snapshot for one column.

    Captures how many rows contain a real value versus a null-like or
    placeholder token, along with a few representative placeholder examples
    for downstream explanation by the agent.
    """
    column_name: str
    pandas_dtype: str
    total_rows: int = Field(ge=0)
    non_null_rows: int = Field(ge=0)
    completeness_pct: float = Field(ge=0, le=100)
    missing_like_count: int = Field(ge=0)
    missing_like_pct: float = Field(ge=0, le=100)
    placeholder_examples: list[str] = Field(default_factory=list)  # distinct raw tokens such as "-", "N/A", "unknown"
    distinct_non_null_values: int = Field(ge=0)

class CompletenessProfile(BaseModel):
    """Full completeness profile for a dataset: one per-column snapshot plus rollups."""
    dataset_name: str
    total_rows: int = Field(ge=0)
    total_columns: int = Field(ge=0)
    overall_completeness_pct: float = Field(ge=0, le=100)
    placeholder_values_detected: list[str] = Field(default_factory=list)  # all distinct placeholder spellings seen anywhere in the dataset
    columns: list[CompletenessColumnProfile] = Field(default_factory=list)
```

`CompletenessColumnFinding` defines the per-column structure for completeness measurements, including whether the column qualifies as a sparse candidate and what action is recommended. `CompletenessAnalysisReport` defines the dataset-level structure for completeness findings, with fields for columns with missing values, sparse columns, and placeholder spellings recorded across the dataset. Where the profile is purely observational, the finding layer introduces interpretation.

``` python
class CompletenessColumnFinding(BaseModel):
    """Interpretation of the completeness profile for one column, including whether it qualifies as a sparse candidate and what action is recommended."""
    column_name: str
    completeness_pct: float = Field(ge=0, le=100)
    missing_like_count: int = Field(ge=0)
    missing_like_examples: list[str] = Field(default_factory=list)
    sparse_candidate: bool = False
    recommended_action: str

class CompletenessAnalysisReport(BaseModel):
    """Dataset-level report of completeness analysis, including per-column findings and overall stats."""
    dataset_name: str
    total_rows: int = Field(ge=0)
    total_columns: int = Field(ge=0)
    overall_completeness_pct: float = Field(ge=0, le=100)
    columns_with_missing_values: list[str] = Field(default_factory=list)
    sparse_columns: list[str] = Field(default_factory=list)
    placeholder_values_detected: list[str] = Field(default_factory=list)
    per_column: list[CompletenessColumnFinding] = Field(default_factory=list)
    summary: str
```

The format consistency models capture a different class of problem: values that are present but do not conform to the expected pattern for their column. `FormatConsistencyFinding` defines the per-column structure for the expected pattern, the number of inconsistent rows, supporting evidence, and the suggested strategy. `ConsistencyValidationReport` defines the dataset-level structure for format consistency findings. `ColumnConsistencyReport` is a lighter per-column wrapper used by the slow-path consistency check, which processes columns individually rather than in bulk.

``` python
class FormatConsistencyFinding(BaseModel):
    """For a single column, what pattern was expected, how many rows deviate from it, and what strategy is suggested to address the inconsistency."""
    column_name: str
    expected_pattern: str
    inconsistent_rows: int = Field(ge=0)
    example_inconsistent_values: list[str] = Field(default_factory=list)
    evidence: str
    suggested_strategy: str

class ConsistencyValidationReport(BaseModel):
    """Dataset-level report of format consistency validation, including per-column findings and overall summary."""
    dataset_name: str
    total_rows: int = Field(ge=0)
    format_consistency_findings: list[FormatConsistencyFinding] = Field(default_factory=list)
    summary: str

class ColumnConsistencyReport(BaseModel):
    """Lighter per-column report used by the slow-path consistency check, which processes columns individually rather than in bulk."""
    finding: FormatConsistencyFinding | None = None
    summary: str

```

The anomaly detection models cover two specific types of problem: numeric outliers and rare categories. `AnomalyFinding` defines the per-column anomaly record, including the anomaly type, severity, affected rows, evidence, and suggested action. `AnomalyDetectionReport` defines the dataset-level structure for anomaly findings. `AnomalySummaryOutput` defines the output contract for the summarisation step, with a field for a human-readable account of the anomalies.

``` python
class AnomalyFinding(BaseModel):
    """A single anomaly detected in the dataset, such as a numeric outlier or a rare category."""
    column_name: str
    anomaly_type: Literal["numeric_outlier", "rare_category"]
    severity: Literal["low", "medium", "high"]
    affected_rows: int = Field(ge=0)
    example_values: list[str] = Field(default_factory=list)
    evidence: str
    suggested_action: str

class AnomalyDetectionReport(BaseModel):
    """Dataset-level report of anomalies, including per-column findings and overall summary."""
    dataset_name: str
    total_rows: int = Field(ge=0)
    total_columns: int = Field(ge=0)
    findings: list[AnomalyFinding] = Field(default_factory=list)
    summary: str = ""

class AnomalySummaryOutput(BaseModel):
    """Output contract for the summarisation step that produces a human-readable account of what was found by anomaly detection."""
    summary: str
```

The final two dimensions address problems that cannot be detected by looking at a single column in isolation. The first covers structural conflicts between pairs or groups of columns: `CrossColumnFinding` defines the shape of such a conflict, classifying it by type, such as semantic duplicates, date order violations, or period mismatches, and providing fields for its severity, the affected rows, and the suggested action. `CrossColumnValidationReport` defines the dataset-level structure for those findings, and `CrossColumnSummaryOutput` defines the output contract for the narration step. The second dimension covers duplicate records: `DuplicateRecordGroup` defines the shape of a group of rows that are identical or near-identical, with fields for the rows involved, the matching key columns, and the suggested action. `DuplicateDetectionReport` defines the dataset-level structure for those groups, and `DuplicateSummaryOutput` is the corresponding narration contract.

``` python
class CrossColumnFinding(BaseModel):
    """A structural inconsistency involving two or more columns, such as semantic duplicates, date order violations, or period mismatches."""
    columns: list[str] = Field(default_factory=list)
    check_type: Literal[
        "duplicate_semantic_conflict",
        "exact_duplicate_columns",
        "near_duplicate_columns",
        "year_month_period_mismatch",
        "date_order_violation",
    ]
    severity: Literal["medium", "high"]
    affected_rows: int = Field(ge=0)
    example_row_indices: list[int] = Field(default_factory=list)
    similarity_pct: float | None = Field(default=None, ge=0, le=100)
    evidence: str
    suggested_action: str

class CrossColumnValidationReport(BaseModel):
    """Dataset-level report of cross-column validation, including per-issue findings and overall summary."""
    dataset_name: str
    total_rows: int = Field(ge=0)
    findings: list[CrossColumnFinding] = Field(default_factory=list)
    summary: str = ""

class CrossColumnSummaryOutput(BaseModel):
    """Output contract for the summarisation step that produces a human-readable account of what was found by cross-column validation."""
    summary: str

class DuplicateRecordGroup(BaseModel):
    """A group of rows that are identical or near-identical, suggesting possible duplicate records."""
    duplicate_type: Literal["exact_row", "near_duplicate"]
    row_indices: list[int] = Field(default_factory=list)
    key_columns: list[str] = Field(default_factory=list)
    evidence: str
    suggested_action: str

class DuplicateDetectionReport(BaseModel):
    """Dataset-level report of duplicate record detection, including per-group findings and overall summary."""
    dataset_name: str
    total_rows: int = Field(ge=0)
    groups: list[DuplicateRecordGroup] = Field(default_factory=list)
    summary: str = ""

class DuplicateSummaryOutput(BaseModel):
    """Output contract for the summarisation step that produces a human-readable account of what was found by duplicate record detection."""
    summary: str

```

## 4.4) Cleaning, execution, and repair models

Once the validation findings are in place, the cleaning stage can begin. The models defined here specify the full contract of that stage, from what information a cleaner must receive to produce meaningful code, to how the result of applying that code to the dataframe is recorded. They also cover the repair loop that kicks in when a generated cleaner fails validation, making the critic and correction cycle fully inspectable.


The cleaning process starts with a request. `ColumnCleaningRequest` defines what information must be provided to the generator for a single dirty column: the expected pattern, a semantic hint, the target dtype and role when known, representative examples of both conforming and inconsistent values, and a suggested strategy. `ExampleTransformation` defines the shape of a single input-output pair illustrating what the cleaner should do to a value, including a rationale. `ColumnCleanerProgram` is the output contract for the generator: it requires the returned code to be a single self-contained Python function, accompanied by example transformations, a verification summary, and a list of residual risks that the generator itself acknowledges.

``` python
class ColumnCleaningRequest(BaseModel):
    """All information needed to produce a cleaner for one column"""
    column_name: str
    expected_pattern: str
    semantic_hint: str
    target_dtype: str | None = None
    target_role: str | None = None
    dominant_shape: str | None = None
    dominant_example_values: list[str] = Field(default_factory=list)
    example_inconsistent_values: list[str] = Field(default_factory=list)
    suggested_strategy: str

class ExampleTransformation(BaseModel):
    """A single input-output pair illustrating what the cleaner should do to a value, including a rationale."""
    original_value: str
    cleaned_value: str | None = None
    rationale: str

    @field_validator("original_value", "cleaned_value", mode="before")
    @classmethod
    def coerce_to_str(cls, v):
        if v is None:
            return None
        return str(v)

class ColumnCleanerProgram(BaseModel):
    """The output contract for the cleaner generator: a single self-contained Python function, accompanied by example transformations, a verification summary, 
    and a list of residual risks that the generator itself acknowledges."""
    column_name: str
    function_name: str
    python_code: str = Field(
        description=(
            "Pure Python source code containing exactly one function definition named function_name. "
            "Must be importable as-is: no test code, no print statements, no variable assignments outside the function, no JSON. "
            "All imports and helper constants must be inside the function body."
        )
    )
    example_transformations: list[ExampleTransformation] = Field(default_factory=list)
    verification_summary: str = ""
    residual_risks: list[str] = Field(default_factory=list)

```

When a generated cleaner does not pass validation, the repair loop is triggered. `CleanerValidationIssue` defines the shape of a single validation failure, including the validation category, severity, message, input value, actual output, and expected behaviour. `CleanerRepairContext` defines the context passed to the repair step, containing the original request, the failing program, and the list of validation issues. `CleanerRepairExample` defines the shape of a concrete input-output correction used to guide the repair. `CleanerRepairDiagnosis` is the output contract for the critic: it requires a root cause analysis, a precise identification of where the bug is located, a planned fix, and a set of concrete repair examples, as well as a decision on whether to retry and with what level of confidence.

``` python
class CleanerValidationIssue(BaseModel):
    """A single validation failure for a generated cleaner."""
    category: VALIDATION_FAILURE_CATEGORY
    severity: Literal["high", "medium"]
    message: str
    input_value: str | None = None
    actual_output: str | None = None
    expected_behavior: str

class CleanerRepairContext(BaseModel):
    """Context passed to the repair critic after a validation failure, containing the original request, the failing program, and the validation issues."""
    request: ColumnCleaningRequest
    previous_program: ColumnCleanerProgram
    validation_issues: list[CleanerValidationIssue] = Field(default_factory=list)

class CleanerRepairExample(BaseModel):
    """A concrete input-output correction used to guide the repair of a generated cleaner that failed validation."""
    input_value: str
    actual_output: str | None = None
    expected_output: str | None = None
    fix_note: str

class CleanerRepairDiagnosis(BaseModel):
    """Output contract for the critic: a root cause analysis and a set of concrete repair examples."""
    should_retry: bool = True
    primary_category: VALIDATION_FAILURE_CATEGORY
    root_cause: str
    bug_location: str
    planned_fix: str
    patch_style: Literal["minimal_edit", "targeted_rewrite"]
    priority_issues: list[str] = Field(default_factory=list)
    exact_repairs: list[CleanerRepairExample] = Field(default_factory=list)
    confidence: Literal["low", "medium", "high"]
```

Once a cleaner passes validation and is applied to the dataframe, its resulting cell-level updates must be represented. `CellUpdate` defines the shape of a single cell-level change, with fields for the row index, the original value, and the replacement value. `ColumnCleanerExecutionReport` defines the per-column execution record, including whether execution succeeded, how many rows changed, and what unresolved risks remain.

``` python
class CellUpdate(BaseModel):
    """A single cell-level change resulting from the application of a cleaner, recording the row index, the original value, and the replacement value."""
    row_index: int = Field(ge=0, description="Zero-based row index in the original column.")
    old_value: str | None = Field(
        default=None,
        description="Original value as text for inspection. Use null when the original value is missing.",
    )
    new_value: str | None = Field(
        default=None,
        description="Replacement value as text. Use null when the cleaned value should become missing.",
    )

class ColumnCleanerExecutionReport(BaseModel):
    """Column-level report of the effects of applying a cleaner, including how many rows were changed, what examples of changes look like, and what issues remain unresolved."""
    column_name: str
    function_name: str
    execution_ok: bool = True
    changed_rows: int = Field(ge=0)
    sample_updates: list[CellUpdate] = Field(default_factory=list)
    unresolved_risks: list[str] = Field(default_factory=list)
    summary: str

```

The final group covers the artifacts produced once the cleaning stage is complete. `GeneratedCleanerArtifact` defines the record of an accepted cleaner: the column it targets, the path to the generated code, the number of rows it changed, and example transformations for inspection. `CleaningReport` defines the dataset-level structure for the cleaning outcome, with fields for the before and after row and column counts, the accepted cleaners, any unresolved risks, and the cleaned dataset encoded as a gzip-compressed base64 string.

``` python
class GeneratedCleanerArtifact(BaseModel):
    """A record of an accepted cleaner, including the column it targets, the generated code path, and the changes it made."""
    column_name: str
    function_name: str
    code_path: str
    changed_rows: int = Field(ge=0)
    summary: str
    example_transformations: list[ExampleTransformation] = Field(default_factory=list)

class CleaningReport(BaseModel):
    """A dataset-level summary of the cleaning process, including the changes made and any unresolved risks."""
    dataset_name: str
    rows_before: int = Field(ge=0)
    rows_after: int = Field(ge=0)
    columns_before: int = Field(ge=0)
    columns_after: int = Field(ge=0)
    generated_cleaners: list[GeneratedCleanerArtifact] = Field(default_factory=list)
    unresolved_risks: list[str] = Field(default_factory=list)
    cleaned_csv_gzip_base64: str = Field(description="The cleaned CSV encoded as gzip+base64.")
    summary: str
```


## 4.5) Post-cleaning verification
Once the cleaning stage has been applied, the pipeline re-runs the consistency checks that were performed during validation and compares the results against the original findings. The two models defined here structure that comparison.

`FindingDiff` defines the shape of a single column-level comparison between the state before and after cleaning. It declares fields for the number of inconsistent rows in both states, the percentage reduction, and any remaining example values. The status field constrains the outcome to one of five possible values: resolved, improved, unchanged, regressed, or new, where new indicates that the finding was not present in the original validation pass. The renamed_to field accounts for the case where a column may have been renamed during cleaning, preserving traceability back to the original finding.
`ConsistencyVerificationReport` defines the dataset-level structure for those per-column diffs, alongside a count of how many findings existed before cleaning, how many remain, and a summary field for a human-readable account of the overall outcome.

``` python
class FindingDiff(BaseModel):
    """A structured representation of how a single finding changed between two runs of validation or verification, used for before-and-after comparisons in the reporting stage."""
    column_name: str
    status: Literal["resolved", "improved", "unchanged", "regressed", "new"]
    before_inconsistent_rows: int = Field(ge=0)
    after_inconsistent_rows: int = Field(ge=0)
    reduction_pct: float = Field(ge=-100)
    remaining_examples: list[str] = Field(default_factory=list)
    renamed_to: str | None = None

class ConsistencyVerificationReport(BaseModel):
    """A structured representation of the consistency verification results across the entire dataset."""
    dataset_name: str
    original_finding_count: int = Field(ge=0)
    remaining_finding_count: int = Field(ge=0)
    diffs: list[FindingDiff] = Field(default_factory=list)
    summary: str

```

## 4.6) Planning, final reporting, and orchestration models

The models defined here are the connective tissue of the contract stack. They specify how validation findings are translated into a structured remediation plan, how the outputs of every stage are represented in a single authoritative report, how that report is rendered as human-readable prose, and finally how the entire pipeline result is packaged into a top-level container.

Before any cleaning can take place, there is a need to declare what to do and under what conditions. `RemediationAction` defines the shape of a single intended intervention, with fields for the action type, object type, target, source check, confidence, risk level, application mode, status, reason, and preview statistics. All of these fields are constrained by the literals declared in 4.1, which means that only pre-approved action types, object types, confidence levels, risk levels, and statuses are representable. `RemediationPlan` defines the dataset-level structure for those actions, with an ordered list.

``` python
class RemediationAction(BaseModel):
    """A single planned intervention to fix a validation finding, including what the action is, what it targets, how confident we are in it, and what risks it carries."""
    action_id: str
    action_type: REMEDIATION_ACTION_TYPE
    object_type: REMEDIATION_OBJECT_TYPE
    target: dict[str, Any] = Field(default_factory=dict)
    source_check: str
    confidence: REMEDIATION_CONFIDENCE
    risk_level: REMEDIATION_RISK_LEVEL
    auto_apply: bool
    status: REMEDIATION_STATUS
    reason: str
    preview_stats: dict[str, Any] = Field(default_factory=dict)

class RemediationPlan(BaseModel):
    """A structured plan for addressing validation findings across a dataset."""
    dataset_name: str
    actions: list[RemediationAction] = Field(default_factory=list)
    summary: str = ""

```

`FinalPipelineReport` defines the authoritative factual record of the validation and cleaning pipeline. It declares fields for the full inventory of remediation actions, partitioned by their outcome: applied, proposed but not applied, failed, not needed, flagged as duplicate row drop candidates, or queued for manual review. It also declares fields for the validation findings, the verification diffs, the generated cleaners, row counts, completeness details, and any unresolved risks. This model is the pre-narrative layer: it holds the evidence in structured form before it is rendered as prose.

``` python
class FinalPipelineReport(BaseModel):
    """A comprehensive report of the validation and cleaning pipeline, including validation findings, remediation actions, and remaining issues."""
    dataset_name: str
    validation_summary: dict[str, int] = Field(default_factory=dict)
    applied_actions: list[RemediationAction] = Field(default_factory=list)
    proposed_not_applied_actions: list[RemediationAction] = Field(default_factory=list)
    failed_actions: list[RemediationAction] = Field(default_factory=list)
    not_needed_actions: list[RemediationAction] = Field(default_factory=list)
    duplicate_row_drop_candidates: list[RemediationAction] = Field(default_factory=list)
    manual_review_queue: list[RemediationAction] = Field(default_factory=list)
    cleaning_summary: str = ""
    verification_summary: str = ""
    verification_diffs: list[FindingDiff] = Field(default_factory=list)
    generated_cleaners: list["GeneratedCleanerArtifact"] = Field(default_factory=list)
    total_rows_cleaned: int = 0
    non_null_counts_cleaned: dict[str, int] = Field(default_factory=dict)
    completeness_details: list[CompletenessColumnFinding] = Field(default_factory=list)
    anomaly_findings: list[AnomalyFinding] = Field(default_factory=list)
    cross_column_findings: list[CrossColumnFinding] = Field(default_factory=list)
    duplicate_groups: list[DuplicateRecordGroup] = Field(default_factory=list)
    unresolved_risks: list[str] = Field(default_factory=list)
    summary: str = ""
```

The narrative models define the presentation layer that sits on top of the factual report. `NarrativeFrontMatter` specifies the structure of the opening section of the report: a title, a comprehensive executive summary, and a prioritised list of at least three actionable recommendations. `NarrativeReportSection` defines the shape of a single section of the report body, requiring a heading and a detailed, evidence-rich prose body of at least 150 words. `NarrativeReport` defines the full report structure, requiring at least eight sections alongside the executive summary and recommendations, ensuring that all aspects of the quality analysis are covered.

``` python
class NarrativeFrontMatter(BaseModel):
    """The introductory section of a narrative report, including the title, executive summary, and recommendations for next steps."""
    title: str = Field(description="Report title including the dataset name.")
    executive_summary: str = Field(
        description=(
            "A comprehensive executive summary (8-12 sentences). Cover: dataset dimensions, "
            "overall quality posture, key findings by category, total actions applied vs. proposed, "
            "verification outcome, and residual risk assessment."
        )
    )
    recommendations: list[str] = Field(
        default_factory=list,
        min_length=3,
        description="Prioritized list of actionable next steps for the data steward. At least 3 items.",
    )

class NarrativeReportSection(BaseModel):
    """A single section of the narrative report, containing a heading and a detailed body."""
    heading: str = Field(description="Section title, e.g. 'Validazione dello Schema'.")
    body: str = Field(
        description=(
            "Markdown-formatted prose for this section. Must be detailed and evidence-rich: "
            "include column names, row counts, percentages, concrete examples, and before/after comparisons. "
            "Use markdown tables, bullet lists, and sub-headings where appropriate. "
            "Minimum 150 words per section."
        )
    )

class NarrativeReport(BaseModel):
    """The full narrative report, including the front matter and all sections."""
    title: str = Field(description="Report title including the dataset name.")
    executive_summary: str = Field(
        description=(
            "A comprehensive executive summary (8-12 sentences). Cover: dataset dimensions, "
            "overall quality posture, key findings by category, total actions applied vs. proposed, "
            "verification outcome, and residual risk assessment."
        )
    )
    sections: list[NarrativeReportSection] = Field(
        min_length=8,
        description="At least 8 sections covering all aspects of the quality analysis.",
    )
    recommendations: list[str] = Field(
        default_factory=list,
        min_length=3,
        description="Prioritized list of actionable next steps for the data steward. At least 3 items.",
    )
```

The final two models are the top-level containers that hold the complete pipeline output. `OrchestrationStepResult` defines the structure that groups the outputs of every validation stage into a single object: schema analysis, completeness, consistency, anomaly detection, cross-column validation, and duplicate detection. `CleaningPipelineResult` is the outermost container: it declares fields for the source and output paths, the full validation results, the remediation plan, the cleaning requests, the generated programs, the execution reports, the cleaning report, the verification report, and the final pipeline report. It is the single object from which the entire history and outcome of a pipeline run can be reconstructed.

``` python
class OrchestrationStepResult(BaseModel):
    """The result of a single orchestration step, containing the outputs of all validation stages."""
    schema_validation: SchemaHandoff
    completeness_analysis: CompletenessAnalysisReport
    consistency_validation: ConsistencyValidationReport
    anomaly_detection: AnomalyDetectionReport | None = None
    cross_column_validation: CrossColumnValidationReport | None = None
    duplicate_detection: DuplicateDetectionReport | None = None

class CleaningPipelineResult(BaseModel):
    """The final result of the entire cleaning pipeline, containing all outputs and reports."""
    dataset_name: str
    source_path: str
    cleaned_path: str
    validation_results: OrchestrationStepResult
    remediation_plan: RemediationPlan | None = None
    cleaning_requests: list[ColumnCleaningRequest] = Field(default_factory=list)
    generated_programs: list[ColumnCleanerProgram] = Field(default_factory=list)
    execution_reports: list[ColumnCleanerExecutionReport] = Field(default_factory=list)
    cleaning_report: CleaningReport
    verification_report: ConsistencyVerificationReport | None = None
    final_report: FinalPipelineReport | None = None
```


# 5) The eleven agents

### The 11 agents at a glance

Before the per-agent cells, here is the cast of LLM actors used by the pipeline. All share the same `MODEL` constant, `retries=4`, and `temperature=0` (narrative agents run slightly hotter, like 0.2). Every call goes through `run_agent_with_backoff(...)` for robust 429 handling.

Some agents are **analytical**, such as dtype_inference_agent, completeness_analysis_agent, and format_consistency_agent; some are **summarizers**, such as schema_summary_agent, anomaly_summary_agent, cross_column_summary_agent, and duplicate_summary_agent; one agent **generates cleaning code**, column_cleaner_generator_agent; one agent **reviews failed cleaning code**, cleaner_repair_critic_agent; and the last two agents **write the final narrative report**. 

This structure is useful because **each agent has a narrow responsibility**, a strict **JSON output schema**, and a **precise prompt**, making the whole pipeline easier to control, validate, and explain.

| # | Agent | Category | Role | Output model | Pipeline stage |
|---|-------|----------|------|--------------|----------------|
| 1 | `dtype_inference_agent` | Analytical | Infers target pandas dtype and semantic role per column | `DatasetDtypeInference` | Schema |
| 2 | `schema_summary_agent` | Summarizer | Narrates the schema handoff for downstream agents | `SchemaSummaryOutput` | Schema |
| 3 | `completeness_analysis_agent` | Analytical | Detects missing values and placeholders; runs its own code to inspect the profile | `CompletenessAnalysisReport` | Completeness |
| 4 | `format_consistency_agent` | Analytical | Checks whether values in one column follow a consistent format | `ColumnConsistencyReport` | Consistency |
| 5 | `anomaly_summary_agent` | Summarizer | Summarizes numeric outlier and rare-category findings | `AnomalySummaryOutput` | Anomaly |
| 6 | `cross_column_summary_agent` | Summarizer | Summarizes conflicts between pairs or groups of columns | `CrossColumnSummaryOutput` | Cross-column |
| 7 | `duplicate_summary_agent` | Summarizer | Summarizes exact and near-duplicate record groups | `DuplicateSummaryOutput` | Duplicates |
| 8 | `column_cleaner_generator_agent` | Generator | Writes and self-tests a Python cleaning function for one dirty column | `ColumnCleanerProgram` | Cleaning |
| 9 | `cleaner_repair_critic_agent` | Critic | Diagnoses why a generated cleaner failed and prescribes a targeted fix | `CleanerRepairDiagnosis` | Cleaning |
| 10 | `narrative_frontmatter_agent` | Narrative | Writes the report title, executive summary, and recommendations | `NarrativeFrontMatter` | Reporting |
| 11 | `narrative_section_agent` | Narrative | Writes one section of the final quality report | `NarrativeReportSection` | Reporting |


### Shared agent configuration

Before looking at individual agents, it is worth understanding what they all have in common.

**Model and sampling settings**  
All agents use the same `MODEL` constant (`openai-responses:gpt-5.4-nano`), so switching the underlying model for the entire pipeline requires changing one line. Every agent also shares `retries=4`, which tells Pydantic AI to automatically retry the LLM call up to four times if the response does not conform to the expected JSON schema. Temperature is set to `0` across all agents to make outputs deterministic and reproducible; the two narrative agents (`narrative_frontmatter_agent` and `narrative_section_agent`) use `temperature=0.2` to allow slightly more natural prose.

**Observability with Logfire**  
`setup_logfire()` configures [Logfire](https://logfire.pydantic.dev), the observability platform built by the Pydantic team. Calling `logfire.instrument_pydantic_ai()` automatically traces every agent call (recording the prompt, the structured response, token usage, and latency) without any changes to agent code. When the environment variable `LOGFIRE_CAPTURE_HTTPX=1` is set, the raw HTTP requests and responses to the OpenAI API are also captured, which is useful for debugging rate-limit errors or unexpected model behavior.


In [ ]:
# MODEL is the shared OpenAI model used by every agent below.
# setup_logfire() enables structured tracing for pydantic-ai runs.
MODEL = "openai-responses:gpt-5.4-mini"

def setup_logfire() -> None:
    logfire.configure(
        data_dir=Path(__file__).parent / ".logfire",
        service_name="pydantic-dataset-smoke-test",
        service_version="1.0.0",
        environment=os.getenv("LOGFIRE_ENVIRONMENT", "dev"),
        send_to_logfire=os.getenv("LOGFIRE_SEND", "1") == "1",
    )
    logfire.instrument_pydantic_ai()
    if os.getenv("LOGFIRE_CAPTURE_HTTPX") == "1":
        logfire.instrument_httpx(capture_all=True)


### 5.1) `dtype_inference_agent`

**Role:**  
The dtype_inference_agent decides the target cleaned pandas data type for each column.

**Why it exists:**  
It exists because raw datasets often contain messy values, such as numbers stored as strings, mixed date formats, placeholders, or corrupted values. This agent decides what each column should become after cleaning.

**Prompt summary:**  
The prompt tells the agent to rely mainly on parse evidence, dominant sample patterns, and semantic meaning to choose among Int64, Float64, datetime64[ns], string, boolean, or object.

**Where it is used:**  
It is used in the schema and data-type validation stage, before cleaning, so the pipeline knows the intended final type of each column.

In [ ]:
dtype_inference_agent = Agent(
    MODEL,
    name="dtype-inference",
    output_type=PromptedOutput(DatasetDtypeInference),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are a data type inference agent working on real-world dirty datasets provided by NoiPA. "
        "NoiPA is the digital platform of the Ministero dell'Economia e delle Finanze Italiane that manages salaries, timesheets, "
        "and tax/social security obligations for employees of the Italian Public Administration. "
        "It allows users to view payslips and annual tax certifications online, update personal information, and manage "
        "administrative and HR-related procedures.\n\n"

        "You receive a column-by-column profile containing: the column name, sample values, non-null counts, distinct counts, "
        "numeric_parse_pct, datetime_parse_pct, and related profiling evidence. "
        "Your task is to infer the TARGET CLEANED pandas dtype the column SHOULD HAVE after cleaning. "
        "You are NOT describing the raw dirty storage format. "
        "Treat placeholders, formatting noise, unit suffixes, mixed separators, mixed date formats, and a minority of corrupted values "
        "as corruption, not as evidence of the true type. "
        "Always ask yourself: 'If this column were cleaned correctly, what physical pandas dtype should it have?'\n\n"

        "STRICT DECISION PRIORITY:\n"
        "1. First determine the main dtype family from parse evidence and dominant sample pattern: numeric, datetime, boolean, or text.\n"
        "2. Treat minority dirty values, placeholders, and formatting noise as corruption.\n"
        "3. Only after choosing the dtype family, use the column name and semantic meaning to refine numeric_role, string_role, and detected_pattern.\n"
        "4. Never let the column name override strong parse evidence.\n"
        "5. Infer the cleaned target dtype, not the messy ingestion dtype.\n\n"

        "PARSE EVIDENCE STRENGTH:\n"
        "- numeric_parse_pct >= 80: strong evidence for numeric.\n"
        "- datetime_parse_pct >= 60: strong evidence for datetime.\n"
        "- 60 to 79 numeric_parse_pct: moderate numeric evidence; inspect the dominant sample pattern.\n"
        "- 40 to 59 datetime_parse_pct: moderate datetime evidence; inspect the dominant sample pattern.\n"
        "- Below these thresholds: rely more on dominant pattern and semantic meaning.\n\n"

        "HARD DTYPE GATES:\n"
        "- If numeric_parse_pct >= 80 and datetime_parse_pct < 20, you MUST choose Int64 or Float64. Do not choose string, boolean, or object in that case.\n"
        "- If datetime_parse_pct >= 60, default to datetime64[ns].\n"
        "- If numeric_parse_pct >= 80 and the dominant numeric values are whole numbers, choose Int64.\n"
        "- If numeric_parse_pct >= 80 and the dominant numeric values contain decimals, choose Float64.\n"
        "- Do NOT choose string only because raw values are stored as strings.\n"
        "- Do NOT choose string when a numeric or datetime family clearly dominates.\n"
        "- Use object only as a last resort when no single clean dtype family dominates.\n\n"

        "IMPORTANT SPECIAL RULES:\n"
        "- Numeric strings that resemble compact period or date-like encodings such as YYYYMM, YYYYWW, YYYYQ, or similar numeric period keys "
        "should still be typed as Int64 if the intended clean value is a numeric code/period key rather than a true date column.\n"
        "- Infer datetime64[ns] only when the intended clean meaning is an actual date, time, or timestamp field.\n"
        "- Codes made only of digits can still be Int64 if they are true numeric codes.\n"
        "- Use string instead of Int64 only when the values must be preserved as text exactly for business meaning, especially when letters are intrinsic to the code format.\n"
        "- Mixed formatting alone is not enough reason to use string or object.\n\n"

        "Choose the dtype only from: Int64, Float64, datetime64[ns], string, boolean, object.\n\n"

        "DTYPE RULES (physical, not logical):\n"
        "- Int64: the clean column stores whole numbers. Use this even if the numbers are identifiers, codes, flags, or period keys.\n"
        "- Float64: the clean column stores decimal numbers.\n"
        "- datetime64[ns]: the clean column stores actual dates, times, or timestamps.\n"
        "- string: the clean column stores text such as names, descriptions, textual labels, or alphanumeric identifiers containing letters as part of the real format.\n"
        "- boolean: the clean column stores true/false, yes/no, 0/1-style logical values whose intended clean meaning is binary.\n"
        "- object: use only when the column genuinely mixes incompatible clean value types with no dominant pattern.\n\n"

        "ROLE RULES:\n"
        "- numeric_role: set only when dtype is Int64 or Float64.\n"
        "  'measure' = a real quantity used arithmetically (price, count, amount, duration, salary, quantity).\n"
        "  'code' = a numeric identifier or bounded calendar/classification code not primarily used arithmetically "
        "(postal code, region code, month number, year code, period key).\n"
        "  'indicator' = a numeric flag or ordinal encoding (0/1 flag, ordered category, status encoding).\n"
        "- string_role: set only when dtype is string.\n"
        "  'identifier' = codes or IDs that must be preserved exactly as text.\n"
        "  'categorical' = bounded low-cardinality labels.\n"
        "  'name' = person, organization, or place names.\n"
        "  'free_text' = unstructured narrative, notes, comments, or descriptions.\n"
        "- If dtype is not numeric, numeric_role must be null.\n"
        "- If dtype is not string, string_role must be null.\n\n"

        "PATTERN RULE:\n"
        "- detected_pattern must describe the dominant clean VALUE FORMAT, not a generic statistical interpretation.\n"
        "- detected_pattern must name exactly ONE canonical target format. Never output unions such as 'month label / month number', 'A or B', or 'mixed ...'.\n"
        "- Prefer specific structural patterns over vague labels.\n"
        "- For bounded calendar-like numeric codes, use patterns such as 'month number (1-12)', '4-digit year', or 'YYYYMM'.\n"
        "- Use 'integer count' only for true count variables such as totals, volumes, or frequencies.\n"
        "- If the column is a numeric code with a recognizable domain pattern, prefer that code pattern over 'integer count'.\n"

        "RATIONALE RULE:\n"
        "- The rationale must explain the chosen dtype using the strongest evidence.\n"
        "- Explicitly mention which signal dominated: parse percentages, dominant sample pattern, or semantic meaning.\n"
        "- Explicitly say when minority dirty values were treated as corruption.\n"
        "- Keep the rationale concise, evidence-based, and generic.\n\n"

        "OUTPUT RULES:\n"
        "- Return one entry per column in the same order as the input.\n"
        "- Be conservative and consistent.\n"
        "- Do not invent information not supported by the profile.\n"
        "- If strong parse evidence exists, follow it unless there is clear evidence the clean values belong to another dtype family."
    ),
)


### 5.2) `schema_summary_agent`

**Role:**  
The schema_summary_agent reads the local schema facts produced by the schema validation stage and creates a short structured summary.

**Why it exists:**  
It exists to give the next agents a clear handoff about the most important schema issues, such as safe column naming fixes, possible duplicate-semantic columns, and data-type risks.

**Prompt summary:**  
The prompt tells the agent to use only the provided schema facts, avoid inventing new information, and return valid JSON matching the SchemaSummaryOutput structure.

**Where it is used:**  
It is used after schema validation, before later validation or cleaning agents need a concise summary of the schema findings.

In [ ]:
schema_summary_agent = Agent(
    MODEL,
    name="schema-summary",
    output_type=PromptedOutput(SchemaSummaryOutput),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Schema Summary agent from the project orchestration. "
        "Inspect the attached local schema facts document. "
        "Return valid JSON only that matches the SchemaSummaryOutput schema exactly. "
        "Do not use markdown or ask follow-up questions. "
        "Execute only the schema-summary scope from Reply_projects.pdf. "
        "Do not infer new facts and do not alter the provided findings. "
        "Your only job is to write a short, precise downstream handoff summary for later validation or cleaning agents. "
        "Use the provided local facts exactly as given. "
        "Mention: how many safe naming fixes were identified, whether any duplicate-semantic groups need review, "
        "and whether any genuine data-type contradictions need manual verification. "
        "If there are no duplicate-semantic groups or no data-type risks, say that clearly. "
        "Keep the summary concrete and grounded in the provided facts, not generic."
    ),
)


### 5.3) `completeness_analysis_agent`

**Role:**  
The completeness_analysis_agent analyzes missing values, placeholder values, completeness percentages, and sparse columns.

**Why it exists:**  
It exists to identify which columns have missing-data problems and to suggest concrete actions, such as no action, targeted review, placeholder normalization, or possible removal due to sparsity.

**Prompt summary:**  
The prompt tells the agent to inspect the completeness profile, detect nulls and placeholders such as N/A, "-", unknown, and empty strings, and return evidence-based JSON.

**Where it is used:**  
It is used in the completeness analysis stage of the pipeline.


In [ ]:
completeness_analysis_agent = Agent(
    MODEL,
    name="completeness-analysis",
    builtin_tools=[CodeExecutionTool()],
    output_type=PromptedOutput(CompletenessAnalysisReport),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Completeness Analysis agent from the project orchestration. "
        "Always use the code execution tool to inspect the attached completeness profile document. "
        "Return valid JSON only that matches the CompletenessAnalysisReport schema exactly. "
        "Do not use markdown or ask follow-up questions. "
        "Execute only the completeness-analysis scope from Reply_projects.pdf. "
        "Use the provided per-column completeness percentages, missing-like counts, missing-like percentages, placeholder examples, "
        "and overall completeness metrics from the attached document. "
        "Identify columns with missing values, placeholder tokens such as N/A, -, unknown, and empty strings, and flag sparse columns that are almost entirely empty. "
        "Be evidence-based and conservative. "
        "Do not invent row-level details that are not present in the profile. "
        "Set recommended_action to a concrete next step based on the evidence, not a generic label. "
        "If completeness_pct is 100 and missing_like_count is 0, recommended_action must be exactly 'No action needed'. "
        "If sparse_candidate is true, recommended_action should clearly say 'Investigate or consider removal due to sparsity'. "
        "If placeholder examples are present, recommend standardizing placeholder tokens and reviewing upstream data entry. "
        "If completeness is high but not perfect, recommend targeted review of missing or placeholder values in that column. "
        "Do not leave recommended_action empty. "
        "The summary should be a short downstream handoff in plain language: mention overall completeness, how many columns have missing values, "
        "which columns are the main sparse or review targets, and whether placeholder normalization should be part of later cleaning."
    ),
)


### 5.4) `format_consistency_agent`

**Role:**  
The format_consistency_agent checks whether values inside one column follow a consistent format.

**Why it exists:**  
It exists to detect cases where most values follow one dominant format but some values are written differently and could be fixed by a cleaner.

**Prompt summary:**  
The prompt tells the agent to report an issue only when there is a clear dominant format, measurable outliers, and a realistic cleaning strategy.

**Where it is used:**  
It is used in the consistency validation stage, specifically for column-level format consistency checks.


In [ ]:
format_consistency_agent = Agent(
    MODEL,
    name="format-consistency",
    output_type=PromptedOutput(ColumnConsistencyReport),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the column-level Format Consistency agent. "
        "You receive a ColumnFormatFacts document for one column and must decide whether a format inconsistency exists and, if so, describe it precisely for the downstream cleaning agent.\n\n"

        "DECISION RULES:\n"
        "- Return finding=null if machine_format_candidate is false, dominant_shape_pct is below 70%, or inconsistent_rows is 0.\n"
        "- Return finding=null for descriptive, free-text, name, note, or categorical columns â€” content variation is not a format issue.\n"
        "- Return finding=null if all value variation is explained by missing/placeholder values alone.\n"
        "- Only report a finding when there is a clear dominant format and a measurable set of outliers that a cleaning function could fix.\n\n"

        "WHEN YOU REPORT A FINDING:\n"
        "- expected_pattern: describe ONE canonical dominant target format only (e.g. 'YYYYMM', 'YYYY-MM', 'ISO timestamp YYYY-MM-DDTHH:MM:SS.ffffff', 'two-digit zero-padded month 01-12').\n"
        "- expected_pattern must never describe multiple acceptable formats. Do not use words like 'mixed', 'various', 'multiple', 'and', or 'or'.\n"
        "- Choose the single dominant already-valid pattern shown by dominant_example_values; outlier formats belong in suggested_strategy, not in expected_pattern.\n"
        "- Copy ALL values from inconsistent_examples verbatim into example_inconsistent_values â€” do not filter, deduplicate, or summarize. The cleaner needs the full set.\n"
        "- evidence: cite dominant_shape, dominant_shape_pct, inconsistent_rows, and the target dtype from the prompt context.\n"
        "- suggested_strategy: this is the most important field â€” the downstream cleaner reads it as its normalization contract. "
        "List every outlier shape group with 2-3 concrete examples and the exact transformation needed. "
        "Be specific: 'shape YYYY-MM (e.g. 2023-09): remove dash, concatenate to YYYYMM' is good. "
        "'normalize dates' is not acceptable. "
        "If the target dtype is Int64 or Float64, note that the output must be a numeric string with no unit or symbol.\n\n"

        "OUTPUT: valid JSON matching ColumnConsistencyReport. No markdown, no follow-up questions."
    ),
)


### 5.5) `column_cleaner_generator_agent`

**Role:**  
The column_cleaner_generator_agent generates a Python cleaning function for a specific column.

**Why it exists:**  
It exists to turn detected format inconsistencies into executable cleaning code that can normalize dirty values into the expected target format.

**Prompt summary:**  
The prompt tells the agent to read the cleaning request, write a self-contained Python function, test it once using code execution, and return the function plus verification results as JSON.

**Where it is used:**  
It is used in the remediation stage, when the pipeline needs to generate column-specific cleaning functions.


In [ ]:
column_cleaner_generator_agent = Agent(
    MODEL,
    name="column-cleaner-generator",
    builtin_tools=[CodeExecutionTool()],
    output_type=PromptedOutput(ColumnCleanerProgram),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Column Cleaner Generator agent. "
        "Given a ColumnCleaningRequest, produce a verified Python cleaning function.\n\n"

        "STEPS:\n"
        "1. Read the request: expected_pattern, dominant_example_values, example_inconsistent_values, suggested_strategy, target_dtype.\n"
        "2. Write the cleaning function.\n"
        "3. Test it once using the mandatory grouped code template below.\n"
        "4. Return JSON output. If the grouped test failed, return the best current function and report the failures honestly.\n\n"

        "EXECUTION DISCIPLINE:\n"
        "- This agent is responsible for one draft-and-test attempt only.\n"
        "- The outer Python orchestration loop plus the critic agent is the ONLY repair loop.\n"
        "- Use code execution exactly once for one grouped test over ALL dominant and inconsistent examples.\n"
        "- Do not patch and re-run inside the same model run, even if the grouped test exposes an obvious bug.\n"
        "- Work in batches, not in one-value-at-a-time loops.\n"
        "- After the grouped test, stop testing and return JSON.\n"
        "- If failures remain, include them in verification_summary and residual_risks; the host-side validator will route them to the critic.\n"
        "- Do not keep checking equivalent values individually once the grouped test already showed the same failure family.\n"
        "- NEVER load request data from uploaded files, request_data variables, or external files. Copy literals into the code block exactly as instructed.\n"
        "- On repair attempts, NEVER read uploaded files to reconstruct context. The request, previous function, validation failures, and critic diagnosis already contain everything needed.\n"
        "- Use code execution to test the function you just wrote, not to inspect attachments or rebuild the prompt context.\n"
        "- FORBIDDEN: repeated micro-diagnoses of equivalent failing values, repeated rewrites of the same function, or a second code-execution call inside a single run.\n\n"

        "FUNCTION CONTRACT:\n"
        "- One pure Python function, fully self-contained (all imports and helpers inside).\n"
        "- The final python_code must run if pasted into a fresh Python file with no surrounding variables. Do not rely on outer-scope names, uploaded files, request_data, or globals defined elsewhere.\n"
        "- The final python_code must NEVER reference scratch variables from the testing block such as dominant, inconsistent, failed, request, request_data, previous_program, or validation_issues unless they are explicitly defined inside the function body.\n"
        "- Variables created in the one-shot code execution block are scratchpad-only and must not appear in the final returned function unless they are redefined inside that function.\n"
        "- Input: any scalar â€” str, int, float, None, NaN. Output: str or None only.\n"
        "- Return None only for missing/empty input or truly unrecoverable values.\n"
        "- Return the value unchanged if it already matches expected_pattern.\n"
        "- Every dominant_example_value is already valid. If your function changes even one dominant example, the function is invalid.\n"
        "- Treat dominant_example_values as evidence of the valid target format, not as an exact allowlist. "
        "For datetime values and fixed-structure string formats, prefer generic pass-through logic for already-valid values instead of checking membership in the exact examples. "
        "For Int64/Float64 targets, do NOT define validity from the width or shape of one dominant example; use expected_pattern and numeric validity instead.\n"
        "- Every value in example_inconsistent_values must be transformed or explicitly nulled â€” never returned as-is.\n"
        "- suggested_strategy is the authoritative contract â€” implement a handler for every shape group it lists, no exceptions.\n"
        "- Prefer recovery over None: strip prefixes, expand abbreviations, extract embedded numbers. "
        "If a value contains any recoverable information, return it transformed â€” not None.\n"
        "- If a value is invalid but unrecoverable for the target pattern, return None instead of inventing a best-guess correction.\n"

        "- For expected_pattern 'YYYYMM', if the input exposes a recoverable 4-digit year but omits month, default the month to '01' rather than returning None, unless the request explicitly says otherwise.\n"
        "OUTPUT FORMAT BY TARGET DTYPE:\n"
        "- datetime64[ns]: string matching the EXACT strftime format seen in dominant_example_values.\n"
        "- Int64 / Float64: numeric string only â€” no units, no symbols. Use zfill/format for zero-padded outputs.\n"
        "- string: clean text matching expected_pattern.\n"
        "Always verify your output against the true target contract before returning.\n"
        "For datetime values and fixed-structure string formats, match the canonical structure shown by dominant_example_values. "
        "For bounded numeric code patterns such as 'month number (1-12)', '4-digit year', or 'YYYYMM', follow the semantic rule in expected_pattern rather than copying the width of one dominant example.\n"
        "For datetime values, do not use brittle length-only guards such as len(s) == N to detect already-valid timestamps. "
        "Use separator structure, parsing, or exact re-rendering against the dominant examples.\n\n"

        "MANDATORY CANONICAL-VALUE EARLY-EXIT GUARD:\n"
        "The FIRST logical step after handling None/empty input MUST be an already-valid guard. "
        "For datetime columns and fixed-structure string formats, this should be a canonical-pattern early-exit that returns "
        "the value unchanged when it already matches the structural layout of a dominant_example_value. "
        "Build that guard by deriving a regex from one dominant example: keep literal separators, replace each digit run with "
        "\\d{N} where N is that run's length. If s.fullmatch(pattern) returns true, return s immediately â€” do not enter any "
        "delimiter-based branch after that point. "
        "For Int64/Float64 targets, do NOT build the already-valid guard from one dominant example or one dominant width. "
        "Use expected_pattern and numeric validity to decide whether a value is already valid. "
        "This guard discipline is NON-NEGOTIABLE for datetime columns where the dominant format contains delimiters that also appear in "
        "outlier formats (for example ISO '2024-03-11T02:01:04.421' vs Italian '11/03/2024' vs '11-03-2024'). Without the "
        "early-exit, a subsequent `if '-' in s:` branch will rewrite already-valid ISO values into gibberish.\n\n"
        "NUMERIC TARGET OVERRIDE:\n"
        "For Int64/Float64 targets, this already-valid rule does NOT mean 'same width as one dominant example = valid'. "
        "Never infer numeric validity from a single sample like '7'. "
        "Instead, implement the numeric rule from expected_pattern directly. "
        "Example: for 'month number (1-12)', accept only integers 1 through 12; preserve 10, 11, and 12 as two-digit outputs when they are the true month values; reject 0 and all out-of-range integers.\n\n"

        "MUTUALLY EXCLUSIVE BRANCHES:\n"
        "Delimiter-based branches must be mutually exclusive and ordered most-specific first. "
        "Never write `if '<sep>' in s:` above another branch that re-inspects the same separator via split() or a regex that "
        "includes that separator. If two branches could both fire, either merge them or gate the generic one on exact structure "
        "(count of separator occurrences AND digit-group shapes). The host validator rejects any program where a generic "
        "`'<sep>' in s` branch precedes a more specific branch for the same separator. "
        "For datetime/date cleaners, prefer shape-first `re.fullmatch(...)` branches for every source layout, or one consolidated "
        "`s.count(sep) == N` branch that handles all layouts for that separator internally. Do not scatter multiple top-level "
        "branches for the same delimiter.\n\n"

        "CODE EXECUTION â€” MANDATORY TEMPLATE (use this exact structure every time):\n"
        "```python\n"
        "# 1. Define test data as literals â€” NEVER use request_data or any external variable\n"
        "dominant = ['...', '...']       # copy exact values from the request\n"
        "inconsistent = ['...', '...']   # copy exact values from the request\n\n"
        "# 2. Define the function â€” all imports and helpers go inside\n"
        "def clean_COLUMN(value):\n"
        "    import re\n"
        "    if value is None or str(value).strip() == '':\n"
        "        return None\n"
        "    s = str(value).strip()\n\n"
        "    # First preserve already-valid values using structural patterns derived from dominant examples.\n"
        "    canonical_examples = ['...']  # copy dominant examples here as literals\n"
        "    def _structural_regex(example):\n"
        "        parts, cursor = [], 0\n"
        "        for match in re.finditer(r'\\d+', example):\n"
        "            start, end = match.span()\n"
        "            if start > cursor:\n"
        "                parts.append(re.escape(example[cursor:start]))\n"
        "            parts.append(r'\\d{' + str(end - start) + '}')\n"
        "            cursor = end\n"
        "        if cursor < len(example):\n"
        "            parts.append(re.escape(example[cursor:]))\n"
        "        return '^' + ''.join(parts) + '$'\n"
        "    canonical_patterns = [_structural_regex(e) for e in canonical_examples if e and e != '...']\n"
        "    if any(re.fullmatch(pattern, s) for pattern in canonical_patterns):\n"
        "        return s\n\n"
        "    # Then handle outlier formats as shape-specific branches. Avoid broad `if '-' in s:` / `if '/' in s:` guards.\n"
        "    m = re.fullmatch(r'([A-Za-z]{3})-(\\d{4})', s)\n"
        "    if m:\n"
        "        mon, year = m.groups()\n"
        "        # map month abbreviation and render target format\n"
        "        pass\n"
        "    m = re.fullmatch(r'(\\d{4})-(\\d{1,2})', s)\n"
        "    if m:\n"
        "        year, month = m.groups()\n"
        "        # render target format\n"
        "        pass\n"
        "    m = re.fullmatch(r'(\\d{1,2})/(\\d{4})', s)\n"
        "    if m:\n"
        "        month, year = m.groups()\n"
        "        # render target format\n"
        "        pass\n"
        "    return None\n\n"
        "# 3. Run and flag failures explicitly. Do not re-run inside this attempt.\n"
        "failed = []\n"
        "for v in dominant + inconsistent:\n"
        "    result = clean_COLUMN(v)\n"
        "    status = 'OK' if result is not None else 'FAIL(None)'\n"
        "    print(f'{status}  {repr(v):40} -> {repr(result)}')\n"
        "    if v in dominant and result != v:\n"
        "        failed.append(v)\n"
        "    elif result is None and v in inconsistent:\n"
        "        failed.append(v)\n"
        "    elif v in inconsistent and result == v:\n"
        "        failed.append(v)\n"
        "if failed:\n"
        "    print(f'\\nFAILED ({len(failed)}): {failed}')\n"
        "    print('Return the current best program; the host validator and critic will handle repair.')\n"
        "```\n\n"
        "ISOLATION: each execution block is a fresh environment â€” nothing from previous runs survives. "
        "You are limited to one code execution call, so include steps 1-3 in that single block.\n\n"

        "OUTPUT RULES:\n"
        "- Return valid JSON matching ColumnCleanerProgram exactly.\n"
        "- python_code must contain ONLY the function definition â€” no test code, no print statements, no variable assignments, no JSON.\n"
        "- verification_summary, example_transformations, and residual_risks are separate top-level fields â€” never embed them inside python_code.\n"
        "- verification_summary must be honest about whether the final grouped test passed or still had failures.\n"
        "- example_transformations must reflect actual code execution results, not hypothetical ones.\n"
        "- cleaned_value must be a string or null â€” never int or float.\n"
        "- No markdown, no follow-up questions."
    ),
)


### 5.6) `cleaner_repair_critic_agent`

**Role:**  
The cleaner_repair_critic_agent diagnoses host-side validation failures in a generated cleaning function and prescribes the smallest credible repair for the next generator attempt.

**Why it exists:**  
It exists because the generator agent produces code in one attempt and may produce a function that fails host-side validation. Rather than asking the generator to retry blindly, the critic provides a precise root-cause analysis and repair brief, making the repair loop more efficient and targeted.

**Prompt summary:**  
The prompt tells the agent to treat host-side validation issues as ground truth, identify the exact failing guard or branch, and choose between a minimal edit and a targeted rewrite depending on how localized the failure is. It must return a structured diagnosis without writing any code.

**Where it is used:**  
It is used in the generator / critic repair loop inside the cleaning stage, once per failed validation attempt, before the generator retries.


In [ ]:
cleaner_repair_critic_agent = Agent(
    MODEL,
    name="cleaner-repair-critic",
    output_type=PromptedOutput(CleanerRepairDiagnosis),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Column Cleaner Repair Critic. "
        "You receive a structured CleanerRepairContext containing the cleaning request, the previous generated function, "
        "and authoritative host-side validation issues. "
        "Your job is to diagnose the smallest credible repair before another generator attempt. "
        "Do not write code. Do not restate the whole prompt. Return valid JSON only.\n\n"

        "GOAL:\n"
        "- Explain why the previous cleaner failed.\n"
        "- Point to the logical bug location or branch responsible.\n"
        "- Give a precise repair brief that a generator can follow.\n"
        "- Prefer minimal_edit unless the validation issues clearly show the current approach is fundamentally wrong.\n\n"

        "DECISION RULES:\n"
        "- Treat host-side validation issues as ground truth.\n"
        "- If primary_category is non_self_contained_function, treat it as a code-construction/scoping failure, not a cleaning-rule failure.\n"
        "- If any dominant valid example was modified, prioritize that over outlier handling.\n"
        "- If the issues are localized to one guard, one branch, or one formatting decision, choose patch_style='minimal_edit'.\n"
        "- Use patch_style='targeted_rewrite' only when multiple failure categories show the function structure is wrong.\n"
        "- Set should_retry=false only when another retry is unlikely to help because the evidence is contradictory, missing, or the current request is underspecified.\n"
        "- For numeric measures, do not recommend fixed-width padding unless the request explicitly requires it.\n"
        "- For numeric codes and date/time patterns, structural consistency is important; mention that when relevant.\n\n"

        "COMPOSITE FAILURES â€” DO NOT FIXATE ON A SINGLE CATEGORY:\n"
        "When the issue list contains BOTH a 'shadowed_specific_branch' (structural/order bug) AND a 'dominant_value_modified' "
        "(behavioral bug), they are usually the same root cause: a generic delimiter branch appears before the canonical-value "
        "guard and rewrites valid inputs. In that case:\n"
        "  - root_cause MUST explicitly name BOTH: the missing/misplaced canonical early-exit AND the shadowed delimiter branch.\n"
        "  - planned_fix MUST prescribe TWO concrete structural changes, not just 'check valid format first':\n"
        "      1) insert (or move to the top) a structural regex guard derived from the dominant example that returns s unchanged on match;\n"
        "      2) reorder or merge delimiter branches so no generic `'<sep>' in s` branch precedes a more specific branch inspecting the same separator.\n"
        "  - patch_style should be 'targeted_rewrite' when both categories are present â€” a minimal edit is not sufficient.\n"
        "  - priority_issues must list the structural bug first, the behavioral bug second â€” they share one fix.\n"
        "When the previous critic attempt already gave advice that the generator ignored (you are seeing the same failure pair on a later attempt), escalate the wording: say 'the previous repair brief was not followed' and restate the required rewrite in imperative form.\n\n"

        "COMPONENT-ORDER REWRITES (not delimiter swaps):\n"
        "When the failing output has the correct delimiters but the wrong component order â€” e.g. input '11/01/2024' becoming "
        "'11-01-2024T00:00:00.000' when the expected canonical output is '2024-01-11T00:00:00.000' â€” the bug is that the generator "
        "is swapping the separator character on the raw string instead of parsing the components and reassembling them in the "
        "canonical order. DO NOT say 'change the output format to YYYY-MM-DD' â€” that phrasing is ambiguous and the generator will "
        "re-interpret it as another delimiter swap. Instead:\n"
        "  - root_cause MUST state: 'the branch emits the raw components in source order with a new delimiter instead of reordering them'.\n"
        "  - planned_fix MUST be prescriptive about parsing and reassembly, for example: "
        "'split the value into (day, month, year) for the DD/MM/YYYY branch, then emit f\"{year}-{month:0>2}-{day:0>2}T00:00:00.000\"; "
        "never apply str.replace(\"/\", \"-\") on the whole string'.\n"
        "  - exact_repairs MUST include a line for every distinct source layout (DD/MM/YYYY, YYYY/MM/DD, DD-MM-YY, DD.MM.YYYY, "
        "textual months, etc.) showing input â†’ expected_output and the explicit (year, month, day) assignment the generator must produce.\n"
        "  - patch_style='targeted_rewrite'. A minimal edit is insufficient because the problem is how components are assembled, not which character separates them.\n\n"

        "FIELD RULES:\n"
        "- primary_category: choose the most important validation category to fix first.\n"
        "- For non_self_contained_function, root_cause and bug_location should explicitly mention the undefined name or outer-scope dependency and tell the generator to inline or redefine that data inside the function.\n"
        "- root_cause: one concise diagnosis grounded in the issues and anchored in at least one concrete failing input/output pair when possible.\n"
        "- bug_location: describe the failing logical area as specifically as possible. Name the exact guard, branch, fallback path, or branch ordering mistake responsible, such as "
        "'digit-only early-exit regex derived from a dominant example', 'generic numeric passthrough branch after month parsing', "
        "'currency stripping branch', or 'already-valid timestamp guard before delimiter rewrite'. Do not use vague labels like 'format logic'.\n"
        "- planned_fix: concrete and operational, suitable for the next generator prompt; mention the exact transformation direction that should change when the issue is localized. "
        "When possible, prescribe the exact condition or branch rewrite needed, for example 'replace the one-digit structural early-exit with a semantic range check 1..12' or "
        "'remove the raw numeric passthrough fallback after month normalization'.\n"
        "- priority_issues: list 1-3 short issue summaries, most important first.\n"
        "- exact_repairs: provide 1-3 concrete repair examples. Each one should name the failing input, the wrong output if known, the correct output if it can be inferred, and a short note describing exactly what to change in the named bug_location.\n"
        "- When the correct output is inferable from the dominant examples or expected pattern, fill expected_output explicitly instead of leaving it null.\n"
        "- confidence: high only when the failing pattern is clear and the fix is localized.\n\n"

        "OUTPUT:\n"
        "- Return JSON matching CleanerRepairDiagnosis exactly.\n"
        "- No markdown, no code, no follow-up questions."
    ),
)


### 5.7) `anomaly_summary_agent`

**Role:**  
The anomaly_summary_agent summarizes anomaly detection findings.

**Why it exists:**  
It exists to translate anomaly outputs into a clear downstream summary, highlighting the columns with the most important or frequent anomalies.

**Prompt summary:**  
The prompt tells the agent to summarize only the provided findings, distinguish numeric outliers from rare categorical values, and clearly state if no anomalies were found.

**Where it is used:**  
It is used after the anomaly detection stage.


In [ ]:
anomaly_summary_agent = Agent(
    MODEL,
    name="anomaly-summary",
    output_type=PromptedOutput(AnomalySummaryOutput),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Anomaly Detection summary agent from the project orchestration. "
        "Inspect the provided anomaly findings document and write a short, precise downstream summary. "
        "Return valid JSON only that matches the AnomalySummaryOutput schema exactly. "
        "Do not infer new anomalies, do not invent remediation beyond the provided findings, and do not use markdown. "
        "Mention which columns carry the most severe or highest-volume anomalies, distinguish numeric outliers from rare-category findings, "
        "and state clearly when no anomalies were found."
    ),
)


### 5.8) `cross_column_summary_agent`

**Role:**  
The cross_column_summary_agent summarizes validation issues that involve relationships between columns.

**Why it exists:**  
It exists because some data-quality problems cannot be detected by looking at one column alone, such as duplicate-semantic columns, period mismatches, or invalid date relationships.

**Prompt summary:**  
The prompt tells the agent to inspect the cross-column findings, highlight the most severe conflicts, and avoid inventing new checks or facts.

**Where it is used:**  
It is used in the cross-column validation stage, after the pipeline has checked for conflicts and inconsistencies between pairs or groups of columns.

In [ ]:
cross_column_summary_agent = Agent(
    MODEL,
    name="cross-column-summary",
    output_type=PromptedOutput(CrossColumnSummaryOutput),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Cross-Column Validation summary agent from the project orchestration. "
        "Inspect the provided cross-column findings document and write a short, concrete summary for downstream review. "
        "Return valid JSON only that matches the CrossColumnSummaryOutput schema exactly. "
        "Do not infer new checks or facts, and do not use markdown. "
        "Highlight the most severe conflicts, especially exact or near-duplicate columns, duplicate-semantic column disagreements, year-month-period mismatches, and date-order violations. "
        "If there are no cross-column findings, say that explicitly."
    ),
)


### 5.9) `duplicate_summary_agent`

**Role:**  
The duplicate_summary_agent summarizes exact and near-duplicate record findings.

**Why it exists:**  
It exists to explain whether the dataset contains duplicate rows or suspicious near-duplicate groups, without making unsafe deletion decisions automatically.

**Prompt summary:**  
The prompt tells the agent to report duplicate volumes, near-duplicate groups, and the key columns involved, while acknowledging uncertainty.

**Where it is used:**  
It is used in the duplicate detection stage, after the pipeline has identified exact and near-duplicate record groups across the dataset.

In [ ]:
duplicate_summary_agent = Agent(
    MODEL,
    name="duplicate-summary",
    output_type=PromptedOutput(DuplicateSummaryOutput),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Duplicate Detection summary agent from the project orchestration. "
        "Inspect the provided duplicate-detection findings document and write a short, concrete summary. "
        "Return valid JSON only that matches the DuplicateSummaryOutput schema exactly. "
        "Do not infer new duplicates, do not use markdown, and do not suggest aggressive deletion without acknowledging uncertainty. "
        "Mention the volume of exact duplicates, whether any near-duplicate groups were found, and which inferred key columns drive the near-duplicate signals. "
        "If there are no duplicate groups, say that explicitly."
    ),
)


### 5.10) `narrative_frontmatter_agent`

**Role:**  
The narrative_frontmatter_agent writes the front matter of the final quality report.

**Why it exists:**  
It exists to create a professional title, executive summary, and prioritized recommendations based on the final pipeline briefing.

**Prompt summary:**  
The prompt tells the agent to use only the provided briefing, write a clear executive summary, include concrete recommendations, and return valid JSON.

**Where it is used:**  
It is used during final report generation.

In [ ]:
narrative_frontmatter_agent = Agent(
    MODEL,
    name="narrative-frontmatter",
    output_type=PromptedOutput(NarrativeFrontMatter),
    retries=4,
    model_settings={"temperature": 0.2},
    instructions=(
        "You write the front matter for the final quality report. "
        "Use only the attached briefing. Return valid JSON matching NarrativeFrontMatter exactly.\n\n"
        "RULES:\n"
        "- title must include the dataset name.\n"
        "- executive_summary must be 8-12 sentences in professional English.\n"
        "- recommendations must contain at least 3 concrete, prioritized actions in English.\n"
        "- Do not use backticks for ordinary labels, column names, percentages, or example values.\n"
        "- Do not invent facts not present in the briefing.\n"
        "- No markdown outside the JSON fields."
    ),
)


### 5.11) `narrative_section_agent`

**Role:**  
The narrative_section_agent writes one section body of the final dataset quality report.

**Why it exists:**  
It exists to transform technical findings into readable professional report sections that explain the results clearly.

**Prompt summary:**  
The prompt tells the agent to write one markdown-formatted section, use only the provided facts, avoid inventing details, and match the requested heading.

**Where it is used:**  
It is used repeatedly when assembling the final quality report.


In [ ]:
narrative_section_agent = Agent(
    MODEL,
    name="narrative-section",
    output_type=PromptedOutput(NarrativeReportSection),
    retries=4,
    model_settings={"temperature": 0.2},
    instructions=(
        "You write exactly one section body for the final dataset quality report. "
        "Use only the attached section briefing. Return valid JSON matching NarrativeReportSection exactly.\n\n"
        "RULES:\n"
        "- heading must exactly match the requested section heading.\n"
        "- body must be markdown-formatted prose in professional English.\n"
        "- body must be at least 150 words and grounded in the provided facts only.\n"
        "- Use tables or bullet lists when they help clarity, but keep everything inside the body field.\n"
        "- Do not use backticks for ordinary column names, labels, values, row counts, or percentages.\n"
        "- Round percentages to one decimal place unless the briefing explicitly requires a different precision.\n"
        "- Do not invent facts, counts, examples, or file paths.\n"
        "- No markdown outside the JSON fields."
    ),
)


# 6) Validation and bundling functions definitions

This section presents the functions that implement the validation half of the pipeline. Not every function from the production codebase appears here: the cells are limited to the main orchestration functions and the helper functions whose logic is most directly relevant to understanding how each validation stage operates. Functions that are omitted are referenced explicitly wherever they are called, and the production code itself is commented in sufficient detail that the overall flow remains intelligible without them. This is possible because every function in the codebase has a single, well-defined responsibility that can be understood in isolation.

Each code cell contains one complete function definition. The surrounding markdown describes the function's purpose, its inputs and dependencies, and the structured object it returns, so that the code cells can be read as a self-contained reference without requiring familiarity with the production modules.

### 6.1) Schema and dtype inference

The schema stage is the first and most foundational pass of the validation pipeline. It is responsible for establishing a per-column record of inferred dtypes, statistical profiles, naming validity, and any duplicate-name groups — the `SchemaHandoff` that every subsequent stage consumes as its baseline.

Two helper functions are shown before the agent calls: `build_dataset_profile`, which turns the raw dataframe into a collection of per-column statistical snapshots, and `build_schema_issues`, which applies deterministic naming rules to that profile and emits structured findings. Several supporting functions are omitted from the notebook — `normalized_schema_name`, `is_valid_schema_name`, `suggest_schema_name`, `load_schema_handoff`, and `save_schema_handoff` — because their roles are either self-explanatory naming utilities or standard cache read/write helpers. The two agent-facing functions, `run_dtype_inference` and `run_schema_validation`, appear at the end of the subsection and coordinate all of the above into the finished handoff artifact.

#### 6.1.1) Build Dataset Profile

`build_dataset_profile` is the deterministic profiling function that turns a raw dataframe into a structured collection of per-column statistical snapshots. It takes the dataframe, a dataset name, and an optional dict of dtype overrides from the LLM inference step, and returns a `DatasetProfile` whose `columns_profiles` list holds one `ColumnProfile` per column. Each profile records non-null row count, distinct value count, parse-rate percentages for numeric and datetime formats, an empty-like value rate, and a small sample of representative non-null values.

In [ ]:
def build_dataset_profile(df, dataset_name: str, dtype_overrides: dict[str, str] | None = None) -> DatasetProfile:
    """Build a DatasetProfile from a raw DataFrame.
    If dtype_overrides is provided (from the LLM dtype inference agent),
    those values replace the raw pandas dtype for the relevant columns."""
    return DatasetProfile(dataset_name=dataset_name, total_rows=len(df), total_columns=len(df.columns),
        # For each column, compute the profile statistics and apply dtype overrides if available. 
        # The profile includes the agent-inferred dtype, which downstream agents can use for more informed analysis and recommendations.
        columns_profiles=[ColumnProfile(
                column_name=column_name,
                pandas_dtype=(dtype_overrides or {}).get(column_name, str(df[column_name].dtype)), # Use the agent-inferred dtype if available, otherwise fall back to pandas dtype
                non_null_rows=int(df[column_name].notna().sum()), # Count of non-null rows for this column
                distinct_non_null_values=int(df[column_name].nunique(dropna=True)), # Count of distinct non-null values, giving a sense of cardinality
                numeric_parse_pct=compute_numeric_parse_pct(df[column_name]), # Percentage of values that can be parsed as numeric, providing a signal for numeric type inference
                datetime_parse_pct=compute_datetime_parse_pct(df[column_name]), # Percentage of values that can be parsed as datetime, providing a signal for datetime type inference
                empty_like_pct=compute_empty_like_pct(df[column_name]), # Percentage of values that are empty or look empty which can be a signal for missingness or data quality issues
                sample_values=sample_non_null_values(df[column_name]) # A small sample of non-null values for this column
                ) for column_name in df.columns]) # iterate over columns to build their profiles


#### 6.1.2) Schema naming rules and issue detection

`build_schema_issues` applies a set of deterministic naming rules to the per-column entries produced by the schema stage and returns a list of `SchemaIssue` objects. It takes the list of `SchemaColumnEntry` records and the list of `SchemaDuplicateGroup` objects already identified during profiling. For each column whose name violates the lowercase snake_case convention, it emits one high-severity naming issue with a suggested fix. For each duplicate group, it emits one issue per member column, pointing to its peers so that the remediation planner can decide which alias to keep.

In [ ]:
def build_schema_issues(columns: list[SchemaColumnEntry], duplicate_groups: list[SchemaDuplicateGroup]) -> list[SchemaIssue]:
    """Build the list of SchemaIssue entries from naming violations and duplicate column groups."""
    issues: list[SchemaIssue] = []

    # Flag columns whose name violates the lowercase snake_case convention
    for col in columns:
        if not col.naming_valid:
            issues.append(
                SchemaIssue(
                    column_name=col.name,
                    issue_type="naming_standard",
                    severity="high",
                    evidence=col.naming_reason or "Column name violates the lowercase snake_case naming rule.",
                    fix_confidence="high",
                    suggested_fix=col.rename_suggestion or "",
                    suggested_strategy=f"Safe local rename to '{col.rename_suggestion}'.",
                )
            )

    # For each duplicate group, emit one issue per member pointing to its peers
    for group in duplicate_groups:
        for column_name in group.columns:
            peer_columns = [peer for peer in group.columns if peer != column_name]
            peer_text = ", ".join(peer_columns)
            issues.append(
                SchemaIssue(
                    column_name=column_name,
                    issue_type="duplicate_column_semantics",
                    severity="medium",
                    evidence=(
                        f"Normalized schema name matches the peer column group '{group.canonical_name}', "
                        f"suggesting overlap with: {peer_text}."
                    ),
                    fix_confidence="medium",
                    suggested_fix="",
                    suggested_strategy=(
                        f"Compare values, null patterns, and business usage with: {peer_text} before any merge or drop."
                    ),
                )
            )

    return issues


#### 6.1.3) Agent calls

`run_dtype_inference` is the entry point for the LLM-assisted part of the schema stage. It takes a dataset path, loads the raw dataframe, formats representative column samples into a text attachment via `build_dtype_inference_text` and `attach_text_document`, and delegates the inference decision to `dtype_inference_agent` through `run_agent_with_backoff`. The result is a `DatasetDtypeInference` object whose per-column dtype conclusions are passed directly into `run_schema_validation` as overrides.

`run_schema_validation` is the stage-level orchestrator that assembles the finished `SchemaHandoff`. It takes a dataset path and an optional cache-reuse flag: when reuse is enabled, it returns the saved artifact from `Data/.validation_cache/` immediately. Otherwise it loads the dataframe, obtains dtype overrides from `run_dtype_inference`, builds the statistical profile with `build_dataset_profile`, applies the naming helpers to derive rename suggestions, identifies duplicate-name groups, collects all issues with `build_schema_issues`, requests a short narrative summary from `schema_summary_agent`, and persists the result with `save_schema_handoff`. The returned `SchemaHandoff` is the single authoritative source of schema context for completeness analysis, consistency checking, anomaly detection, and cleaner generation.

In [ ]:
def run_dtype_inference(path: Path) -> DatasetDtypeInference:
    """Run the dtype inference agent on the given dataset path and return the inferred dtypes for each column."""
    df = load_dataset_frame(path)
    # Build the text document for the agent: include column names and samples for each column, formatted clearly.
    text = build_dtype_inference_text(df)
    # Attach the text as a document and run the agent with backoff retries. The agent will return a DatasetDtypeInference object.
    prompt = ["Infer the correct pandas dtype for each column based on the attached CSV sample.",attach_text_document(text)]
    print(f"[orchestrator][schema][dtype-inference] dataset='{path.stem}'", file=sys.stderr, flush=True)
    # Run the agent with backoff retries and return the output field, which contains the inferred dtypes.
    result = run_agent_with_backoff(dtype_inference_agent, prompt)
    return result.output

def run_schema_validation(path: Path, reuse_cache: bool = False) -> SchemaHandoff:
    """Run the schema validation agent on the given dataset path and return a SchemaHandoff object containing the analysis results."""
    if reuse_cache:
        # If reuse_cache is True, load the existing handoff from disk instead of re-running the analysis. This allows us to skip expensive agent calls when we just want to inspect or use the results.
        return load_schema_handoff(path)

    df = load_dataset_frame(path)

    # Run dtype inference first to get the agent's best guess at each column's dtype.
    dtype_inference = run_dtype_inference(path)
    # Build a mapping from column name to inferred dtype column for easy lookup.
    dtype_map = {col.column_name: col for col in dtype_inference.columns}
    dtype_overrides = {name: col.pandas_dtype for name, col in dtype_map.items()}

    # Build the dataset profile, which includes statistics and samples for each column. Pass the dtype overrides to ensure the profile uses the agent's inferred dtypes.
    profile = build_dataset_profile(df, path.stem, dtype_overrides=dtype_overrides)

    # Store columns with the same normalized name together to detect potential duplicates.
    duplicate_groups_by_name: dict[str, list[str]] = {}
    for col_name in df.columns:
        canonical = normalized_schema_name(col_name)
        duplicate_groups_by_name.setdefault(canonical, []).append(col_name)
    
    # Only keep groups with more than one column, as singletons are not duplicates.
    duplicate_groups = [SchemaDuplicateGroup(canonical_name=cn, columns=cols) for cn, cols in duplicate_groups_by_name.items() if len(cols) > 1]

    # Build the list of SchemaColumnEntry objects that contain the analysis for each column
    columns: list[SchemaColumnEntry] = []

    # Iterate over the column profiles from the dataset profile 
    for col_profile in profile.columns_profiles:
        name = col_profile.column_name
        dtype_col = dtype_map.get(name)
        # Determine the final pandas dtype and roles for this column, based on the agent's inference and the column profile.
        dtype_inference_choice = _normalize_dtype_inference_choice(name, dtype_col, col_profile)
        columns.append(SchemaColumnEntry(
            name = col_profile.column_name,
            pandas_dtype = dtype_inference_choice[0], # The final pandas dtype to assign to this column, based on the agent's inference and the profile statistics.
            numeric_role = dtype_inference_choice[1], # The numeric role assigned to this column based on the agent's inference and the profile statistics.
            string_role = dtype_inference_choice[2], # The string role assigned to this column based on the agent's inference and the profile statistics.
            detected_pattern = dtype_inference_choice[3], # The pattern detected for this column based on the agent's inference and the profile statistics.
            rationale = dtype_inference_choice[4], # The rationale for the dtype inference choice.
            non_null_rows = col_profile.non_null_rows,
            distinct_non_null_values = col_profile.distinct_non_null_values,
            numeric_parse_pct=col_profile.numeric_parse_pct,
            datetime_parse_pct=col_profile.datetime_parse_pct,
            empty_like_pct=col_profile.empty_like_pct,
            sample_values=col_profile.sample_values,
            naming_valid = is_valid_schema_name(name), # Check if the column name is valid according to the naming rules.
            naming_reason = naming_rule_reason(name) if not is_valid_schema_name(name) else None, # If the name is not valid, provide a reason why it violates the naming rules
            rename_suggestion = suggest_schema_name(name) if not is_valid_schema_name(name) else None,# If the name is not valid, suggest a valid name based on the original.
        ))

    # Build the list of schema issues based on the column analysis and duplicate groups. 
    issues = build_schema_issues(columns, duplicate_groups)

    # Create the SchemaHandoff object that contains the dataset name, column analysis, and detected issues.
    handoff = SchemaHandoff(
        dataset_name=path.stem,
        total_rows=len(df),
        total_columns=len(df.columns),
        columns=columns,
        issues=issues,
        duplicate_groups=duplicate_groups,
    )
    print(f"[orchestrator][schema][summary] dataset='{path.stem}'", file=sys.stderr, flush=True)

    # Run the schema summary agent to generate a concise summary of the schema analysis, which will be included in the handoff for downstream use.
    prompt = f"Summarize the provided schema analysis for dataset {path.stem}. Do not infer new findings."                                                                                                                                                  
    result = run_agent_with_backoff(schema_summary_agent, [prompt, attach_profile_text(handoff)])
    
    # Update the handoff with the generated summary and save it to disk for downstream stages to consume.
    handoff = handoff.model_copy(update={"summary": result.output.summary})
    save_schema_handoff(path, handoff)
    return handoff


### 6.2) Completeness analysis stage

The completeness stage measures how thoroughly each column is populated with genuine values, distinguishing true nulls from placeholder tokens such as `"-"`, `"n/a"`, and `"0"` that carry no informational content. Its output is a `CompletenessAnalysisReport` that records null rates, missing-like token rates, sparse-column candidates, and the full catalogue of placeholder spellings found in the dataset.

One helper function is shown before the agent call: `build_completeness_profile`, which computes per-column missingness statistics locally without involving the model. The supporting helpers `compute_missing_like_mask`, `attach_profile_text`, `load_completeness`, and `save_completeness` are omitted because they are either single-purpose statistical utilities or standard cache I/O helpers. The agent call `run_completeness_analysis` appears at the end of the subsection and coordinates the profile with the model to produce the structured report.

#### 6.2.1) Build completeness profile

`build_completeness_profile` is the deterministic function that turns a raw dataframe and a dataset name into a structured `CompletenessProfile`. For each column it computes null counts, missing-like token counts using `compute_missing_like_mask`, a completeness percentage, and a short list of representative placeholder examples. It also rolls those per-column measurements up into a dataset-wide overall completeness percentage and a deduplicated list of all placeholder spellings encountered. The returned profile is purely observational: it records what was found without interpreting severity or recommending action.

In [ ]:
def build_completeness_profile(df, dataset_name: str) -> CompletenessProfile:
    """Build the dataset-level completeness profile from a raw DataFrame.

    Computes one CompletenessColumnProfile per column, then rolls those values
    up into an overall completeness percentage and a dataset-wide placeholder list.
    """
    column_profiles: list[CompletenessColumnProfile] = []
    total_cells = len(df) * len(df.columns)
    total_present_cells = 0

    for column_name in df.columns:
        series = df[column_name]
        # Missing-like means true nulls plus configured placeholders such as "-", "n/a", and similar tokens.
        missing_like_mask = compute_missing_like_mask(series)
        missing_like_count = int(missing_like_mask.sum())
        total_rows = len(series)
        # "Present" cells exclude any missing-like token because downstream cleaning should treat them as absent values.
        non_missing_like_rows = total_rows - missing_like_count
        total_present_cells += non_missing_like_rows

        completeness_pct = 0.0 if total_rows == 0 else float((non_missing_like_rows / total_rows) * 100)
        missing_like_pct = 0.0 if total_rows == 0 else float((missing_like_count / total_rows) * 100)

        column_profiles.append(
            CompletenessColumnProfile(
                column_name=column_name,
                pandas_dtype=str(series.dtype),  # raw pandas dtype before any cleaning or schema normalization
                total_rows=total_rows,
                non_null_rows=non_missing_like_rows,
                completeness_pct=completeness_pct,
                missing_like_count=missing_like_count,
                missing_like_pct=missing_like_pct,
                placeholder_examples=sample_placeholder_examples(series),
                distinct_non_null_values=int(series[~missing_like_mask].nunique(dropna=True)),  # cardinality of genuinely present values only
            )
        )

    overall_completeness_pct = 0.0 if total_cells == 0 else float((total_present_cells / total_cells) * 100)
    return CompletenessProfile(
        dataset_name=dataset_name,
        total_rows=len(df),
        total_columns=len(df.columns),
        overall_completeness_pct=overall_completeness_pct,
        placeholder_values_detected=detect_placeholder_values(df),
        columns=column_profiles,
    )


#### 6.2.2) Agent call

`run_completeness_analysis` takes a dataset path and an optional cache-reuse flag, and returns a `CompletenessAnalysisReport`. When reuse is enabled, it delegates immediately to `load_completeness`. Otherwise it loads the dataframe, builds the per-column completeness profile with `build_completeness_profile`, formats that profile as a text attachment with `attach_profile_text`, and submits the combined prompt to `completeness_analysis_agent` via `run_agent_with_backoff`. The agent interprets the profile to identify missing values, placeholder tokens, and sparse columns, and structures its findings into the returned report, which is then persisted with `save_completeness`.

In [ ]:
def run_completeness_analysis(path: Path, reuse_cache: bool = False) -> CompletenessAnalysisReport:
    """Build a completeness profile for the dataset and return the agent's structured analysis.
    The profile is computed locally from the raw data, then handed to the agent which
    identifies missing values, placeholder tokens, and sparse columns. The result is cached.
    """
    if reuse_cache:
        return load_completeness(path)
    df = load_dataset_frame(path)
    # Build a per-column completeness profile to attach to the agent prompt
    profile = build_completeness_profile(df, path.stem)
    prompt = [
        (
            f"Analyze the attached completeness profile for dataset {path.stem}. "
            "Use Python in code execution to inspect the profile document. "
            "Use the provided metrics to summarize per-column completeness, detect missing-like and placeholder values, "
            "identify actual placeholder tokens present in the dataset, and flag sparse columns that may be candidates for removal or investigation."
        ),
        attach_profile_text(profile),
    ]
    print(f"[orchestrator][completeness] dataset='{path.stem}'", file=sys.stderr, flush=True)
    result = run_agent_with_backoff(completeness_analysis_agent, prompt)
    report = result.output
    save_completeness(path, report)
    return report


### 6.3) Format consistency validation stage

The format consistency stage examines whether values within a column conform to a single dominant structural pattern. A date column where most values follow `DD/MM/YYYY` but a handful follow `YYYY-MM-DD`, or a numeric code column that mixes zero-padded and unpadded representations, constitute the kind of problem this stage is designed to surface. Its output is a `ConsistencyValidationReport` containing one `FormatConsistencyFinding` per inconsistent column, each carrying the expected pattern, the count of non-conforming rows, representative outlier examples, and a suggested strategy.

Two helper functions are shown before the agent call: `build_column_format_facts`, which extracts the dominant structural shape and outlier evidence for a single column, and `run_column_format_check`, which decides for that column whether a fast deterministic path suffices or whether the model must be consulted. The supporting helpers `build_column_format_profile`, `infer_format_semantic_hint`, `_run_column_format_checks_async`, `load_consistency`, and `save_consistency` are omitted because they are either low-level profiling utilities, async wrappers, or cache I/O helpers. The stage-level entrypoint `run_format_consistency_validation` appears at the end of the subsection.

#### 6.3.1) Build column format facts

`build_column_format_facts` is the deterministic per-column profiling helper for this stage. It takes a dataframe and a column name, removes placeholder tokens from the evidence using `PLACEHOLDER_TOKENS`, applies structural heuristics for temporal, numeric, and identifier columns via `infer_format_semantic_hint`, and returns a `ColumnFormatFacts` object. That object exposes the dominant structural shape, a list of representative dominant-pattern examples, a list of outlier examples that deviate from that shape, and a flag indicating whether the column is a machine-format candidate for which automated cleaning is feasible.

In [ ]:
def build_column_format_facts(df, column_name: str) -> ColumnFormatFacts:
    """Build the format-consistency facts for one column.

    Starts from the raw structural profile, removes placeholder tokens, then
    applies heuristics tailored to temporal, numeric, and identifier columns
    to decide whether the column is a machine-format candidate with actionable
    inconsistencies.
    """
    # build_column_format_profile computes the raw per-column statistics first:
    # parse rates, sample values, and the most common structural shapes.
    profile = build_column_format_profile(df, column_name)
    series = df[column_name]
    rendered = series.dropna().astype(str).str.strip()
    rendered = rendered[rendered != ""]
    normalized = rendered.str.lower()
    # Placeholder tokens count as missingness, not as evidence of a format family.
    rendered = rendered[~normalized.isin(PLACEHOLDER_TOKENS)]

    # infer_format_semantic_hint uses the column name as a lightweight prior so
    # later heuristics distinguish dates/codes/measures from descriptive text.
    semantic_hint = infer_format_semantic_hint(column_name)
    machine_format_candidate = False

    # If nothing remains after filtering null-like values, return a facts object with no dominant pattern.
    if rendered.empty:
        return ColumnFormatFacts(
            column_name=profile.column_name,
            pandas_dtype=profile.pandas_dtype,
            total_rows=profile.total_rows,
            non_null_rows=profile.non_null_rows,
            distinct_non_null_values=profile.distinct_non_null_values,
            numeric_parse_pct=profile.numeric_parse_pct,
            datetime_parse_pct=profile.datetime_parse_pct,
            empty_like_pct=profile.empty_like_pct,
            semantic_hint=semantic_hint,
            machine_format_candidate=machine_format_candidate,
            top_value_shapes=profile.top_value_shapes,
        )

    if semantic_hint == "numeric_amount_or_measure":
        # Separate plain numeric values from contaminated ones such as currency/unit suffixed strings.
        # is_plain_numeric_value separates clean numeric literals from strings that
        # still carry units, currency markers, or other contamination.
        pure_numeric_values = rendered[rendered.apply(is_plain_numeric_value)]
        contaminated_values = rendered[~rendered.apply(is_plain_numeric_value)]

        dominant_example_values = []
        for value in pure_numeric_values:
            if value not in dominant_example_values:
                dominant_example_values.append(value[:80])
            if len(dominant_example_values) >= 5:
                break

        # select_outlier_examples groups the contaminated values by shape and keeps
        # only representative examples so downstream prompts stay compact.
        inconsistent_examples = select_outlier_examples(
            contaminated_values,
            max_shapes=10,
            max_per_shape=10,
            max_total=60,
        )

        inconsistent_rows = len(contaminated_values)
        # Only treat the column as an actionable machine-format issue when the global parse signal is strongly numeric.
        machine_format_candidate = inconsistent_rows > 0 and profile.numeric_parse_pct >= 80

        return ColumnFormatFacts(
            column_name=profile.column_name,
            pandas_dtype=profile.pandas_dtype,
            total_rows=profile.total_rows,
            non_null_rows=profile.non_null_rows,
            distinct_non_null_values=profile.distinct_non_null_values,
            numeric_parse_pct=profile.numeric_parse_pct,
            datetime_parse_pct=profile.datetime_parse_pct,
            empty_like_pct=profile.empty_like_pct,
            semantic_hint=semantic_hint,
            machine_format_candidate=machine_format_candidate,
            dominant_shape="numeric",
            dominant_shape_pct=0.0 if rendered.empty else float((len(pure_numeric_values) / len(rendered)) * 100),
            dominant_example_values=dominant_example_values,
            inconsistent_rows=inconsistent_rows,
            inconsistent_examples=inconsistent_examples,
            top_value_shapes=profile.top_value_shapes,
        )

    # For non-measure columns, define the dominant format as the most common structural shape.
    # value_shape reduces each raw value to a structural signature such as YYYYMM or
    # DIGITx2-SLASH-DIGITx4 so we can reason about dominant vs outlier formats.
    shape_counts = Counter(value_shape(value) for value in rendered)
    dominant_shape, dominant_count = shape_counts.most_common(1)[0]
    dominant_shape_pct = float((dominant_count / len(rendered)) * 100)
    distinct_shape_count = len(shape_counts)

    dominant_example_values: list[str] = []
    for value in rendered:
        # Keep a few concrete examples of the dominant shape for prompt grounding.
        if value_shape(value) == dominant_shape and value not in dominant_example_values:
            dominant_example_values.append(value[:80])
        if len(dominant_example_values) >= 5:
            break

    inconsistent_rows = len(rendered) - dominant_count
    inconsistent_examples: list[FormatOutlierExample] = []
    if inconsistent_rows > 0:
        # Show only non-dominant shapes as outliers; the dominant family is already represented separately.
        # select_outlier_examples keeps only non-dominant shape families here,
        # because the dominant family is already represented separately.
        inconsistent_examples = select_outlier_examples(
            rendered,
            exclude_shape=dominant_shape,
            max_shapes=10,
            max_per_shape=10,
            max_total=60,
        )

    # Heuristic thresholds vary by semantic type because "good enough" consistency differs for dates, codes, and measures.
    # For temporal and identifier columns, numeric_parse_pct >= 85 acts as a secondary gate: a column that is
    # overwhelmingly numeric even when its values span multiple digit-widths (e.g. month numbers 1-12 split
    # across shape "9" and shape "99") should still be flagged. The dominant_shape and its validation target
    # remain unchanged, so zero-padded forms like "03" are still treated as inconsistent against a dominant "9".
    if semantic_hint == "temporal_period":
        machine_format_candidate = (
            (dominant_shape_pct >= 70 or profile.numeric_parse_pct >= 85)
            and inconsistent_rows > 0
            and (profile.datetime_parse_pct >= 20 or profile.numeric_parse_pct >= 70)
        )
    elif semantic_hint == "numeric_amount_or_measure":
        machine_format_candidate = (
            dominant_shape_pct >= 70
            and inconsistent_rows > 0
            and profile.numeric_parse_pct >= 80
        )
    elif semantic_hint == "code_or_identifier":
        machine_format_candidate = (
            (dominant_shape_pct >= 85 or profile.numeric_parse_pct >= 85)
            and inconsistent_rows > 0
            and distinct_shape_count <= 3
        )

    return ColumnFormatFacts(
        column_name=profile.column_name,
        pandas_dtype=profile.pandas_dtype,
        total_rows=profile.total_rows,
        non_null_rows=profile.non_null_rows,
        distinct_non_null_values=profile.distinct_non_null_values,
        numeric_parse_pct=profile.numeric_parse_pct,
        datetime_parse_pct=profile.datetime_parse_pct,
        empty_like_pct=profile.empty_like_pct,
        semantic_hint=semantic_hint,
        machine_format_candidate=machine_format_candidate,
        dominant_shape=dominant_shape,
        dominant_shape_pct=dominant_shape_pct,
        dominant_example_values=dominant_example_values,
        inconsistent_rows=inconsistent_rows,
        inconsistent_examples=inconsistent_examples,
        top_value_shapes=profile.top_value_shapes,
    )


#### 6.3.2) Check one column for format consistency

`run_column_format_check` is the per-column decision point for the consistency stage. It takes a dataframe, a column name, a dataset name, a context label, and an optional `SchemaColumnEntry`. When the schema entry specifies an unambiguous target pattern — or when the column's string role indicates free-form content where format variation is expected — the function returns a `ColumnConsistencyReport` on the fast path without consulting the model. For ambiguous columns it delegates to `format_consistency_agent`, whose structured output is wrapped into the same `ColumnConsistencyReport` that the dataset-level runner aggregates.

In [ ]:
def run_column_format_check(
    df: pd.DataFrame,
    column_name: str,
    dataset_name: str,
    context_label: str,
    schema_entry: SchemaColumnEntry | None = None,
) -> ColumnConsistencyReport:
    """Check a single column for format inconsistencies, using the schema pattern when available.

    Takes the fast path (no LLM) when the schema provides an unambiguous pattern, otherwise
    falls back to the format_consistency_agent to infer the dominant shape.
    """
    # Skip columns whose role indicates free-form content where format variation is expected
    if schema_entry is not None and schema_entry.string_role in ("name", "free_text"):
        return ColumnConsistencyReport(
            finding=None,
            summary=(
                f"Column '{column_name}' skipped: string_role='{schema_entry.string_role}' "
                f"is not a machine-format candidate."
            ),
        )

    # build_column_format_facts profiles the column locally: dominant shape,
    # representative valid examples, outlier examples, and whether the column
    # even looks like a machine-format field worth validating further.
    format_facts = build_column_format_facts(df, column_name)

    # Schema-driven override: for numeric dtype columns with a concrete, unambiguous
    # schema pattern, bypass the shape-based machine_format_candidate heuristic.
    # The schema's detected_pattern is a direct answer to "is there a machine-enforceable
    # format?", making the heuristic redundant for these columns. This covers columns
    # whose names don't match the semantic-hint vocabulary (e.g. "revenue", "expenses")
    # as well as temporal columns where values split across multiple digit-widths
    # (e.g. month numbers 1-12 split across shapes "9" and "99").
    schema_numeric_override = (
        schema_entry is not None
        and schema_entry.detected_pattern
        and not _schema_pattern_is_ambiguous(schema_entry.detected_pattern)
        and schema_entry.pandas_dtype in {"Int64", "Float64"}
    )

    # If the profiler found no actionable inconsistency and the schema does not
    # force a re-check, stop here and avoid both schema-specific checks and any agent call.
    if not schema_numeric_override and (not format_facts.machine_format_candidate or format_facts.inconsistent_rows <= 0):
        return ColumnConsistencyReport(
            finding=None,
            summary=(
                f"No actionable machine-format inconsistency detected for column '{column_name}' "
                f"in {context_label}."
            ),
        )

    # Fast path: schema already identified the dominant pattern — no LLM needed.
    if (
        schema_entry is not None
        and schema_entry.detected_pattern
        # _schema_pattern_is_ambiguous filters out vague schema labels such as
        # mixed or multi-format descriptions; the fast path only uses one clear
        # target pattern that downstream cleaning can normalize toward.
        and not _schema_pattern_is_ambiguous(schema_entry.detected_pattern)
    ):
        inconsistent_rows = format_facts.inconsistent_rows
        inconsistent_example_profiles = format_facts.inconsistent_examples
        used_schema_override = False

        # _profile_schema_guided_inconsistencies is the stricter schema-guided
        # numeric check: when schema already knows the target numeric pattern,
        # validate values directly against that representation instead of only
        # relying on broad shape groups.
        guided_profile = _profile_schema_guided_inconsistencies(df, column_name, schema_entry)
        if guided_profile is not None:
            inconsistent_rows, inconsistent_example_profiles = guided_profile
            used_schema_override = True

        if inconsistent_rows <= 0:
            return ColumnConsistencyReport(
                finding=None,
                summary=(
                    f"No actionable machine-format inconsistency detected for column '{column_name}' "
                    f"against schema pattern '{schema_entry.detected_pattern}' in {context_label}."
                ),
            )

        # FormatConsistencyFinding stores raw example strings, so strip the
        # richer FormatOutlierExample objects down to their value field here.
        inconsistent_examples = [ex.value for ex in inconsistent_example_profiles]
        evidence = (
            f"Schema handoff identified dominant pattern '{schema_entry.detected_pattern}'. "
            f"Profiler found {inconsistent_rows} rows deviating from the "
            f"dominant shape '{format_facts.dominant_shape}' "
            f"({format_facts.dominant_shape_pct:.1f}% of non-null rows match the dominant shape)."
        )
        if used_schema_override:
            evidence = (
                f"Schema handoff identified target dtype '{schema_entry.pandas_dtype}' and pattern "
                f"'{schema_entry.detected_pattern}'. Schema-guided validation found "
                f"{inconsistent_rows} rows that do not match the target numeric representation."
            )

        return ColumnConsistencyReport(
            finding=FormatConsistencyFinding(
                column_name=column_name,
                expected_pattern=schema_entry.detected_pattern,
                inconsistent_rows=inconsistent_rows,
                example_inconsistent_values=inconsistent_examples,
                evidence=evidence,
                # _build_suggested_strategy converts the grouped outlier shapes
                # into a concrete normalization brief that the cleaner generator
                # will later use as its implementation contract.
                suggested_strategy=_build_suggested_strategy(
                    schema_entry.detected_pattern,
                    format_facts.dominant_shape,
                    inconsistent_example_profiles,
                    format_facts.dominant_example_values,
                    # numeric_pattern_allows_variable_width relaxes the shape
                    # requirement for numeric targets like measures, where valid
                    # values may naturally have different widths.
                    allow_variable_numeric_width=numeric_pattern_allows_variable_width(
                        pandas_dtype=schema_entry.pandas_dtype,
                        numeric_role=schema_entry.numeric_role,
                        detected_pattern=schema_entry.detected_pattern,
                    ),
                ),
            ),
            summary=(
                f"Column '{column_name}': {inconsistent_rows} inconsistent rows detected "
                f"against schema pattern '{schema_entry.detected_pattern}' (no LLM call)."
            ),
        )

    # Slow path: schema does not give us one safe canonical pattern, so defer to
    # the format-consistency agent to decide whether the observed variation is a
    # genuine cleanable inconsistency or acceptable heterogeneity.
    # Include dtype and semantic-role hints in the prompt when schema exists.
    schema_context = ""
    if schema_entry is not None:
        schema_context = (
            f" Target dtype: {schema_entry.pandas_dtype}."
            f" Semantic role: {schema_entry.numeric_role or schema_entry.string_role or 'unknown'}."
        )
    prompt = [
        (
            f"Analyze the attached ColumnFormatFacts for dataset '{dataset_name}', column '{column_name}'."
            f"{schema_context}"
            f" Total rows: {len(df)}."
            f" Dominant shape: '{format_facts.dominant_shape}' ({format_facts.dominant_shape_pct:.1f}% of non-null rows)."
            f" Inconsistent rows: {format_facts.inconsistent_rows}."
            " Determine whether a format inconsistency exists that a cleaning function could fix."
            " If yes, write a precise suggested_strategy listing every outlier shape with concrete transformation steps."
            " Return finding=null if the column is consistent or format consistency is not applicable."
        ),
        # attach_profile_text serializes ColumnFormatFacts into the compact text
        # attachment that the agent reads as its evidence bundle.
        attach_profile_text(format_facts),
    ]
    print(
        f"[orchestrator][consistency][format-agent] dataset='{dataset_name}' column='{column_name}'",
        file=sys.stderr,
        flush=True,
    )
    # run_agent_with_backoff standardizes retries around the agent call so a
    # transient model/runtime failure does not abort the full dataset stage.
    result = run_agent_with_backoff(format_consistency_agent, prompt)
    output = result.output
    # Discard findings where the agent reported zero inconsistent rows
    if output.finding is not None and output.finding.inconsistent_rows <= 0:
        return ColumnConsistencyReport(finding=None, summary=output.summary)
    # Normalise the expected_pattern if the agent returned a vague or multi-valued description
    if output.finding is not None:
        # _normalize_expected_pattern cleans vague agent labels back into one
        # canonical pattern string before downstream stages consume the finding.
        normalized_pattern = _normalize_expected_pattern(output.finding.expected_pattern, format_facts)
        if normalized_pattern != output.finding.expected_pattern:
            output = output.model_copy(
                update={"finding": output.finding.model_copy(update={"expected_pattern": normalized_pattern})}
            )
    return output


#### 6.3.3) Agent call

`run_format_consistency_validation` is the dataset-level entrypoint for this stage. It takes a dataset path and several optional flags: a cache-reuse flag, a flag to load the dataset as raw strings, a worker count for parallel execution, and an optional schema path. When reuse is enabled, it returns the saved `ConsistencyValidationReport` from `load_consistency`. Otherwise it loads the dataframe, optionally loads schema context to supply the fast path with known patterns, and applies `run_column_format_check` to every column — concurrently when `max_workers > 1` via `_run_column_format_checks_async`. The combined findings are assembled into a `ConsistencyValidationReport` and persisted with `save_consistency`. This report is the primary input to cleaner generation and post-cleaning verification.

In [ ]:
def run_format_consistency_validation(
    path: Path,
    reuse_cache: bool = False,
    read_as_str: bool = False,
    max_workers: int = 1,
    schema_path: Path | None = None,
) -> ConsistencyValidationReport:
    """Run format consistency checks for all columns and return the combined report.

    When max_workers > 1, columns are checked concurrently using async tasks. Schema context
    is loaded if available to enable the fast path for columns with known patterns.
    """
    if max_workers < 1:
        raise ValueError("max_workers must be at least 1.")
    if reuse_cache:
        return load_consistency(path)
    df = load_dataset_frame(path, dtype=str if read_as_str else None)
    format_findings: list[FormatConsistencyFinding] = []

    # Load schema to provide pattern and dtype context to each column check.
    # Verification reads the cleaned CSV but must still use the original dataset's
    # schema cache, so callers can pass schema_path separately from the data path.
    handoff = load_schema_handoff(schema_path or path)
    schema_map = {col.name: col for col in handoff.columns}

    column_names = list(df.columns)
    # Use the synchronous path for single-worker runs to keep the event loop simple
    if max_workers == 1 or len(column_names) <= 1:
        reports = []
        for column_name in column_names:
            schema_entry = schema_map.get(column_name)
            # Compute the report for this column and append it to the list of reports
            reports.append(run_column_format_check(df, column_name, path.stem, "original validation", schema_entry=schema_entry))
    else:
        # Cap the worker count so we never spawn more tasks than there are columns
        worker_count = min(max_workers, len(column_names))
        print(
            f"[orchestrator][consistency] running {len(column_names)} column checks with {worker_count} async workers",
            file=sys.stderr,
            flush=True,
        )
        # Run all column checks concurrently and collect the reports as they complete, then reorder them to match the original column order
        reports = asyncio.run(_run_column_format_checks_async(df, column_names, path.stem, schema_map, worker_count))

    # Collect only the columns that actually have a finding
    for result in reports:
        if result.finding is not None:
            format_findings.append(result.finding)

    # Build the final consistency report with a summary of the findings
    report = ConsistencyValidationReport(
        dataset_name=path.stem,
        total_rows=len(df),
        format_consistency_findings=format_findings,
        summary=(
            f"Analyzed all {len(df.columns)} columns individually for format consistency and "
            f"detected {len(format_findings)} format issues."
        ),
    )
    # Save it to disk for downstream stages to consume.
    save_consistency(path, report)
    return report


### 6.4) Anomaly detection stage

The anomaly detection stage surfaces two classes of suspicious values that sit within a column rather than violating its format: numeric outliers detected by a robust IQR rule, and rare category labels whose frequency is low enough to suggest a spelling variant, a stale code, or a data entry error. Its output is an `AnomalyDetectionReport` containing one `AnomalyFinding` per suspicious column, together with a short narrative summary.

This stage is almost entirely deterministic. Two helper functions are shown: `detect_numeric_outlier_candidates`, which applies the IQR heuristic to measure-like numeric columns, and `detect_rare_category_candidates`, which scans low-to-moderate-cardinality categorical columns for suspiciously infrequent labels. The model is involved only in the stage-level entrypoint `run_anomaly_detection`, which narrates the deterministic findings rather than producing them.

#### 6.4.1) Detect numeric outlier candidates

`detect_numeric_outlier_candidates` takes a dataframe and the list of `SchemaColumnEntry` objects from the schema handoff, and returns a list of raw finding dicts. It restricts itself to columns whose dtype is `Int64` or `Float64` and whose numeric role is not `code` or `indicator`, since discrete codes should not be subject to continuous-distribution outlier tests. For eligible columns it normalises the series with `_series_to_numeric`, requires a minimum sample size and value spread, then applies a conservative IQR fence to identify values that fall outside the expected range. Only rows whose values cross that fence are included in the returned evidence.

In [ ]:
def detect_numeric_outlier_candidates(df: pd.DataFrame, schema_columns: list[Any]) -> list[dict[str, Any]]:
    """Detect robust-IQR numeric outliers for measure-like numeric columns only."""
    findings: list[dict[str, Any]] = []
    for column in schema_columns:
        # Codes and indicators may be numeric but should not be judged as continuous measures.
        if column.pandas_dtype not in {"Int64", "Float64"}:
            continue
        if column.numeric_role in {"code", "indicator"}:
            continue

        # _series_to_numeric normalizes decimal commas and other light formatting
        # noise before coercing the column into a numeric series for outlier checks.
        numeric = _series_to_numeric(df[column.name]).dropna()
        # Require enough sample size and spread to avoid noisy outlier labels on tiny columns.
        if len(numeric) < 20 or numeric.nunique() < 10:
            continue

        q1 = float(numeric.quantile(0.25))
        q3 = float(numeric.quantile(0.75))
        iqr = q3 - q1
        if iqr == 0:
            continue

        # Use a conservative 3*IQR band to reduce false positives on naturally skewed public-data distributions.
        lower = q1 - 3 * iqr
        upper = q3 + 3 * iqr
        mask = (numeric < lower) | (numeric > upper)
        outlier_index = list(numeric[mask].index)
        if not outlier_index:
            continue

        # render_cell_text preserves the human-readable raw value so the anomaly
        # finding shows examples exactly as they appeared in the dataset.
        raw_examples = [
            render_cell_text(df.at[index, column.name])
            for index in outlier_index[:10]
        ]
        example_values = [value for value in raw_examples if value is not None]
        severity = "high" if len(outlier_index) / max(len(df), 1) >= 0.02 else "medium"
        findings.append(
            {
                "column_name": column.name,
                "anomaly_type": "numeric_outlier",
                "severity": severity,
                "affected_rows": len(outlier_index),
                "example_values": list(dict.fromkeys(example_values))[:8],
                "evidence": (
                    f"{len(outlier_index)} rows fall outside the robust IQR band "
                    f"[{lower:.3f}, {upper:.3f}] computed from Q1={q1:.3f}, Q3={q3:.3f}."
                ),
                "suggested_action": (
                    "Review whether these values are genuine extreme cases or unit/format errors before imputation or removal."
                ),
            }
        )

    return findings


#### 6.4.2) Detect rare category candidates

`detect_rare_category_candidates` takes a dataframe and the list of `SchemaColumnEntry` objects and returns a list of raw finding dicts. It restricts itself to string or object columns whose role is not `free_text`, `name`, or `identifier`, since high-cardinality descriptive fields are expected to contain many distinct values. For eligible columns it normalises the series to trimmed strings, excludes placeholder tokens, and computes value frequencies. Labels whose relative frequency falls below a configurable rarity threshold and whose absolute count is small enough to warrant review are collected into the returned evidence.

In [ ]:
def detect_rare_category_candidates(df: pd.DataFrame, schema_columns: list[Any]) -> list[dict[str, Any]]:
    """Detect suspiciously rare labels in low-to-moderate-cardinality categorical columns."""
    findings: list[dict[str, Any]] = []
    for column in schema_columns:
        # Free text, names, and identifiers are expected to have many unique values and should not trigger this rule.
        if column.pandas_dtype not in {"string", "object"}:
            continue
        if column.string_role in {"free_text", "name", "identifier"}:
            continue

        # Render the column as trimmed strings first so frequency counting and
        # placeholder filtering operate on one normalized text representation.
        rendered = df[column.name].dropna().astype(str).str.strip()
        rendered = rendered[rendered != ""]
        if rendered.empty:
            continue

        normalized = rendered.str.lower()
        rendered = rendered[~normalized.isin(PLACEHOLDER_TOKENS)]
        if rendered.empty:
            continue

        distinct = rendered.nunique()
        distinct_ratio = distinct / len(rendered)
        # Skip columns that are too small, too high-cardinality, or too diverse for rare-category heuristics to be meaningful.
        if distinct < 5 or distinct_ratio > 0.2 or distinct > 50:
            continue

        # Counter gives the empirical category distribution that the rare-label
        # heuristic uses to decide what counts as an outlier category.
        counts = Counter(rendered)
        # If no category is even moderately common, the column does not have a stable baseline to define "rare" against.
        if counts and (counts.most_common(1)[0][1] / len(rendered)) < 0.2:
            continue
        rare_threshold = max(1, math.floor(len(rendered) * 0.005))
        rare_values = [value for value, count in counts.items() if count <= rare_threshold]
        if not rare_values:
            continue

        affected_rows = sum(counts[value] for value in rare_values)
        example_values = sorted(rare_values, key=lambda value: (counts[value], value))[:8]
        severity = "medium" if affected_rows <= 5 else "low"
        findings.append(
            {
                "column_name": column.name,
                "anomaly_type": "rare_category",
                "severity": severity,
                "affected_rows": affected_rows,
                "example_values": example_values,
                "evidence": (
                    f"{len(rare_values)} rare categories occur at or below {rare_threshold} row(s) each "
                    f"out of {len(rendered)} non-null values."
                ),
                "suggested_action": (
                    "Review whether these rare labels are valid edge cases, spelling variants, or categories that should be consolidated."
                ),
            }
        )

    return findings


#### 6.4.3) Agent call

`run_anomaly_detection` takes a dataset path and an optional cache-reuse flag, and returns an `AnomalyDetectionReport`. When reuse is enabled, it returns the saved artifact from `load_anomaly`. Otherwise it loads the dataframe and the schema handoff, uses `_duplicate_semantic_suppressed_columns` to identify alias columns that should not produce duplicate anomaly findings, and applies both `detect_numeric_outlier_candidates` and `detect_rare_category_candidates` to the non-suppressed columns. The combined findings are sorted and passed as a text attachment to `anomaly_summary_agent`, which returns a short narrative summary. The completed report is saved with `save_anomaly` before being returned.

In [ ]:
def run_anomaly_detection(path: Path, reuse_cache: bool = False) -> AnomalyDetectionReport:
    """Run heuristic anomaly detection and return a structured report with an agent-written summary.

    Findings are built entirely from local heuristics â€” the agent only narrates the result.
    Schema information is loaded if available to suppress duplicate-semantic column aliases.
    """
    if reuse_cache:
        # load_anomaly returns the cached report when we only want to inspect the
        # prior anomaly result instead of recomputing the heuristics and summary.
        return load_anomaly(path)

    # load_dataset_frame reads the raw CSV that the anomaly heuristics will scan.
    df = load_dataset_frame(path)
    # Load the schema handoff to get the list of columns and duplicate groups for suppression
    handoff = load_schema_handoff(path)
    schema_columns = handoff.columns
    # Get the set of column names that should be suppressed in anomaly reporting due to being duplicate-semantic aliases of a preferred column
    # _duplicate_semantic_suppressed_columns keeps only one preferred alias from
    # each duplicate-semantic group so anomaly findings are not duplicated.
    suppressed_columns = _duplicate_semantic_suppressed_columns(handoff.columns, handoff.duplicate_groups)

    # Run both detectors and filter out suppressed duplicate-semantic aliases
    findings = [
        AnomalyFinding(**finding)
        for finding in (
            # detect_numeric_outlier_candidates handles measure-like numeric columns.
            detect_numeric_outlier_candidates(df, schema_columns)
            # detect_rare_category_candidates handles low-to-moderate-cardinality categorical columns.
            + detect_rare_category_candidates(df, schema_columns)
        )
        if finding["column_name"] not in suppressed_columns
    ]
    # Sort by volume descending so the most impactful findings appear first
    findings.sort(key=lambda finding: (-finding.affected_rows, finding.column_name, finding.anomaly_type))
    fallback_summary = (
        f"Detected {len(findings)} anomaly findings across numeric outliers and rare categorical values."
        if findings
        else "No anomaly findings were detected by the current heuristic checks."
    )
    # Build the initial report with findings and a fallback summary, which will be replaced by the agent's summary after narrative generation.
    report = AnomalyDetectionReport(dataset_name=path.stem,total_rows=len(df),total_columns=len(df.columns),findings=findings,summary=fallback_summary)
    # Ask the summary agent to write a human-readable narrative over the already-built findings
    print(f"[orchestrator][anomaly][summary] dataset='{path.stem}'", file=sys.stderr, flush=True)
    # summarize_validation_report gives the summary agent the already-built report
    # plus a fallback summary, then swaps only the summary field on success.
    report = report.model_copy(
        update={
            "summary": summarize_validation_report(
                anomaly_summary_agent,
                (
                    f"Summarize the provided anomaly-detection findings for dataset {path.stem}. "
                    "Do not infer new findings or alter the provided findings."
                ),
                report,
                fallback_summary,
            )
        }
    )
    # Save the final report
    # save_anomaly persists the finished report for remediation planning and later
    # notebook sections that want to inspect anomaly findings directly.
    save_anomaly(path, report)
    return report


### 6.5) Cross-column validation stage

The cross-column stage examines relationships between pairs or groups of columns rather than the values within any single column. It covers four distinct relational checks: semantic duplicate conflicts (pairs of columns that carry the same information under different names), exact and near-duplicate column pairs, year/month/period internal consistency, and date ordering constraints. Its output is a `CrossColumnValidationReport` containing one `CrossColumnFinding` per detected conflict.

Two helper functions are shown: `detect_duplicate_like_columns`, which compares column pairs row by row to identify exact and near-duplicate content, and `detect_year_month_period_mismatches`, which reconstructs an expected `YYYYMM` key and flags rows where year, month, and period columns disagree. Two companion checks, `detect_duplicate_semantic_conflicts` and `detect_date_order_violations`, are omitted from the notebook because they follow the same structural pattern as the shown helpers and add no additional conceptual complexity. The stage-level orchestrator `run_cross_column_validation` coordinates all four checks.

#### 6.5.1) Detect duplicate-like columns

`detect_duplicate_like_columns` takes a dataframe and the schema column list, and returns a list of raw finding dicts. It uses `_schema_column_map` to build a name-to-metadata lookup, then iterates over all pairs of schema-compatible columns, filtering out semantically meaningless comparisons with `_eligible_column_pair`. For each eligible pair it computes row-wise normalised agreement: pairs whose agreement rate meets the exact-duplicate threshold are reported as exact duplicates, while pairs above a lower near-duplicate threshold are reported as near duplicates. Already-seen pairs are tracked to avoid emitting the same finding twice.

In [ ]:
def detect_duplicate_like_columns(df: pd.DataFrame, schema_columns: list[Any]) -> list[dict[str, Any]]:
    """Detect exact and near-duplicate columns based on normalized row-wise agreement."""
    findings: list[dict[str, Any]] = []
    # _schema_column_map converts the schema list into a quick name->metadata lookup
    # so each column pair can cheaply access dtype and role information.
    schema_map = _schema_column_map(schema_columns)
    seen_pairs: set[tuple[str, str]] = set()

    for left_index, left_name in enumerate(df.columns):
        left_meta = schema_map.get(left_name)
        for right_name in df.columns[left_index + 1:]:
            right_meta = schema_map.get(right_name)
            # _eligible_column_pair filters out comparisons that are semantically
            # meaningless, such as mismatched dtype families or free-text columns.
            if not _eligible_column_pair(left_meta, right_meta):
                continue

            pair_key = tuple(sorted((left_name, right_name)))
            if pair_key in seen_pairs:
                continue

            # _normalized_series lowercases and whitespace-normalizes each column so
            # duplicate checks compare the content rather than superficial formatting.
            left_norm = _normalized_series(df, left_name)
            right_norm = _normalized_series(df, right_name)
            # Compare only rows where both columns contain a real, non-placeholder value.
            comparable = (
                left_norm.ne("")
                & right_norm.ne("")
                & ~left_norm.isin(PLACEHOLDER_TOKENS)
                & ~right_norm.isin(PLACEHOLDER_TOKENS)
            )
            comparable_count = int(comparable.sum())
            if comparable_count < 20:
                continue

            left_present = int((left_norm.ne("") & ~left_norm.isin(PLACEHOLDER_TOKENS)).sum())
            right_present = int((right_norm.ne("") & ~right_norm.isin(PLACEHOLDER_TOKENS)).sum())
            # Require strong overlap so we do not compare columns that rarely co-occur.
            overlap_share = comparable_count / max(min(left_present, right_present), 1)
            if overlap_share < 0.8:
                continue

            agreement = comparable & left_norm.eq(right_norm)
            agreement_count = int(agreement.sum())
            similarity_pct = round((agreement_count / comparable_count) * 100, 2)
            mismatch_mask = comparable & ~left_norm.eq(right_norm)
            mismatch_indices = list(df.index[mismatch_mask])

            if agreement_count == comparable_count:
                findings.append(
                    {
                        "columns": [left_name, right_name],
                        "check_type": "exact_duplicate_columns",
                        "severity": "high",
                        "affected_rows": comparable_count,
                        "example_row_indices": list(df.index[comparable])[:8],
                        "similarity_pct": similarity_pct,
                        "evidence": (
                            f"Columns {left_name!r} and {right_name!r} match exactly on all {comparable_count} "
                            f"comparable non-missing rows ({similarity_pct:.2f}% similarity)."
                        ),
                        "suggested_action": (
                            "Review whether one column is redundant and can be dropped, or whether both should remain for lineage reasons."
                        ),
                    }
                )
                seen_pairs.add(pair_key)
                continue

            mismatch_count = len(mismatch_indices)
            if similarity_pct < 95 or mismatch_count > max(10, math.ceil(comparable_count * 0.05)):
                continue

            findings.append(
                {
                    "columns": [left_name, right_name],
                    "check_type": "near_duplicate_columns",
                    "severity": "medium",
                    "affected_rows": comparable_count,
                    "example_row_indices": mismatch_indices[:8],
                    "similarity_pct": similarity_pct,
                    "evidence": (
                        f"Columns {left_name!r} and {right_name!r} agree on {agreement_count} of {comparable_count} "
                        f"comparable non-missing rows ({similarity_pct:.2f}% similarity), with {mismatch_count} mismatches."
                    ),
                    "suggested_action": (
                        "Review the mismatching rows to decide whether one column is a stale copy, a lightly diverged duplicate, or a distinct field."
                    ),
                }
            )
            seen_pairs.add(pair_key)

    return findings


#### 6.5.2) Detect year-month-period mismatches

`detect_year_month_period_mismatches` takes a dataframe and the schema column list, and returns a list of raw finding dicts. It uses `_find_pattern_columns` to locate columns labelled as four-digit year or month-number columns in the schema handoff, and `_looks_like_period_column` to identify `YYYYMM`-style period key columns. For each valid triple of year, month, and period columns it reconstructs an expected period key from the year and month values and compares it against the actual period column, reporting the row indices and example values where the internal relationship is violated.

In [ ]:
def detect_year_month_period_mismatches(df: pd.DataFrame, schema_columns: list[Any]) -> list[dict[str, Any]]:
    """Detect rows where year/month columns disagree with a companion YYYYMM period key."""
    findings: list[dict[str, Any]] = []
    # _find_pattern_columns uses the schema handoff to find columns already labeled
    # as year-like and month-like by the earlier validation stages.
    year_columns = _find_pattern_columns(schema_columns, "4-digit year")
    month_columns = _find_pattern_columns(schema_columns, "month number (1-12)")
    # _looks_like_period_column is a lighter heuristic for YYYYMM-style period keys
    # when the schema metadata suggests a period column but not with one exact label.
    period_columns = [column for column in schema_columns if _looks_like_period_column(column)]

    for year_column in year_columns:
        for month_column in month_columns:
            for period_column in period_columns:
                if len({year_column.name, month_column.name, period_column.name}) < 3:
                    continue

                year_values = pd.to_numeric(df[year_column.name], errors="coerce")
                month_values = pd.to_numeric(df[month_column.name], errors="coerce")
                period_values = df[period_column.name].astype(str).str.strip()
                valid_period = period_values.str.fullmatch(r"\d{6}")
                comparable = year_values.notna() & month_values.notna() & valid_period
                if not comparable.any():
                    continue

                # Rebuild the expected YYYYMM key from year + zero-padded month and compare it with the stored period.
                expected = (
                    year_values[comparable].astype(int).astype(str)
                    + month_values[comparable].astype(int).astype(str).str.zfill(2)
                )
                mismatch_mask = comparable.copy()
                mismatch_mask.loc[comparable] = period_values[comparable].ne(expected)
                mismatch_indices = list(df.index[mismatch_mask])
                if not mismatch_indices:
                    continue

                findings.append(
                    {
                        "columns": [year_column.name, month_column.name, period_column.name],
                        "check_type": "year_month_period_mismatch",
                        "severity": "high",
                        "affected_rows": len(mismatch_indices),
                        "example_row_indices": mismatch_indices[:8],
                        "evidence": (
                            f"Period column {period_column.name!r} disagrees with year/month columns "
                            f"{year_column.name!r}/{month_column.name!r} on {len(mismatch_indices)} comparable rows."
                        ),
                        "suggested_action": (
                            "Check whether the period key should be regenerated from year/month or whether one of the source columns is unreliable."
                        ),
                    }
                )

    return findings


#### 6.5.3) Agent call

`run_cross_column_validation` takes a dataset path and an optional cache-reuse flag, and returns a `CrossColumnValidationReport`. When reuse is enabled, it returns the saved artifact from `load_cross_column`. Otherwise it loads the dataframe and the schema handoff, then applies all four relational detectors — `detect_duplicate_like_columns`, `detect_year_month_period_mismatches`, `detect_duplicate_semantic_conflicts`, and `detect_date_order_violations` — merging their outputs into a single sorted findings list. The combined findings are passed as a text attachment to `cross_column_summary_agent`, which returns a short narrative summary. The finished report is saved with `save_cross_column` before being returned.

In [ ]:
def run_cross_column_validation(path: Path, reuse_cache: bool = False) -> CrossColumnValidationReport:
    """Run all cross-column heuristic checks and return a report with an agent-written summary.

    Schema information is loaded if available to guide duplicate-semantic conflict detection;
    the pipeline continues with empty schema context if the schema stage has not run yet.
    """
    if reuse_cache:
        # load_cross_column returns the cached stage artifact when we want to inspect
        # prior cross-column findings without rerunning the heuristics.
        return load_cross_column(path)

    # load_dataset_frame reads the raw CSV that all relational heuristics will scan.
    df = load_dataset_frame(path)
    # Load schema to provide dtype and duplicate-group context to the detectors
    handoff = load_schema_handoff(path)
    schema_columns = handoff.columns
    duplicate_groups = handoff.duplicate_groups

    # Run all four detectors and merge their findings into a single list
    findings = [
        CrossColumnFinding(**finding)
        for finding in (
            # detect_duplicate_like_columns finds exact and near-overlapping columns.
            detect_duplicate_like_columns(df, schema_columns)
            # detect_duplicate_semantic_conflicts checks duplicate-name column groups
            # for rows where the supposed aliases actually disagree.
            + detect_duplicate_semantic_conflicts(df, duplicate_groups)
            # detect_year_month_period_mismatches checks internal YYYY/MM/period coherence.
            + detect_year_month_period_mismatches(df, schema_columns)
            # detect_date_order_violations checks start/end-style datetime relationships.
            + detect_date_order_violations(df, schema_columns)
        )
    ]
    # Sort by volume descending so the most impactful findings appear first
    findings.sort(key=lambda finding: (-finding.affected_rows, ",".join(finding.columns), finding.check_type))
    fallback_summary = (
        f"Detected {len(findings)} cross-column consistency findings."
        if findings
        else "No cross-column consistency findings were detected by the current rule set."
    )
    # Build the initial report with findings and a fallback summary
    report = CrossColumnValidationReport(
        dataset_name=path.stem,
        total_rows=len(df),
        findings=findings,
        summary=fallback_summary,
    )
    # Ask the summary agent to write a human-readable narrative over the already-built findings
    print(f"[orchestrator][cross-column][summary] dataset='{path.stem}'", file=sys.stderr, flush=True)
    # summarize_validation_report gives the summary agent the structured report plus
    # a deterministic fallback summary, then only replaces the summary text.
    report = report.model_copy(
        update={
            "summary": summarize_validation_report(
                cross_column_summary_agent,
                (
                    f"Summarize the provided cross-column validation findings for dataset {path.stem}. "
                    "Do not infer new findings or alter the provided findings."
                ),
                report,
                fallback_summary,
            )
        }
    )
    # save_cross_column persists the final report for remediation planning and later
    # notebook sections that inspect cross-column findings directly.
    save_cross_column(path, report)
    return report


### 6.6) Duplicate record detection stage

The duplicate record stage identifies groups of rows that are either identical across all columns or share a common business key while differing in one or more other fields. Its output is a `DuplicateDetectionReport` that records both exact duplicate groups and near-duplicate groups, together with a short narrative summary.

One helper function is shown before the agent call: `detect_near_duplicate_groups`, which takes inferred business-key columns and groups rows by their normalised key values. The companion helper `detect_exact_duplicate_groups` is omitted because it is a straightforward pandas deduplication utility that requires no structural explanation. The key inference itself is handled by schema-guided logic inside the stage-level entrypoint `run_duplicate_detection`, which appears at the end of the subsection.

#### 6.6.1) Detect near-duplicate groups

`detect_near_duplicate_groups` takes a dataframe, a list of key column names, and an optional maximum group count, and returns a list of raw near-duplicate group dicts. It uses `normalize_cell_text` to lowercase and whitespace-normalise each key value, groups row indices under the resulting key tuples via a `defaultdict`, and discards groups whose inferred key is entirely empty. Only groups that contain more than one row — and that are not already exact duplicates — are included in the returned evidence, up to the `max_groups` limit.

In [ ]:
def detect_near_duplicate_groups(
    df: pd.DataFrame,
    key_columns: list[str],
    max_groups: int = 25,
) -> list[dict[str, Any]]:
    """Detect groups of rows that share inferred business keys but differ elsewhere."""
    if not key_columns:
        return []

    # defaultdict groups row indices under each inferred business-key signature.
    grouped: dict[tuple[str, ...], list[int]] = defaultdict(list)
    for index, (_, row) in enumerate(df[key_columns].iterrows()):
        # normalize_cell_text lowercases and whitespace-normalizes each key value so
        # grouping is driven by content rather than surface formatting differences.
        key = tuple(normalize_cell_text(value) for value in row.tolist())
        # Skip groups whose inferred key is entirely empty; they are not meaningful identifiers.
        if not any(key):
            continue
        grouped[key].append(index)

    # _normalized_row_signature captures the whole row in normalized form so we can
    # distinguish true near duplicates from groups that are actually exact duplicates.
    full_signatures = [_normalized_row_signature(row) for _, row in df.iterrows()]
    findings: list[dict[str, Any]] = []
    for row_indices in grouped.values():
        if len(row_indices) < 2:
            continue
        unique_rows = {full_signatures[index] for index in row_indices}
        if len(unique_rows) <= 1:
            continue
        findings.append(
            {
                "duplicate_type": "near_duplicate",
                "row_indices": row_indices[:20],
                "key_columns": key_columns,
                "evidence": (
                    f"{len(row_indices)} rows share the inferred key columns {key_columns} but differ elsewhere, "
                    "suggesting potential near-duplicate records."
                ),
                "suggested_action": "Review whether these rows are conflicting duplicates, valid multiple events, or records that need merge rules.",
            }
        )
        if len(findings) >= max_groups:
            break

    return findings


#### 6.6.2) Agent call

`run_duplicate_detection` takes a dataset path and an optional cache-reuse flag, and returns a `DuplicateDetectionReport`. When reuse is enabled, it returns the saved artifact from `load_duplicates`. Otherwise it loads the dataframe and, when available, the schema handoff to infer the logical row key columns for near-duplicate grouping. It applies `detect_exact_duplicate_groups` and `detect_near_duplicate_groups` to those key columns, merges both sets of findings, and passes the combined evidence as a text attachment to `duplicate_summary_agent`. The agent returns a short narrative summary that is included in the saved `DuplicateDetectionReport`.

In [ ]:
def run_duplicate_detection(path: Path, reuse_cache: bool = False) -> DuplicateDetectionReport:
    """Run exact and near-duplicate detection and return a report with an agent-written summary.

    Key columns for near-duplicate grouping are inferred from the schema if available;
    the pipeline falls back to empty schema context if the schema stage has not run yet.
    """
    if reuse_cache:
        # load_duplicates returns the cached duplicate-detection report when we want
        # to inspect prior results instead of rerunning the grouping heuristics.
        return load_duplicates(path)

    # load_dataset_frame reads the raw CSV that both exact and near-duplicate
    # detectors will analyze.
    df = load_dataset_frame(path)
    # Load schema to help infer which columns are likely identifier keys for near-duplicate detection
    try:
        handoff = load_schema_handoff(path)
        schema_columns = handoff.columns
    except FileNotFoundError:
        schema_columns = []

    # Infer the columns most likely to serve as a logical row key
    # infer_duplicate_key_columns chooses the small set of columns most likely to
    # act as a business key for near-duplicate grouping.
    key_columns = infer_duplicate_key_columns(schema_columns, df)
    # Run both detectors and merge their groups into a single list
    groups = [
        DuplicateRecordGroup(**group)
        for group in (
            # detect_exact_duplicate_groups catches rows that are fully identical
            # after normalization across the whole row.
            detect_exact_duplicate_groups(df)
            # detect_near_duplicate_groups catches rows that share the inferred key
            # but still differ somewhere else in the row.
            + detect_near_duplicate_groups(df, key_columns)
        )
    ]
    fallback_summary = (
        f"Detected {len(groups)} duplicate-record groups."
        if groups
        else "No duplicate-record groups were detected by the current exact and near-duplicate checks."
    )
    report = DuplicateDetectionReport(
        dataset_name=path.stem,
        total_rows=len(df),
        groups=groups,
        summary=fallback_summary,
    )
    # Ask the summary agent to write a human-readable narrative over the already-built findings
    print(f"[orchestrator][duplicates][summary] dataset='{path.stem}'", file=sys.stderr, flush=True)
    # summarize_validation_report hands the structured groups to the summary agent
    # and only swaps in the generated summary text on success.
    report = report.model_copy(
        update={
            "summary": summarize_validation_report(
                duplicate_summary_agent,
                (
                    f"Summarize the provided duplicate-detection findings for dataset {path.stem}. "
                    "Do not infer new findings or alter the provided findings."
                ),
                report,
                fallback_summary,
            )
        }
    )
    # save_duplicates persists the finished report for remediation planning and for
    # later notebook cells that inspect duplicate groups directly.
    save_duplicates(path, report)
    return report


### 6.7) Validation results bundling stage

`build_validation_results` is the orchestration checkpoint that converts the six individual validation entrypoints into a single `OrchestrationStepResult`. It takes a dataset path and three boolean cache-reuse flags — one each for schema, completeness, and consistency — and returns the combined result of all six stages.

The function sequences the stages in dependency order: schema runs first because every later stage consumes its cached dtype and pattern context; completeness and consistency follow, each respecting their reuse flag; anomaly, cross-column, and duplicate detection then run against the schema cache produced earlier in the same call, with no reuse flags of their own because they are fast and always depend on up-to-date schema information. The finished `OrchestrationStepResult` groups all six stage outputs into a single object that the cleaning half of the pipeline receives as its authoritative source of validation evidence.

In [ ]:
def build_validation_results(
    path: Path,
    reuse_schema: bool = False,
    reuse_completeness: bool = False,
    reuse_consistency: bool = False,
) -> OrchestrationStepResult:
    """Run all validation stages in order and return the combined result.

    Schema, completeness, and consistency support cache reuse via their respective flags.
    Anomaly, cross-column, and duplicate detection always run fresh as they are fast and
    depend on the schema cache produced earlier in the same run.
    """
    # Schema must run first because later validation stages reuse its cached dtype,
    # pattern, and duplicate-semantic context.
    schema_validation = run_schema_validation(path, reuse_cache=reuse_schema)
    # run_completeness_analysis builds the missingness/placeholder report used later
    # by remediation planning and application.
    completeness_analysis = run_completeness_analysis(path, reuse_cache=reuse_completeness)
    # run_format_consistency_validation produces the per-column findings that later
    # drive cleaner generation and verification.
    consistency_validation = run_format_consistency_validation(path, reuse_cache=reuse_consistency)
    # Anomaly, cross-column, and duplicate stages always run fresh
    print(f"[orchestrator][anomaly] dataset='{path.stem}'", file=sys.stderr, flush=True)
    # run_anomaly_detection adds numeric-outlier and rare-category findings on top
    # of the schema context already built above.
    anomaly_detection = run_anomaly_detection(path)
    print(f"[orchestrator][cross-column] dataset='{path.stem}'", file=sys.stderr, flush=True)
    # run_cross_column_validation checks relationships between columns rather than
    # one column at a time.
    cross_column_validation = run_cross_column_validation(path)
    print(f"[orchestrator][duplicates] dataset='{path.stem}'", file=sys.stderr, flush=True)
    # run_duplicate_detection looks for exact and near-duplicate rows using the
    # schema-guided key inference from the earlier stages.
    duplicate_detection = run_duplicate_detection(path)
    # Collect all stage results into a single object and persist it
    validation_results = OrchestrationStepResult(
        schema_validation=schema_validation,
        completeness_analysis=completeness_analysis,
        consistency_validation=consistency_validation,
        anomaly_detection=anomaly_detection,
        cross_column_validation=cross_column_validation,
        duplicate_detection=duplicate_detection,
    )
    # save_validation_results persists the full multi-stage bundle so the cleaning
    # half can consume one cached object instead of recomputing each stage separately.
    save_validation_results(path, validation_results)
    return validation_results


# 7) Bridge to cleaning and generator / validator / critic loop

Section 7 is the explanation of how the pipeline moves from validation evidence to executable cleaning logic. The code cells below show the real production functions from `cleaning/*`.

The core idea is that **no LLM output is trusted on its own**. For each column flagged as inconsistent, the pipeline asks a generator agent to write a small Python cleaning function. That function is then tested deterministically by a host-side validator, i.e. no model involved, which runs the generated code against real examples from the dataset and checks whether dominant (valid) values are preserved and whether outlier values are actually fixed. Only if that test passes is the function accepted.

When the test fails, the pipeline does not simply retry blindly. A second agent, the **critic**, reads the exact failure report and produces a structured diagnosis: what went wrong, where in the code, and how to fix it. That diagnosis is handed back to the generator as repair guidance. 

This generate → validate → critique → retry cycle repeats up to a configurable number of attempts. The diagram below shows the full flow at a glance.

The sequence is intentional:
- build one request from validation evidence;
- run the generator loop for one column;
- validate the candidate host-side;
- ask the critic for repair guidance when validation fails;
- scale the loop to all inconsistent columns;
- rebuild the deterministic remediation plan;
- apply both the plan actions and the accepted cleaners.


![Cleaner generation flow](Resources/cleaner_generation_flow.png)


The sections that follow show the production code for each step in this order:

1. **7.1**: build a self-contained cleaning request from validation evidence
2. **7.2**: run the generate → validate → critique → retry loop for one column
3. **7.3**: the host-side validator (the deterministic correctness gate)
4. **7.4**: the repair critic (structured diagnosis on failure)
5. **7.5**: scale the loop to all inconsistent columns in the dataset
6. **7.6**: rebuild the deterministic remediation plan (renames, dtype casts, null replacement)
7. **7.7**: apply the plan and all accepted cleaners to produce the final cleaned CSV

### 7.1) Build one `ColumnCleaningRequest`

`build_column_cleaning_request` is the bridge from validation into generation. It packages one format-consistency finding together with the column's deterministic `ColumnFormatFacts` and any schema handoff, so the generator receives a single self-contained `ColumnCleaningRequest` with everything it needs to write the cleaner.

Inside the function, the important steps are:
- deduplicate `finding.example_inconsistent_values` so the prompt stays compact and the same outlier value does not appear multiple times;
- fall back to `format_facts.inconsistent_examples` when the finding did not include explicit outlier examples;
- read `schema_entry.pandas_dtype`, `numeric_role`, and `string_role` so the generator knows the intended output type and the semantic role of the column;
- call `_build_datetime_expected_pattern(...)` and `_augment_datetime_strategy(...)` when the target dtype is datetime-like, giving date columns a stronger normalization contract than the generic pattern alone;
- call `_augment_yyyymm_strategy(...)` when `expected_pattern` is `YYYYMM`, so recoverable values like `Rata 2024` or `Rata 2023` default the missing month to `01` instead of being nulled.


In [ ]:
def build_column_cleaning_request(
    dataset_name: str, 
    column_name: str,
    finding: Any, #This parameter contains the inconsistency finding for the column
    format_facts: Any, #This parameter contains information about the observed format of the column values
    schema_entry: Any | None = None, #This optional parameter contains schema information for the column. If it is not provided, the default is None
) -> ColumnCleaningRequest:
    '''It builds a ColumnCleaningRequest object by combining dataset information, inconsistency findings, format facts, and optional schema details'''
    #It takes the inconsistent example values from finding, removes duplicates while keeping the original order, and converts the result to a list
    example_inconsistent_values = list(dict.fromkeys(finding.example_inconsistent_values))
    #This checks whether the list is empty
    if not example_inconsistent_values:
        #If the list is empty, the code starts building a fallback list of inconsistent example values 
        # by looking at the examples in format_facts.inconsistent_examples, extracting their value attribute, removing duplicates while preserving order, and converting the result to a list
        example_inconsistent_values = list(
            dict.fromkeys(example.value for example in format_facts.inconsistent_examples)
        )
    #If schema_entry exists, this gets its pandas dtype. Otherwise, it stores None
    target_dtype = schema_entry.pandas_dtype if schema_entry else None
    #If schema_entry exists, this gets its numeric_role if it exists; if not (empty or false), it gets its string_role; if schema_entry does not exist, it stores None
    target_role = schema_entry.numeric_role or schema_entry.string_role if schema_entry else None
    #Starts with the expected pattern from the inconsistency finding
    expected_pattern = finding.expected_pattern
    #starts with the suggested cleaning strategy from the finding
    suggested_strategy = finding.suggested_strategy
    # This stays false unless the YYYYMM year-only fallback rule is explicitly activated.
    enforce_year_only_yyyymm_january = False
    #This checks if the target column type is a datetime column based on the schema information.
    # If it is, it applies datetime-specific logic, this replaces the normal expected pattern with a better datetime-specific one
    if target_dtype == "datetime64[ns]":
        expected_pattern = _build_datetime_expected_pattern(format_facts, expected_pattern)
        suggested_strategy = _augment_datetime_strategy(format_facts, suggested_strategy)
    #For YYYYMM targets, add an explicit fallback rule for year-only values.
    #This keeps recoverable examples such as 'Rata 2024' as 202401 instead of dropping them to None.
    elif expected_pattern.strip().lower().startswith("yyyymm"):
        suggested_strategy, enforce_year_only_yyyymm_january = _augment_yyyymm_strategy(
            example_inconsistent_values,
            suggested_strategy,
        )

    #starts creating and returning the final ColumnCleaningRequest object
    return ColumnCleaningRequest(
        dataset_name=dataset_name, 
        column_name=column_name,
        expected_pattern=expected_pattern,
        semantic_hint=format_facts.semantic_hint, #Adds the semantic hint from the format facts
        target_dtype=target_dtype,
        target_role=target_role,
        dominant_shape=format_facts.dominant_shape, #Adds the dominant structural shape of the values
        dominant_example_values=format_facts.dominant_example_values,
        example_inconsistent_values=example_inconsistent_values,
        enforce_year_only_yyyymm_january=enforce_year_only_yyyymm_january,
        suggested_strategy=suggested_strategy,
    )


### 7.2) The per-column generator loop

`run_column_cleaner_program` runs the full generate → validate → critique → retry cycle for one column. Its goal is to obtain a `ColumnCleanerProgram` that passes host-side validation before it is accepted by the pipeline.

For each attempt, the function builds a new generation prompt using the current cleaning request, the previously generated program, the latest validation issues, and, when available, the critic's repair diagnosis. It then calls the `column_cleaner_generator_agent` through `run_agent_with_backoff`. The generated cleaner is not trusted immediately: it is first checked locally through `validate_generated_cleaner_program`. If no validation issues are found, the function reports success and returns `rebuild_verified_program(request, program)`, which attaches verified example transformations to the accepted cleaner.

If validation fails, the function records the failure and checks whether the loop is stagnating. Stagnation is detected when the generator either repeats exactly the same code as in the previous attempt or produces the same validation failure fingerprint. In that case, the stagnation counter is increased. From the next attempt onward, this can activate a stagnation override, slightly increasing the model temperature through `_stagnation_temperature(...)` to encourage the generator to produce a different solution.

When there are attempts remaining, the function calls `run_cleaner_repair_critic(...)`. The critic receives the failed program and the validation issues, then returns a structured diagnosis explaining the likely root cause and whether another retry is worthwhile. If `repair_diagnosis.should_retry` is `False`, the loop stops early and raises a `ValueError` with the critic's explanation and the main validation problems. If all attempts are used without obtaining a valid cleaner, the function raises a final `ValueError` summarizing the first validation issues.

The key internal calls are:

- `_GenerationProgress(on_event)`: wraps the optional progress callback into a small reporter object. Its methods emit progress events such as attempt start, validation failure, accepted generation, stagnation detection, critic start, and critic diagnosis.
- `_build_cleaner_generation_prompt(...)`: builds the generator prompt using the dataset name, the column cleaning request, previous failed output, validation feedback, critic feedback, attempt number, and stagnation flag.
- `_stagnation_temperature(...)`: provides a higher temperature when stagnation has already been detected, encouraging a more different generation in the next attempt.
- `run_agent_with_backoff(column_cleaner_generator_agent, ...)`: calls the generator agent with retry/backoff handling and optional model settings. If the configured usage limits are exceeded, the function converts `UsageLimitExceeded` into a clearer `ValueError`.
- `validate_generated_cleaner_program(...)`: performs local host-side validation of the generated cleaner before it can be accepted.
- `rebuild_verified_program(...)`: rebuilds the accepted cleaner with verified example transformations before returning it.
- `validation_issue_fingerprint(...)`: converts the current validation issues into a compact fingerprint used to detect repeated failures.
- `run_cleaner_repair_critic(...)`: asks the critic agent to diagnose why the generated cleaner failed and whether another retry should be attempted.
- `format_validation_issue(...)`: formats validation issues into readable text for early-stop and final failure errors.

In [ ]:
def run_column_cleaner_program(
    dataset_name: str,
    request: ColumnCleaningRequest,
    max_attempts: int = 10,
    on_event: ProgressCallback | None = None,
) -> ColumnCleanerProgram:
    '''Runs the full generator, validator, and critic retry loop for one column.'''
    progress = _GenerationProgress(on_event) #Create the progress reporter
    previous_program: ColumnCleanerProgram | None = None #At first there is no previous program
    validation_issues: list[CleanerValidationIssue] = [] #Start with no validation issues
    repair_diagnosis: CleanerRepairDiagnosis | None = None #At first there is no critic diagnosis
    last_fingerprint: tuple[str, ...] | None = None #At first there is no fingerprint of previous failures
    consecutive_stagnant = 0 #Start the stagnation counter at 0

    # Try up to max_attempts times to get a cleaner that passes host-side validation
    for attempt in range(1, max_attempts + 1):
        # Once the loop starts repeating, enable the stagnation override behavior
        stagnation = consecutive_stagnant >= 1
        #Report the start of this attempt
        progress.attempt_start(request.column_name, attempt, max_attempts, stagnation)

        # Build the prompt using the latest request, failures, and critic guidance
        prompt = _build_cleaner_generation_prompt(
            dataset_name, request,
            previous_program=previous_program,
            validation_issues=validation_issues,
            repair_diagnosis=repair_diagnosis,
            attempt_number=attempt,
            stagnation_detected=stagnation,
        )
        # When stagnation is detected, slightly increase temperature to encourage a different solution
        model_settings = {"temperature": _stagnation_temperature(consecutive_stagnant)} if stagnation else None
        try:
            # Ask the generator agent for one cleaner program
            program = run_agent_with_backoff(
                column_cleaner_generator_agent, prompt,
                usage_limits=GENERATOR_USAGE_LIMITS,
                model_settings=model_settings,
            ).output
        #If the generator exceeded the allowed single code-execution check, raise ValueError( ... ) from error. Raise a clear user-facing error
        except UsageLimitExceeded as error:
            # The prompt explicitly allows only one code-execution check inside the generator call
            raise ValueError(
                "Generator exceeded the one-code-execution limit. "
                "This prevents hidden self-repair loops; simplify the prompt or raise the explicit limit if needed."
            ) from error

        # Validate the generated cleaner locally without using another model
        validation_issues = validate_generated_cleaner_program(request, program)
        #if there are no validation issues report success
        if not validation_issues:
            progress.generator_accepted(request.column_name, attempt)
            # Attach verified example transformations before returning the accepted program
            return rebuild_verified_program(request, program)
        #If there are issues, report the failure
        progress.validation_failed(request.column_name, attempt, max_attempts, validation_issues[0])

        # Detect whether the model is stuck by comparing the code and the failure fingerprint
        #Build a compact fingerprint of the issues
        fingerprint = validation_issue_fingerprint(validation_issues)
        #Check whether the new code is exactly the same as the previous code
        same_code = previous_program is not None and program.python_code.strip() == previous_program.python_code.strip()
        #Check whether the new set of failures is exactly the same as last time
        repeated_failure = last_fingerprint is not None and fingerprint == last_fingerprint
        if same_code or repeated_failure:
            #Build a human-readable reason
            reason = "repeated the same code" if same_code else "repeated the same host-side failures"
            #Log stagnation
            progress.stagnation_noted(request.column_name, attempt, max_attempts, reason)
            #Increase the stagnation counter
            consecutive_stagnant += 1
        #if there is a progress reseat stagnation counter to 0
        else:
            consecutive_stagnant = 0
        #Store the current program and current failure fingerprint for the next attempt
        previous_program, last_fingerprint = program, fingerprint

        # If attempts remain, ask the critic for a structured repair diagnosis.
        if attempt < max_attempts:
            #Report that the critic is starting
            progress.critic_started(request.column_name, len(validation_issues))
            #Ask the critic to analyze the failed cleaner
            repair_diagnosis = run_cleaner_repair_critic(dataset_name, request, program, validation_issues)
            #Log the criticâ€™s diagnosis
            progress.critic_diagnosis(request.column_name, repair_diagnosis)
            # Stop early if the critic says another retry is not worthwhile
            if not repair_diagnosis.should_retry:
                #Build a readable summary of the first validation issues
                failure_lines = "\n".join(f"- {format_validation_issue(issue)}" for issue in validation_issues[:10])
                #Stop the loop and raise an error explaining why the critic advised against retrying
                raise ValueError(
                    f"Cleaner generation stopped for column '{request.column_name}' because the critic advised against retrying: "
                    f"{repair_diagnosis.root_cause}\n{failure_lines}"
                )

    # If all attempts fail, raise one final error containing the main validation problems
    # Build a final failure summary
    failure_lines = "\n".join(f"- {format_validation_issue(issue)}" for issue in validation_issues[:10])
    #Raise an error saying all attempts failed
    raise ValueError(
        f"Cleaner generation failed local validation for column '{request.column_name}' after {max_attempts} attempts:\n"
        f"{failure_lines}"
    )


### 7.3) The host-side validator

`validate_generated_cleaner_program` is the deterministic correctness gate for generated column cleaners. It does not trust the generator directly: it loads the generated Python code, runs it on the examples contained in the `ColumnCleaningRequest`, and returns structured `CleanerValidationIssue` objects when something is wrong.

The validator checks three main things: the cleaner must target the correct column, preserve already-valid dominant examples exactly, and normalize inconsistent examples into outputs that match the expected shape, dtype, datetime format, and target pattern when applicable. If the generated code cannot be loaded, the function immediately returns a high-severity issue instead of continuing.

The helper calls are easiest to follow in order:

- `load_cleaner_callable(...)`: loads the generated Python code into a callable cleaner and rejects code that cannot be executed safely as a self-contained function.
- `detect_shadowed_delimiter_branches(...)`: performs a static check for problematic delimiter logic, such as unreachable branches caused by overly broad separator conditions.
- `dominant_output_shape(...)`, `requires_fixed_output_shape(...)`, and `dominant_datetime_example(...)`: derive the expected output structure from the dominant examples in the cleaning request.
- `build_validation_issue(...)` and `build_runtime_exception_issue(...)`: convert validation failures and runtime errors into structured `CleanerValidationIssue` objects.
- `matches_dominant_datetime_format(...)`, `_suggest_corrected_datetime(...)`, and `_diagnose_datetime_component_order(...)`: perform datetime-specific validation and generate concrete repair hints when the cleaned value does not match the dominant datetime layout.
- `value_shape(...)`: compares non-datetime cleaned outputs against the dominant structural shape when fixed-shape validation is required.
- `is_parseable_output(...)`: checks whether the cleaned non-`None` output can be parsed as the requested target dtype.
- `matches_request_target_pattern(...)`: checks whether the cleaned output satisfies the expected target pattern defined in the request.

Overall, this function makes the validator, not the generator, the authority on correctness. The generator can propose a cleaning function, but the program is accepted only if it passes these deterministic host-side checks.

In [ ]:
def validate_generated_cleaner_program(
    request: ColumnCleaningRequest, #cleaning request that contains dominant examples, inconsistent examples, target dtype, and expected pattern
    program: ColumnCleanerProgram, #generated cleaner program to validate
) -> list[CleanerValidationIssue]:
    '''Validates a generated cleaner by running it on valid and invalid example values'''
    issues: list[CleanerValidationIssue] = [] #Starts with an empty list of issues

    # First check that the cleaner was generated for the correct column
    if program.column_name != request.column_name:
        issues.append(
            build_validation_issue(
                category="program_mismatch",
                severity="high",
                message=f"Program column_name was {program.column_name!r}, expected {request.column_name!r}.",
                expected_behavior=f"column_name must equal {request.column_name!r}.",
            )
        )

    # It tries to load the cleaner code into a callable Python function
    try:
        cleaner = load_cleaner_callable(program)
    except Exception as error: #If loading fails, catch the error
        # Report NameError separately because it suggests missing outer-scope dependencies
        if isinstance(error, NameError):
            return [
                build_validation_issue(
                    category="non_self_contained_function",
                    severity="high",
                    message=(
                        f"Generated cleaner code could not be loaded because it referenced an undefined name: {error}. "
                        "The function is not self-contained."
                    ),
                    expected_behavior=(
                        "generated python_code must load into a callable cleaner without relying on outer-scope "
                        "variables, scratchpad names, or test-block state."
                    ),
                )
            ]
        # Report any other load failure as a runtime exception
        return [
            build_validation_issue(
                category="runtime_exception",
                severity="high",
                message=f"Generated cleaner code could not be loaded: {error}",
                expected_behavior="generated python_code must load into a callable cleaner without exceptions.",
            )
        ]

    # Add static code issues found before running the cleaner
    issues.extend(detect_shadowed_delimiter_branches(program))

    # Precompute the expected output shape and canonical datetime example
    target_shape = dominant_output_shape(request) #Compute the most common dominant output shape
    require_fixed_shape = requires_fixed_output_shape(request) #Decide whether fixed output shape should be enforced
    target_datetime_example = dominant_datetime_example(request) #Get the canonical dominant datetime example, if one exists

    # Dominant examples are already valid, so the cleaner must preserve them exactly
    for value in request.dominant_example_values:
        try:
            cleaned = cleaner(value) #Apply the generated cleaner to the dominant example value
        except Exception as error:
            #Add a runtime exception issue for that dominant example if the cleaner raised an error when processing it, since dominant examples should be handled without errors
            issues.append(build_runtime_exception_issue(stage_label="Dominant example", input_value=value, error=error))
            continue
        #Convert the cleaned result to string unless it is None, so it can be compared and reported in issues
        cleaned_str = None if cleaned is None else str(cleaned)
        # Flag any dominant example that was changed by the cleaner
        if cleaned != value:
            issues.append(
                build_validation_issue(
                    category="dominant_value_modified",
                    severity="high",
                    message=f"Dominant example {value!r} changed to {cleaned_str!r}; already-valid values must be preserved exactly.",
                    input_value=value,
                    actual_output=cleaned_str,
                    expected_behavior="return the dominant example unchanged.",
                )
            )

    # Now loop through all inconsistent examples. Inconsistent examples should be normalized into the target format.
    for value in request.example_inconsistent_values:
        try:
            cleaned = cleaner(value) #run the cleaner
        except Exception as error:
            #Add a runtime issue
            issues.append(build_runtime_exception_issue(stage_label="Inconsistent example", input_value=value, error=error))
            continue
        #if cleaner returns None,skip further checks for that example
        if cleaned is None:
            continue

        cleaned_str = str(cleaned) #Convert the cleaned output to string
        # Flag any inconsistent example that was returned unchanged
        if cleaned == value:
            issues.append(
                build_validation_issue(
                    category="outlier_unchanged",
                    severity="high",
                    message=f"Inconsistent example {value!r} was returned unchanged.",
                    input_value=value,
                    actual_output=cleaned_str,
                    expected_behavior="the outlier should be normalized, not passed through unchanged.",
                )
            )

        # Only do output-shape validation if fixed shape is required.
        if require_fixed_shape:
            # Datetime outputs must match the canonical datetime layout
            if request.target_dtype == "datetime64[ns]" and target_datetime_example is not None:
                if matches_dominant_datetime_format(cleaned_str, request) is False:
                    # Build extra help text when the datetime component order looks wrong.
                    suggested = _suggest_corrected_datetime(value, target_datetime_example) #Try to build a suggested corrected datetime
                    diagnosis = _diagnose_datetime_component_order(value, cleaned_str, target_datetime_example) #Try to generate a diagnosis of component-order mistakes
                    suggestion_note = f" Correct output for this input: {suggested!r}." if suggested else "" #If a suggested corrected value exists, create extra message text
                    #Add a wrong_output_shape issue because the cleaned datetime does not match the expected canonical format, and include the diagnosis and suggestion in the message to help guide correction
                    issues.append(
                        build_validation_issue(
                            category="wrong_output_shape",
                            severity="medium",
                            message=(
                                f"Inconsistent example {value!r} cleaned to {cleaned_str!r}, which does not match "
                                f"the canonical datetime format used by dominant examples such as {target_datetime_example!r}."
                                f"{diagnosis}{suggestion_note}"
                            ),
                            input_value=value,
                            actual_output=cleaned_str,
                            expected_behavior=(
                                "produce output matching the canonical datetime layout shown by dominant examples, "
                                f"for example {target_datetime_example!r}."
                                + (f" Expected for this input: {suggested!r}." if suggested else "")
                            ),
                        )
                    )
            # For non-datetime values, compare the cleaned shape with the dominant shape
            elif target_shape is not None and value_shape(cleaned_str) != target_shape:
                #If the shape is wrong, add a wrong_output_shape issue that explains the inconsistency and reports the expected dominant shape, so the developer can adjust the code to produce consistent output shapes.
                issues.append(
                    build_validation_issue(
                        category="wrong_output_shape",
                        severity="medium",
                        message=(
                            f"Inconsistent example {value!r} cleaned to {cleaned_str!r} with shape "
                            f"{value_shape(cleaned_str)!r}, expected dominant output shape {target_shape!r}."
                        ),
                        input_value=value,
                        actual_output=cleaned_str,
                        expected_behavior=f"produce output matching the dominant structural shape {target_shape!r}.",
                    )
                )

        # Check that the cleaned value can actually be parsed as the requested dtype
        if not is_parseable_output(cleaned_str, request.target_dtype):
            #If not, add a parsing issue
            issues.append(
                build_validation_issue(
                    category="not_parseable_as_target_dtype",
                    severity="high",
                    message=f"Inconsistent example {value!r} cleaned to {cleaned_str!r}, which is not parseable as {request.target_dtype}.",
                    input_value=value,
                    actual_output=cleaned_str,
                    expected_behavior=f"produce a value parseable as {request.target_dtype}.",
                )
            )
            continue

        # If the request defines a numeric pattern, verify the cleaned value also matches it
        pattern_match = matches_request_target_pattern(cleaned_str, request)
        #If it explicitly fails
        if pattern_match is False:
            #add a pattern mismatch issue because the cleaned value does not conform to the expected numeric pattern defined in the request, 
            #which may indicate that the cleaner is not correctly normalizing numeric formats as required by the schema.
            issues.append(
                build_validation_issue(
                    category="not_matching_target_pattern",
                    severity="high",
                    message=(
                        f"Inconsistent example {value!r} cleaned to {cleaned_str!r}, which does not match "
                        f"the target pattern {request.expected_pattern!r}."
                    ),
                    input_value=value,
                    actual_output=cleaned_str,
                    expected_behavior=f"produce a value matching the target pattern {request.expected_pattern!r}.",
                )
            )

    return issues


### 7.4) The repair critic

When validation fails, the loop does not regenerate blindly. `run_cleaner_repair_critic` packages the cleaning request, the failed cleaner program, and the authoritative validation issues into a `CleanerRepairContext`. It then asks `cleaner_repair_critic_agent` to return a structured `CleanerRepairDiagnosis` that can guide the next generator attempt.

The internal steps are deliberately small:

- build `CleanerRepairContext(...)` so the critic receives the same structured failure state used by the host loop;
- call `attach_profile_text(...)` to serialize that context into a compact prompt attachment;
- invoke `run_agent_with_backoff(cleaner_repair_critic_agent, ...)` and return the agent's structured diagnosis.

The returned diagnosis is then passed by the retry loop into the next generator prompt as targeted repair guidance, helping the generator focus on the actual validation failures instead of producing another blind attempt.

In [ ]:
def run_cleaner_repair_critic(
    dataset_name: str,
    request: ColumnCleaningRequest,
    previous_program: ColumnCleanerProgram,
    validation_issues: list[CleanerValidationIssue],
) -> CleanerRepairDiagnosis:
    '''This defines the synchronous function that calls the critic agent. It calls the critic agent to diagnose why the previous cleaner failed validation.'''
    # Bundle the request, failed program, and validation issues into one structured object
    context = CleanerRepairContext(
        request=request,
        previous_program=previous_program, #Include the failed cleaner
        validation_issues=validation_issues, #Include the validation failures
    )
    # Build a short instruction plus a structured profile document for the critic
    #This first prompt item is a plain instruction telling the critic what to do
    prompt = [
        (
            f"Diagnose the failed cleaner for dataset {dataset_name}, column {request.column_name}. "
            "Use the structured repair context only. "
            "Return a precise repair diagnosis for the next generator attempt."
        ),
        attach_profile_text(context), #This attaches the structured context as a profile/document
    ]
    # Run the critic synchronously and return its structured diagnosis
    return run_agent_with_backoff(cleaner_repair_critic_agent, prompt).output


### 7.5 Generate one cleaner per inconsistent column

Once the single-column loop is clear, `run_cleaner_generation` scales it to the dataset level. It is intentionally a thin orchestration wrapper: validate the run settings, acquire exclusive access to the dataset's cleaning cache, and then hand the real work to the locked inner driver.

The important internal calls are:
- `_generation_lock(path)`: prevents overlapping generation runs from writing into the same dataset cache directory at the same time;
- `_run_cleaner_generation_locked(...)`: performs the actual dataset-level workflow once exclusive access is in place.

That inner locked workflow is the stage that reloads consistency findings, builds one request per inconsistent column, calls `run_column_cleaner_program(...)`, and persists the accepted cleaners plus `cleaner_manifest.json`.


**More in depth:**
`_run_cleaner_generation_locked(...)` contains the real dataset-level workflow executed after the lock is acquired. It loads the format-consistency findings and the dataset, optionally adds schema metadata, filters to a specific column if requested, and builds a `ColumnCleaningRequest` for each target column. It can run the generation sequentially or asynchronously depending on `max_workers`, then saves each accepted cleaner with `save_generated_cleaner(...)` and records it as a `GeneratedCleanerArtifact`.

The function also manages failures at dataset level: if one column fails in a multi-column run, it records the failure and continues; if the requested single column fails, or if every column fails, it raises a `ValueError`. At the end, successful artifacts are written to `cleaner_manifest.json` through `save_cleaner_manifest(...)`.

In [ ]:
def run_cleaner_generation(
    path,
    reuse_consistency: bool = False,
    column_name: str | None = None,
    max_attempts: int = 10,
    max_workers: int = 1,
    on_event: ProgressCallback | None = None,
) -> list[GeneratedCleanerArtifact]:
    '''Runs cleaner generation for one or more columns, protected by a dataset-level lock.'''
    # Validate user-controlled settings before starting any work
    if max_attempts < 1: #Validate the retry count
        raise ValueError("max_attempts must be at least 1.") #Reject invalid retry counts
    if max_workers < 1: #Validate the worker count
        raise ValueError("max_workers must be at least 1.") #Reject invalid worker counts

    # Protect the whole generation process so two runs do not write conflicting files
    with _generation_lock(path): #Acquire the dataset-level lock in order to ensure exclusive access to cache
    #Run the real generation workflow while holding the lock
        return _run_cleaner_generation_locked(path, reuse_consistency, column_name, max_attempts, max_workers, on_event)
    #It validates inputs and ensures the whole generation run is protected by a lock


### 7.6) Bring back the deterministic remediation plan

Accepted generated cleaners are only one part of the remediation stage. `run_remediation_planning(...)` builds or reloads a deterministic `RemediationPlan` from the validation results before the dataset is actually modified. If `reuse_saved_remediation=True`, it first tries to load an existing plan from cache; otherwise, it resolves the validation bundle, builds a new plan, saves it, and returns it.

The core logic is inside `build_remediation_plan(...)`, which converts validation findings into structured `RemediationAction` objects. Schema validation can produce column-renaming actions and dtype-casting actions. Completeness analysis can produce placeholder-to-null replacement actions. Format-consistency findings produce `generate_cleaner` actions for columns that need generated cleaning rules. Cross-column validation can produce automatic exact-duplicate-column drops, while more ambiguous cross-column issues are marked for manual review. Anomaly findings are never auto-applied: high-severity anomalies become manual-review actions, while lower-severity ones are reported only. Duplicate detection can auto-plan removal of exact duplicate rows, while other duplicate groups are sent to manual review.

The important helper calls are:

- `load_remediation_plan(...)`: optional fast path used when `reuse_saved_remediation=True`; if no cached plan exists, the function falls back to building a new one.
- `_resolve_validation_results(...)`: selects the validation bundle to consume, using the caller-provided `validation_results` when available, otherwise loading or rebuilding validation results depending on `reuse_saved_validation`.
- `build_remediation_plan(...)`: converts schema, completeness, consistency, cross-column, anomaly, and duplicate findings into structured remediation actions.
- `save_remediation_plan(...)`: persists the finished plan so later notebook cells or CLI stages can reuse the same remediation decisions without recalculating them.

Before returning, `build_remediation_plan(...)` sorts actions so auto-applied actions come first, followed by action type and action ID for reproducibility. The final `RemediationPlan` contains the dataset name, the ordered action list, and a generated summary.

In [ ]:
def run_remediation_planning(
    path: Path,
    validation_results: OrchestrationStepResult | None = None,
    reuse_saved_validation: bool = False,
    reuse_saved_remediation: bool = False,
) -> RemediationPlan:
    '''The main entry point for remediation planning. It resolves validation results, builds the remediation plan, and handles caching based on the provided flags.'''
    # reuse of saved remediation is allowed, try loading the cached plan; ignore FileNotFoundError and proceed to resolve validation results and build a new remediation plan if cached plan is not found
    if reuse_saved_remediation:
        try:
            return load_remediation_plan(path)
        except FileNotFoundError:
            pass
    #resolve validation results using the helper above
    resolved_validation = _resolve_validation_results(path, validation_results, reuse_saved_validation)
    #build the remediation plan based on the resolved validation results
    plan = build_remediation_plan(resolved_validation)
    #save the plan to cache/storage for future reuse and return it
    save_remediation_plan(path, plan)
    return plan


### 7.7) Apply the remediation plan and accepted cleaners

`run_cleaner_application_with_plan` is where the two streams meet: the deterministic actions from `run_remediation_planning` and the accepted generated cleaners from the loop. It loads the saved artifacts, applies them in a fixed order, updates action statuses, writes the cleaned CSV, and returns the `CleaningReport` plus per-cleaner execution reports.

The function applies operations in a fixed order. First, it runs the generated format cleaners from the cleaner manifest. For each artifact, it reloads the saved cleaner program, checks that the target column still exists, applies the cleaner with `apply_cleaner_to_series(...)`, replaces the dataframe column only if execution succeeds, and records unresolved risks otherwise. It also updates the corresponding `generate_cleaner` remediation actions as `applied` or `failed`.

After the generated cleaners, the function applies deterministic remediation steps: placeholder-like values are converted to nulls, exact duplicate columns approved by the remediation plan are dropped, schema-based column renames are applied, schema-based dtype casts are attempted, and string columns are lowercased as a final normalization pass. The function updates action statuses where applicable, but manual-review, anomaly, report-only, and other non-auto-applied actions are not directly executed here.

The main internal calls are easiest to read as an ordered pipeline:
- `load_remediation_plan(...)` and `_clone_remediation_plan(...)`: recover the plan and make a writable working copy of its statuses;
- `load_cleaner_manifest(...)` and `load_dataset_frame(...)`: load the accepted cleaners and the raw dataframe to mutate;
- `_load_artifact_program(...)` and `apply_cleaner_to_series(...)`: reconstruct each saved cleaner and execute it on the target dataframe column;
- `_apply_placeholder_nulls(...)`: replace completeness-stage placeholder tokens with true missing values;
- `_apply_exact_duplicate_column_drops(...)`: remove exact duplicate columns that the remediation plan marked as safe to drop;
- `_apply_column_renames(...)`: apply schema-driven rename suggestions;
- `_apply_dtype_casts(...)`: cast cleaned columns into their target dtypes;
- `_apply_string_lowercasing(...)`: final text-normalization pass for string columns;
- `cleaning_cache_dir(...)`, `cleaned_dataset_path(...)`, `save_cleaner_manifest(...)`, and `save_remediation_plan(...)`: persist the cleaned dataset and the updated artifacts back to disk.

At the end, the function builds a `CleaningReport` containing the dataset shape before and after cleaning, the applied cleaner artifacts, unresolved risks, a compressed base64 snapshot of the cleaned CSV, and a textual summary of the operations performed. This final report, together with the execution reports and updated remediation plan, is returned to the caller for inspection or downstream verification. This is the stage that turns accepted code and deterministic actions into the final cleaned dataframe that verification will inspect.


In [ ]:
def run_cleaner_application_with_plan(
    path: Path,
    remediation_plan: RemediationPlan | None = None,
    on_event=None,
) -> tuple[CleaningReport, list[ColumnCleanerExecutionReport], RemediationPlan | None]:
    '''Applies generated cleaners and remediation steps, saves the cleaned CSV, and returns the final report.'''
    #it returns the cleaning report, a list of execution reports and the updated remediation plan
    def _emit(message: str) -> None:
        '''Sends progress updates to the optional callback without interrupting the main workflow.'''
        #If no callback exists, do nothing
        if on_event is None:
            return
        #Try calling the callback and send the message
        try:
            on_event(message)
        #If callback fails, ignore it
        except Exception:
            pass

    # Load the remediation plan from cache if the caller did not provide one
    if remediation_plan is None:
        remediation_plan = load_remediation_plan(path)
    # Clone the plan so action statuses can be updated without mutating a shared object
    remediation_plan = _clone_remediation_plan(remediation_plan)
    #Get the planâ€™s action list, or use an empty list if there is no plan
    actions = remediation_plan.actions if remediation_plan is not None else []

    # Load the generated cleaner manifest if it exists
    artifacts = load_cleaner_manifest(path)
    # Load the raw dataset and record the initial shape for the final report
    df = load_dataset_frame(path)
    #the rows and columns before cleaning are recorded
    rows_before = len(df)
    columns_before = len(df.columns)
    #Start a list for cleaner execution reports and applied artificats and unresolved risks
    execution_reports: list[ColumnCleanerExecutionReport] = []
    applied_artifacts: list[GeneratedCleanerArtifact] = []
    unresolved_risks: list[str] = []

    #Log step 1
    print(f"\n[apply] step 1 - format cleaners ({len(artifacts)} columns)", file=sys.stderr)
    #Send a short progress update
    _emit(f"step 1 â€” running {len(artifacts)} format cleaner{'s' if len(artifacts) != 1 else ''}")
    #If no generated cleaners were found, Log that step 1 has nothing to do
    if not artifacts:
        print("  no cleaner manifest found or no generated cleaners recorded.", file=sys.stderr)
    #Loop through each generated cleaner artifact
    for idx, artifact in enumerate(artifacts, start=1):
        # Load the cleaner code saved for this artifact
        #Try to reconstruct the cleaner program from disk
        program = _load_artifact_program(artifact)
        #If the code file is missing,
        if program is None:
            #Add a risk message
            unresolved_risks.append(f"{artifact.column_name}: cleaner file missing at {artifact.code_path}")
            continue #and skip the artifact
        #If the source column is missing from the dataset,
        if artifact.column_name not in df.columns:
            #add a risk message and skip the artifact
            unresolved_risks.append(f"{artifact.column_name}: source column missing from dataset")
            continue

        # Execute the cleaner against the dataframe column and collect an execution report
        cleaned_series, report = apply_cleaner_to_series(df[artifact.column_name], program)
        execution_reports.append(report)
        
        #If the cleaner ran successfully and returned a cleaned series
        if report.execution_ok and cleaned_series is not None:
            # Replace the original column with the cleaned output series when execution succeeds
            df[artifact.column_name] = cleaned_series
            print(f"  '{artifact.column_name}' OK - {report.changed_rows} rows changed", file=sys.stderr)
            _emit(f"  [{idx}/{len(artifacts)}] âœ“ '{artifact.column_name}' â€” {report.changed_rows} rows changed")
        else:
            # Keep the original column unchanged and store the reported risks
            print(f"  '{artifact.column_name}' FAILED - {'; '.join(report.unresolved_risks)}", file=sys.stderr)
            _emit(f"  [{idx}/{len(artifacts)}] âœ— '{artifact.column_name}' failed")
            #Add all cleaner failure risks to the global unresolved risks list
            unresolved_risks.extend(f"{artifact.column_name}: {risk}" for risk in report.unresolved_risks)

        # Create a new artifact that records the execution results of this cleaner, including changed row count and summary
        applied_artifacts.append(
            GeneratedCleanerArtifact(
                column_name=artifact.column_name,
                function_name=artifact.function_name,
                code_path=artifact.code_path,
                changed_rows=report.changed_rows,
                summary=report.summary,
                example_transformations=list(artifact.example_transformations),
            )
        )

    # Map execution reports by column name so remediation actions can be updated later
    execution_by_column = {report.column_name: report for report in execution_reports}
    for action in actions:
        #Skip non-cleaner actions
        if action.action_type != "generate_cleaner":
            continue
        #Look up the execution report for the target column
        report = execution_by_column.get(str(action.target.get("column_name", "")))
        if report is None:
            action.status = "failed"
        elif report.execution_ok:
            action.status = "applied"
        else:
            action.status = "failed"

    # Print a short summary of cleaner execution success and failure counts
    failed = [report for report in execution_reports if not report.execution_ok]
    #Print a summary saying how many cleaner executions succeeded and which ones failed
    print(
        f"\n  execution summary: {len(execution_reports) - len(failed)}/{len(execution_reports)} succeeded"
        + (f", {len(failed)} FAILED: {[report.column_name for report in failed]}" if failed else ""),
        file=sys.stderr,
    )

    #Start step 2 placeholder -> null
    print("\n[apply] step 2 - placeholder -> null (from completeness cache)", file=sys.stderr)
    _emit("step 2 â€” replacing placeholder tokens with null")
    # Replace placeholder-like tokens such as "n/a" with proper missing values
    df, total_replaced, placeholder_by_column = _apply_placeholder_nulls(df, path)
    print(f"  total placeholder replacements: {total_replaced}", file=sys.stderr)
    _emit(f"  {total_replaced} placeholder values replaced across {len(placeholder_by_column)} columns")
    #Loop through remediation actions again
    for action in actions:
        if action.action_type != "replace_placeholders_with_null":
            continue # Skip unrelated actions
        column_name = str(action.target.get("column_name", ""))
        #mark the action as applied if at least one placeholder was replaced otheriwse not needed
        action.status = "applied" if placeholder_by_column.get(column_name, 0) > 0 else "not_needed"

    #Start step 3 exact duplicate column drops
    print("\n[apply] step 3 - exact duplicate column drops (from remediation plan)", file=sys.stderr)
    _emit("step 3 â€” dropping exact duplicate columns")
    #If there is a remediation plan with actions,Apply the duplicate-column drops
    if actions:
        # Apply auto-approved duplicate column drops from the remediation plan
        df, drop_risks = _apply_exact_duplicate_column_drops(df, actions)
        unresolved_risks.extend(drop_risks) #Add any new risk messages
    else: #if there is no plan log the skip
        print("  no remediation plan loaded - skipping exact duplicate column drops.", file=sys.stderr)

    #Start step 4 columns rename
    print("\n[apply] step 4 - column renames (from schema cache)", file=sys.stderr)
    _emit("step 4 â€” applying schema renames")
    # Rename columns according to schema-based naming suggestions.
    df, rename_map = _apply_column_renames(df, path)
    if rename_map:
        _emit(f"  renamed {len(rename_map)} columns")
    for action in actions:
        if action.action_type != "rename_column":
            continue
        column_name = str(action.target.get("column_name", ""))
        new_name = str(action.target.get("new_name", ""))
        if rename_map.get(column_name) == new_name:
            action.status = "applied"
        elif column_name not in df.columns:
            action.status = "not_needed"
        else:
            action.status = "not_needed"

    #Start step 5 dtype casting
    print("\n[apply] step 5 - dtype casting (from schema cache)", file=sys.stderr)
    _emit("step 5 â€” casting dtypes")
    # Cast columns into their target dtypes using schema metadata.
    df, cast_results = _apply_dtype_casts(df, path)
    _applied_casts = sum(1 for s in cast_results.values() if s == "applied")
    if _applied_casts:
        _emit(f"  {_applied_casts} dtype cast{'s' if _applied_casts != 1 else ''} applied")
    for action in actions:
        if action.action_type != "cast_dtype":
            continue
        column_name = str(action.target.get("column_name", ""))
        status = cast_results.get(column_name, "not_needed")
        action.status = status if status in {"applied", "failed"} else "not_needed"

    #Start step 6 lowercase string columns
    print("\n[apply] step 6 - lowercase string columns", file=sys.stderr)
    _emit("step 6 â€” lowercasing string columns")
    # Lowercase string columns as a final text-normalization pass.
    df, lowercased_total, lowercased_by_column = _apply_string_lowercasing(df, path)
    if lowercased_total:
        _emit(
            f"  lowercased {lowercased_total} string value{'s' if lowercased_total != 1 else ''} "
            f"across {len(lowercased_by_column)} column{'s' if len(lowercased_by_column) != 1 else ''}"
        )

    #Start step 7 exact duplicate row drops
    print("\n[apply] step 7 - exact duplicate row drops (from remediation plan)", file=sys.stderr)
    _emit("step 7 — dropping exact duplicate rows")
    # Drop exact duplicate rows encoded as deterministic remediation actions.
    if actions:
        df, dropped_duplicate_rows, duplicate_row_risks = _apply_exact_duplicate_row_drops(df, actions)
        unresolved_risks.extend(duplicate_row_risks)
        if dropped_duplicate_rows:
            print(f"  dropped {dropped_duplicate_rows} exact duplicate row(s)", file=sys.stderr)
            _emit(f"  dropped {dropped_duplicate_rows} exact duplicate row(s)")
    else:
        dropped_duplicate_rows = 0
        print("  no remediation plan loaded - skipping exact duplicate row drops.", file=sys.stderr)

    # Save the final cleaned dataframe to the cleaning cache directory.
    output_dir = cleaning_cache_dir(path)
    output_dir.mkdir(parents=True, exist_ok=True)
    cleaned_path = cleaned_dataset_path(path)
    df.to_csv(cleaned_path, index=False)
    print(f"\n[apply] cleaned dataset saved -> {cleaned_path}", file=sys.stderr)

    # Persist the updated cleaner manifest and remediation plan statuses.
    save_cleaner_manifest(path, applied_artifacts)
    if remediation_plan is not None:
        save_remediation_plan(path, remediation_plan)

    # Build the final cleaning report with dataset shape changes, unresolved risks, and a compressed CSV snapshot.
    cleaning_report = CleaningReport(
        dataset_name=path.stem,
        rows_before=rows_before,
        rows_after=len(df),
        columns_before=columns_before,
        columns_after=len(df.columns),
        generated_cleaners=applied_artifacts,
        unresolved_risks=unresolved_risks,
        cleaned_csv_gzip_base64=gzip_text_to_base64(df.to_csv(index=False)),
        summary=(
            f"Applied {len(applied_artifacts)} format cleaners, replaced {total_replaced} placeholder values, "
            f"renamed {len(rename_map)} columns, cast dtypes, lowercased {lowercased_total} string values, "
            f"and dropped {dropped_duplicate_rows} exact duplicate row(s). "
            f"Cleaned dataset saved to `{cleaned_path.as_posix()}`."
        ),
    )
    return cleaning_report, execution_reports, remediation_plan


# 8) Verification and reporting

Application is not the end of the pipeline. After the cleaned CSV is written, the system first verifies whether the original format-consistency findings were actually improved or resolved, and then assembles the final outputs in two layers: a structured factual report and a narrative Markdown report built on top of that factual base.

The section is intentionally split in two parts:
- `run_verify` is shown as source because it is the key post-cleaning correctness check;
- `build_final_report` is not expanded because it is a long deterministic aggregation function, so the notebook explains its role and shows the compact call pattern instead;
- the two narrative agents are shown because they are the model-facing part of the final report generation.


### 8.1) `run_verify`

`run_verify` re-runs format-consistency validation on the cleaned CSV and compares the result against the original consistency findings. Before comparing, it aligns renamed columns through `_schema_rename_map` and excludes columns that, after cleaning, became proper numeric dtypes, because format-consistency findings are no longer meaningful for them in the same way.

The function then builds a `FindingDiff` comparison for each original finding and also records any new post-cleaning findings that were not present before. The final statuses therefore include `resolved`, `improved`, `unchanged`, `regressed`, and `new`.

The important helper calls inside the function are:
- `cleaned_dataset_path(...)`: locates the cleaned CSV produced by application;
- `load_consistency(...)`: loads the original pre-cleaning baseline;
- `_schema_rename_map(...)`: maps original column names to their cleaned names;
- `load_dataset_frame(...)`: loads the cleaned CSV to inspect the final dtypes;
- `run_format_consistency_validation(..., read_as_str=True)`: re-runs the same format check on cleaned data;
- `_print_diff_table(...)` and `_diff_summary(...)`: produce the readable verification output.

If the cleaned CSV does not exist yet, verification stops with an error instead of silently proceeding.


In [ ]:
def run_verify(path: Path, on_event=None, max_workers: int = 1) -> ConsistencyVerificationReport:
    '''Re-runs consistency validation on the cleaned CSV and compares the results with the original findings.'''
    # Verification must use at least one worker
    if max_workers < 1:
        raise ValueError("max_workers must be at least 1.")

    def _emit(message: str) -> None:
        '''Sends progress updates to the optional callback without breaking verification.'''
        if on_event is None:
            return
        try:
            on_event(message)
        except Exception:
            pass #ignore the error so verification continues normally

    # The cleaned dataset must already exist before verification can compare before vs after
    cleaned_path = cleaned_dataset_path(path)
    if not cleaned_path.exists():
        #If the cleaned CSV does not exist, raise an error telling the user to run the apply stage first
        raise FileNotFoundError(
            f"Cleaned dataset not found at {cleaned_path}. Run --stage apply first."
        )

    # Load the original consistency findings and index them by original column name
    original = load_consistency(path)
    original_map = {finding.column_name: finding for finding in original.format_consistency_findings}

    # Build rename maps so cleaned columns can be aligned with original column names
    rename_map = _schema_rename_map(path)
    #Build the reverse mapping: cleaned/renamed name -> original name. This is useful because after cleaning a column may have a different name than before.
    reverse_rename = {new: old for old, new in rename_map.items()}

    # Load the cleaned dataset and identify columns that became numeric after cleaning, then map them back to original names
    cleaned_df = load_dataset_frame(cleaned_path)
    numeric_original_names = _numeric_original_names(cleaned_df, reverse_rename)

    print(f"\n[verify] running consistency on cleaned dataset: {cleaned_path}", file=sys.stderr)
    _emit(f"re-running consistency on {len(original_map)} original finding columns")
    # Re-run consistency validation on the cleaned CSV without using cached results
    after = run_format_consistency_validation(
        cleaned_path,
        reuse_cache=False,
        read_as_str=True, #means read values as strings for consistency analysis
        max_workers=max_workers, #controls parallelism
        schema_path=path,
    )
    # Map the new findings back to original column names and ignore columns that are now numeric,
    # because format-consistency findings on numeric columns are no longer meaningful here
    after_map = {
        reverse_rename.get(finding.column_name, finding.column_name): finding
        for finding in after.format_consistency_findings
        if reverse_rename.get(finding.column_name, finding.column_name) not in numeric_original_names
    }
    #if a column became numeric after cleaning, do not include its format-consistency finding here.

    # Build one diff entry per original finding to show whether cleaning resolved or changed it
    #Start an empty list to store the comparison results
    diffs: list[FindingDiff] = []
    for column_name, before_finding in original_map.items():
        after_finding = after_map.get(column_name)
        before_rows = before_finding.inconsistent_rows
        after_rows = after_finding.inconsistent_rows if after_finding else 0
        # Compute the percentage reduction in inconsistent rows for this column
        reduction_pct = round((before_rows - after_rows) / before_rows * 100, 1) if before_rows > 0 else 0.0

        # Classify the outcome by comparing the number of inconsistent rows before and after cleaning
        if after_finding is None or after_rows == 0:
            status = "resolved"
        elif after_rows < before_rows:
            status = "improved"
        elif after_rows == before_rows:
            status = "unchanged"
        else:
            status = "regressed"

        # Save the diff result, including remaining examples and any schema rename
        diffs.append(
            FindingDiff(
                column_name=column_name,
                status=status,
                before_inconsistent_rows=before_rows,
                after_inconsistent_rows=after_rows,
                reduction_pct=reduction_pct,
                #If the finding still exists after cleaning, store its remaining inconsistent examples. Otherwise use an empty list
                remaining_examples=after_finding.example_inconsistent_values if after_finding else [],
                #Store the new name if this column was renamed
                renamed_to=rename_map.get(column_name),
            )
        )
        # Emit a short progress line using the renamed display name when available
        display_name = rename_map.get(column_name, column_name)
        _emit(f"  {status}: '{display_name}' ({before_rows}â†’{after_rows} rows)")

    # Add any findings that appear only after cleaning as brand-new issues
    for column_name, after_finding in after_map.items():
        #If the column already existed in the original findings, skip it, because it was already handled abov
        if column_name in original_map:
            continue
        diffs.append(
            FindingDiff(
                column_name=column_name,
                status="new", #Mark it as a new finding introduced after cleaning
                before_inconsistent_rows=0,
                #Store how many inconsistent rows it has after cleaning
                after_inconsistent_rows=after_finding.inconsistent_rows,
                #Use -100.0 to show this is not an improvement but a new issue
                reduction_pct=-100.0,
                #Store the problematic examples
                remaining_examples=after_finding.example_inconsistent_values,
            )
        )

    # Print a detailed table to stderr for CLI visibility.
    _print_diff_table(diffs, len(original_map), len(after_map))

    # Return the final verification report with the diffs and a short summary sentence
    return ConsistencyVerificationReport(
        dataset_name=path.stem,
        original_finding_count=len(original_map),
        remaining_finding_count=len(after_map),
        diffs=diffs, #Store the full list of per-column diff results
        summary=_diff_summary(diffs), 
    )


### 8.2) Final structured report

`build_final_report` is the deterministic merger for the full run. It does not call an agent. It combines the validation bundle, remediation plan, cleaning report, and verification report into one `FinalPipelineReport` that can be saved as JSON and used as the factual source for the narrative report.

The function is intentionally not expanded here because it is mostly long aggregation code, but its role is important: it not only counts findings and groups remediation actions by status, it also computes concrete final reporting fields such as duplicate-row removal totals, manual-review queues, verification diffs, and real non-null counts read back from the cleaned CSV.

In other words, this step produces the authoritative factual object before any narrative agent writes prose.


In [ ]:
def build_final_report(
    validation_results: OrchestrationStepResult,
    remediation_plan: RemediationPlan,
    cleaning_report: CleaningReport,
    verification_report: ConsistencyVerificationReport | None,
    dataset_path: Path | None = None,
) -> FinalPipelineReport:
    '''Combines validation, remediation, cleaning, and verification outputs into one final report object.'''
    # Build compact count summaries for each validation area.
    validation_summary = {
        "schema_issues": len(validation_results.schema_validation.issues),
        "completeness_columns_with_missing": len(validation_results.completeness_analysis.columns_with_missing_values),
        "consistency_findings": len(validation_results.consistency_validation.format_consistency_findings),
        "anomaly_findings": len(validation_results.anomaly_detection.findings) if validation_results.anomaly_detection else 0,
        "cross_column_findings": len(validation_results.cross_column_validation.findings) if validation_results.cross_column_validation else 0,
        "duplicate_groups": len(validation_results.duplicate_detection.groups) if validation_results.duplicate_detection else 0,
    }

    # Group remediation actions by status so the final report can summarize what happened.
    applied_actions = [action for action in remediation_plan.actions if action.status == "applied"]
    proposed_not_applied_actions = [action for action in remediation_plan.actions if action.status == "proposed_not_applied"]
    failed_actions = [action for action in remediation_plan.actions if action.status == "failed"]
    not_needed_actions = [action for action in remediation_plan.actions if action.status == "not_needed"]
    duplicate_row_drop_candidates = [
        action
        for action in remediation_plan.actions
        if action.action_type in {"drop_rows_candidate", "drop_exact_duplicate_rows"}
    ]
    applied_duplicate_row_drop_actions = [
        action
        for action in remediation_plan.actions
        if action.status == "applied" and action.action_type == "drop_exact_duplicate_rows"
    ]
    dropped_exact_duplicate_rows = sum(
        len(action.target.get("drop_row_indices", []))
        for action in applied_duplicate_row_drop_actions
    )
    manual_review_queue = [
        action
        for action in remediation_plan.actions
        if action.status == "proposed_not_applied" and action.action_type == "manual_review"
    ]

    # Use the verification summary when available, otherwise provide a fallback message.
    verification_summary = verification_report.summary if verification_report is not None else "Verification was not run."
    # Build one overall summary sentence for the final report.
    summary = (
        f"Validation found {sum(validation_summary.values())} section-level findings/signals. "
        f"Applied {len(applied_actions)} remediation actions, left {len(proposed_not_applied_actions)} proposed without auto-apply, "
        f"recorded {len(failed_actions)} failed actions, and dropped {dropped_exact_duplicate_rows} exact duplicate row(s)."
    )

    # Read authoritative non-null counts from the cleaned CSV for later narrative tables.
    total_rows_cleaned, non_null_counts_cleaned = _compute_cleaned_non_null_counts(dataset_path)

    # Normalize optional validation sections into plain lists.
    anomaly_findings = (
        list(validation_results.anomaly_detection.findings)
        if validation_results.anomaly_detection is not None
        else []
    )
    cross_column_findings = (
        list(validation_results.cross_column_validation.findings)
        if validation_results.cross_column_validation is not None
        else []
    )
    duplicate_groups = (
        list(validation_results.duplicate_detection.groups)
        if validation_results.duplicate_detection is not None
        else []
    )
    completeness_details = list(validation_results.completeness_analysis.per_column)

    # Assemble the final structured report model.
    return FinalPipelineReport(
        dataset_name=validation_results.schema_validation.dataset_name,
        validation_summary=validation_summary,
        applied_actions=applied_actions,
        proposed_not_applied_actions=proposed_not_applied_actions,
        failed_actions=failed_actions,
        not_needed_actions=not_needed_actions,
        duplicate_row_drop_candidates=duplicate_row_drop_candidates,
        manual_review_queue=manual_review_queue,
        cleaning_summary=cleaning_report.summary,
        verification_summary=verification_summary,
        verification_diffs=verification_report.diffs if verification_report is not None else [],
        generated_cleaners=cleaning_report.generated_cleaners,
        total_rows_cleaned=total_rows_cleaned,
        non_null_counts_cleaned=non_null_counts_cleaned,
        completeness_details=completeness_details,
        anomaly_findings=anomaly_findings,
        cross_column_findings=cross_column_findings,
        duplicate_groups=duplicate_groups,
        unresolved_risks=cleaning_report.unresolved_risks,
        summary=summary,
    )


### 8.3) Chunked narrative report generation

`_generate_narrative_report_chunked` is the model-facing part of the final reporting stage. Instead of asking one agent to write the entire report in a single large prompt, it splits the task into smaller grounded calls built from the structured final report.

The function uses two narrative agents internally:
- `narrative_frontmatter_agent` writes the title, executive summary, and recommendations from `_build_frontmatter_brief(...)`;
- `narrative_section_agent` writes one report section at a time from `_build_narrative_section_specs(...)`.

Both calls are wrapped through `run_agent_with_backoff(...)`, so generation is retried through the shared agent runner. After each section call, the function forces the returned heading to match the requested heading and lightly normalizes the generated prose with `_polish_narrative_body(...)`.

There is no hand-written deterministic narrative fallback here: the factual report is deterministic, but the prose layer still depends on the narrative agents succeeding.


In [ ]:
def _generate_narrative_report_chunked(final_report: FinalPipelineReport) -> NarrativeReport:
    '''Generates the narrative report in chunks using separate agents for frontmatter and body sections.'''
    # Import the agents lazily because they are only needed when generating the narrative.

    # Generate the title, executive summary, and recommendations first.
    frontmatter = run_agent_with_backoff(
        narrative_frontmatter_agent,
        [
            f"Write the front matter for dataset '{final_report.dataset_name}'.",
            attach_text_document(_build_frontmatter_brief(final_report)),
        ],
    ).output

    # Generate each narrative section independently from its own briefing block.
    sections: list[NarrativeReportSection] = []
    for heading, briefing in _build_narrative_section_specs(final_report):
        section = run_agent_with_backoff(
            narrative_section_agent,
            [
                (
                    f"Write exactly one report section with heading '{heading}' for dataset "
                    f"'{final_report.dataset_name}'."
                ),
                attach_text_document(briefing),
            ],
        ).output
        # Force the returned heading to match the expected section heading if needed.
        if section.heading != heading:
            section = section.model_copy(update={"heading": heading})
        # Lightly clean the generated body text before saving it.
        section = section.model_copy(update={"body": _polish_narrative_body(section.body)})
        sections.append(section)

    # Assemble the final narrative report from generated frontmatter and sections.
    return NarrativeReport(
        title=frontmatter.title,
        executive_summary=_polish_narrative_body(frontmatter.executive_summary),
        sections=sections,
        recommendations=[_polish_narrative_body(item) for item in frontmatter.recommendations],
    )


# 9) End-to-End Pipeline Run

This final section executes the real production pipeline on `DATASET_PATH` using the same staged helpers introduced above. Instead of collapsing everything into a single `run_cleaning(...)` call, the notebook keeps each phase explicit so the reader can inspect the real intermediate artifacts.

The compact orchestration entry point still exists for application and CLI usage, but here we intentionally run the same workflow step by step for transparency.


In [ ]:
# Configure the end-to-end run.
# These defaults intentionally force a fresh execution so the notebook demonstrates
# the full real pipeline rather than silently reusing cached artifacts.
REUSE_SAVED_VALIDATION = False
REUSE_SAVED_REMEDIATION = False
CLEANER_ATTEMPTS = 10
CLEANER_WORKERS = 1
VERIFY_WORKERS = 1

print({
    "reuse_saved_validation": REUSE_SAVED_VALIDATION,
    "reuse_saved_remediation": REUSE_SAVED_REMEDIATION,
    "cleaner_attempts": CLEANER_ATTEMPTS,
    "cleaner_workers": CLEANER_WORKERS,
    "verify_workers": VERIFY_WORKERS,
})


### 9.1 Resolve the validation bundle

The cleaning half of the system consumes one `OrchestrationStepResult`. This cell either loads that bundle from cache or rebuilds it from scratch, depending on the configuration above.


In [ ]:
# Resolve or build the full validation bundle consumed by the cleaning stages.
# When cache reuse is disabled, this runs the schema, completeness, consistency,
# anomaly, cross-column, and duplicate stages from scratch.
validation_results = _resolve_validation_results(
    DATASET_PATH,
    validation_results=None,
    reuse_saved_validation=REUSE_SAVED_VALIDATION,
)

# Show a compact validation summary before moving into cleaning.
display(Markdown(f"""
**Validation bundle ready for** `{validation_results.schema_validation.dataset_name}`

- Schema issues: **{len(validation_results.schema_validation.issues)}**
- Completeness columns with missing values: **{len(validation_results.completeness_analysis.columns_with_missing_values)}**
- Format consistency findings: **{len(validation_results.consistency_validation.format_consistency_findings)}**
- Anomaly findings: **{len(validation_results.anomaly_detection.findings) if validation_results.anomaly_detection else 0}**
- Cross-column findings: **{len(validation_results.cross_column_validation.findings) if validation_results.cross_column_validation else 0}**
- Duplicate groups: **{len(validation_results.duplicate_detection.groups) if validation_results.duplicate_detection else 0}**
"""))


### 9.2 Build the remediation plan

This stage converts the validation findings into deterministic `RemediationAction` objects such as renames, dtype casts, placeholder replacement, and human-review queues.


In [ ]:
# Build the deterministic remediation plan that accompanies the generated cleaners.
remediation_plan = run_remediation_planning(
    DATASET_PATH,
    validation_results=validation_results,
    reuse_saved_validation=REUSE_SAVED_VALIDATION,
    reuse_saved_remediation=REUSE_SAVED_REMEDIATION,
)

# Summarize the plan by action status so the next stages start from visible decisions.
applied_plan_actions = sum(1 for action in remediation_plan.actions if action.status == "applied")
deferred_plan_actions = sum(1 for action in remediation_plan.actions if action.status == "proposed_not_applied")
failed_plan_actions = sum(1 for action in remediation_plan.actions if action.status == "failed")
not_needed_plan_actions = sum(1 for action in remediation_plan.actions if action.status == "not_needed")

display(Markdown(f"""
**Remediation plan ready**

- Total actions: **{len(remediation_plan.actions)}**
- Applied: **{applied_plan_actions}**
- Deferred to manual review: **{deferred_plan_actions}**
- Failed: **{failed_plan_actions}**
- Not needed: **{not_needed_plan_actions}**
"""))


### 9.3 Build cleaning requests

Each format-consistency finding becomes one `ColumnCleaningRequest`. This is the structured contract the generator uses to synthesize one cleaner per dirty column.


In [ ]:
# Build the per-column cleaning requests that describe the generator input for each
# format-consistency finding found during validation.
raw_df = load_dataset_frame(DATASET_PATH)

# Load schema metadata when available so each request can carry dtype and semantic hints.
try:
    schema_handoff = load_schema_handoff(DATASET_PATH)
    schema_map = {column.name: column for column in schema_handoff.columns}
except FileNotFoundError:
    schema_map = {}

cleaning_requests = []
for finding in validation_results.consistency_validation.format_consistency_findings:
    format_facts = build_column_format_facts(raw_df, finding.column_name)
    cleaning_requests.append(
        build_column_cleaning_request(
            DATASET_PATH.stem,
            finding.column_name,
            finding,
            format_facts,
            schema_entry=schema_map.get(finding.column_name),
        )
    )

# Show the number of generator inputs and preview the first few target columns.
request_preview = ", ".join(request.column_name for request in cleaning_requests[:10]) or "none"
display(Markdown(f"""
**Cleaning requests built:** **{len(cleaning_requests)}**

**Target columns:** {request_preview}
"""))


### 9.4 Run cleaner generation

This stage launches the real generator / critic loop and persists the accepted cleaner artifacts for later application.


In [ ]:
# Run the real cleaner-generation stage.
# The generation workflow reloads the saved consistency findings, synthesizes one
# cleaner per requested column, validates the code, and persists accepted artifacts.
generated_cleaner_artifacts = run_cleaner_generation(
    DATASET_PATH,
    reuse_consistency=True,
    max_attempts=CLEANER_ATTEMPTS,
    max_workers=CLEANER_WORKERS,
)

display(Markdown(f"""
**Generated cleaner artifacts:** **{len(generated_cleaner_artifacts)}**
"""))


### 9.5 Apply the remediation plan and accepted cleaners

This is the first stage that mutates the dataset. It applies the accepted cleaner programs together with the deterministic remediation actions and writes the cleaned CSV.


In [ ]:
# Apply the generated cleaners together with the deterministic remediation plan.
# The function returns the final cleaning report, one execution report per cleaner,
# and the updated remediation plan with final action statuses.
cleaning_report, execution_reports, remediation_plan = run_cleaner_application_with_plan(
    DATASET_PATH,
    remediation_plan,
)
cleaned_dataset_output_path = cleaned_dataset_path(DATASET_PATH)

display(Markdown(f"""
**Cleaning stage complete**

- Rows: **{cleaning_report.rows_before} -> {cleaning_report.rows_after}**
- Columns: **{cleaning_report.columns_before} -> {cleaning_report.columns_after}**
- Cleaner execution reports: **{len(execution_reports)}**
- Cleaned dataset path: `{cleaned_dataset_output_path}`
"""))


### 9.6 Verify the cleaned dataset

Verification re-runs the consistency checks on the cleaned CSV and compares the result with the original format findings to measure how much the cleaners improved the data.


In [ ]:
# Re-run consistency validation on the cleaned CSV and compare the output against
# the original baseline to measure what changed after cleaning.
verification_report = run_verify(
    DATASET_PATH,
    max_workers=VERIFY_WORKERS,
)

display(Markdown(f"""
**Verification summary:** {verification_report.summary}
"""))


### 9.7 Build and save the final structured report

This deterministic stage merges validation, remediation, application, and verification into the factual `FinalPipelineReport` that the narrative writer consumes.


In [ ]:
# Build the final structured report and save it to the standard cache location.
final_report = build_final_report(
    validation_results,
    remediation_plan,
    cleaning_report,
    verification_report,
    dataset_path=DATASET_PATH,
)
final_report_output_path = save_final_report(DATASET_PATH, final_report)

display(Markdown(f"""
**Final structured report saved:** `{final_report_output_path}`
"""))


### 9.8 Generate and save the narrative report

The final model-facing stage turns the factual structured report into the human-readable Markdown narrative saved next to the cleaned dataset artifacts.


In [ ]:
# Generate the chunked narrative report from the structured final report and save it.
narrative_report = _generate_narrative_report_chunked(final_report)
narrative_output_path = save_narrative_report(DATASET_PATH, narrative_report)

display(Markdown(f"""
**Narrative report saved:** `{narrative_output_path}`

- Sections: **{len(narrative_report.sections)}**
- Recommendations: **{len(narrative_report.recommendations)}**
"""))


### 9.9 Final pipeline summary

This compact summary collects the key outputs of the full run in one place before previewing the generated Markdown narrative.


In [ ]:
# Summarize the key outputs of the real end-to-end run.
validation_signal_count = sum(final_report.validation_summary.values())
applied_action_count = len(final_report.applied_actions)
failed_action_count = len(final_report.failed_actions)

display(Markdown(f"""
**End-to-end pipeline completed for** `{final_report.dataset_name}`

- Cleaned dataset path: `{cleaned_dataset_output_path}`
- Validation findings / signals: **{validation_signal_count}**
- Applied remediation actions: **{applied_action_count}**
- Failed remediation actions: **{failed_action_count}**
- Verification summary: {verification_report.summary}
- Final JSON report path: `{final_report_output_path}`
- Narrative Markdown path: `{narrative_output_path}`
"""))


### 9.10 Full narrative report

The cell below renders the full saved narrative report directly in the notebook.


In [ ]:
# Render the full saved narrative report directly in the notebook.
narrative_markdown = narrative_output_path.read_text(encoding="utf-8").strip()
display(Markdown(narrative_markdown or "_Narrative report is empty._"))
